# Akun/VM Colab baru: Electric Bass konteks 90 detik

Source bridge sudah disertakan; tidak perlu ZIP dari Mac, VM lama, atau login GitHub. Pilih **Runtime → Change runtime type → GPU**, kemudian **Run all**. Upload piano asli saat diminta.

Notebook memasang runtime Python 3.11 terisolasi dan bobot ACE-Step yang pinned, lalu menjalankan seluruh lagu dengan batas konteks **90 detik termasuk margin**. Tidak mengulang uji 60/90/120 detik. Akun baru tetap memerlukan akses GPU Colab yang tersedia.

VM baru memerlukan sekitar **45–50 GiB ruang kosong**, internet, dan unduhan bobot sekitar **13.86 GiB** di luar dependensi runtime. Filesystem/bobot VM lama tidak otomatis pindah antar akun. Simpan hasil sebelum sesi berakhir. `DRY_RUN=True` hanya memeriksa source/plan tanpa instalasi, unduhan atau inferensi; bukan hasil GPU nyata.


In [ ]:
from pathlib import Path, PurePosixPath
import base64, gzip, hashlib, importlib.util, json, shutil, subprocess, sys, uuid, zipfile

DRY_RUN = False
BASE = Path(globals().get("WORKSPACE_BASE", "/content" if Path("/content").is_dir() else Path.cwd())).resolve()
BASE.mkdir(parents=True, exist_ok=True)
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU pada Runtime → Change runtime type; akun ini harus memiliki akses GPU"
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

# Runtime source generated by scripts/build_context_bridge.py; no weights/audio.
# BEGIN EMBEDDED BRIDGE SOURCE
BRIDGE_SOURCE = {
  "configs/acoustic-drums.json": {
    "sha256": "bcfd73641fddbda1fb0975febfe28cccb3a3121ec48cec1d877933e64a6d70ca",
    "gzip_base64": "H4sIAAAAAAAC/0WRTWoDMQyF9zmFmPUk9IdQ6K60m56hlKDYmoyoxzKWJiGU3r3yTEo3Br/PenqSvzcAHcfuGToMMqtx2MY6T7o933d9g5NESisnNSoO9tsjKq040pkDNR7miKtWKgVWltzkIQna48NKOKu5O2VraGm0AiWK6tqHXwDu/Pxc9NPMEXOggwZMrc3T7u7mNFClhXioVrpfgQytYzyYHEKZXfeGtJCAxW6h3lUSGkX4GxpaFvhicyXIVDBzS7mDF6g+dkXOFHvIaHPFBKcqcia4sI1eFL560IyVAHOEkETdeeTtiNbDICnJhfMJbCQojFmgUB2kTm0wMJ4c9lDGitqeNY94zThx0B28JsK6phvZtAdDH3iYEwycki6vtfjfwJHsQpRXI/LSt7ZekJyuHlzgLL5Cv1eYKEn0mf9/Q3fd5mfzC1x9T/sNAgAA"
  },
  "configs/assets-base.json": {
    "sha256": "b87ff04e90500281d47e37eea933dd756cca895df4eea796f9d4c7a98c2f1dde",
    "gzip_base64": "H4sIAAAAAAAC/+2bS3MbyQ3H7/4ULJ0tGd0N9GNv3sSpzSGVxx5TKRa6gZboFR9FUtp4t/a7B6RJxbWRZHHopMa1OfA1D8z0rzHA/z8kf341mVxs2o3OeXqv681subj4ZuJe7xbPl6K39umCm262urq8d3RZeaMX+9Wb5d266W79n3uftRnfTr67u76eLa4nf7A9Jm//8sfJPsRkrlsW3vLryWq2WKhM1no/2x1q8zFSn93qxgL93T5MJj/vn23xWlfL6Ux2R3j7u3eX39spvHnbdP/GXdF+38OGH8PttnQlJtcRokQIzsfiY3M1qWdpLhZutbGo9//ee3f0Bc/3I/nrj7oIl+/mVUVsIJdwFb99wyIq0+3yB11srt5v7DgP+4qBmS14ezi4cWw/rJazxXbz5tRQm9lPu1NIkB4W8e31cj3b3sx3sa9n28t6u6yXmxt2n5zB7NrOYbdBJezFBVLE5jQ7wKRZgFoFn7jn2iAUVbnY7/rL63HCbje8nW51vrrlrV69ny3e83DczwY7AEfn4kDisZTesxgA1xVTzuRJS3O91By5FLXpjj7AyIkvF312fWZiPxrkQNhysgwk7F2V5HxL0TmU5j3l1KJqzcg1osuxFfFI4yY817UN6Gr7z+1wwI/FOPK1E84UBiIOLmAhclLAMEPswVNX6M7VWnMV6gySWho54l2judpw162V1uV6cwbpp0MdgbviKFv6PVE4DLWn+BhswJAUqRWkGCoUdaHVxBGtevRmRAomLrYsICcFy3boLUsTzOpDH3nt3qx0JwMO/W0659WZVeWzAQ8TEt3Q7OfmQwNm9hiQ7D0WDAFYayqNcigUkrXSMG7wez6GYn0m76fiPOQ9+pCATs160Z5ir5BjSa6lHFPNvhh59tFHpyFLkYQK3oXiKdSQbQKKmGbBSsBfCfzpF+mknwl3mApCwIEpL8I9lgzZO2TELuKjWJZTjcYiRKv12ac0cur3y8b1TNSPxTjwNQAxh6FVxap56OqAW7ARqiNuuQfK2gmjT6GYUmwUy2gQ/+3d29//6d3VXF4E85GtH9KyDBXTHJyjps0bumpSTwisLhN06d04tpw0xlzGk5ZHV0zf/XR5O790V2moVTwx0jFFvUspD3WLlBESgcsqyk2iNLOHrgb1rdZeIIeoldGNGvdQs3hyrAevGPNA3uCqlhogiWkNQdPYyQadW+nVIbHPUYF7xXHzPrHBvTjGg1PMNJCvM9uSizPjYgkMUdn7YmamtFwAk0AREDZrPmq+JxrFl4b4bfrER+kMsoknRjrgNqVsbd6Rz6fq5e6ilYYinIKSNu1WHnpPGYScNcbiGaADRTHBLKmE0FIqKtRikyQ1j3oKzjGJg+Idsz97dGXojSjeW5VolwBDD7lqhKyQrGC1XX0JZudhd3WMGv0Am3hKmKMuweBdCeXUrA/EnWILnZrJEPatUcrGPXMuIaUEJTsThdhdDtyo5WbzwB1rFbQX+TrQT79ED32ZR3Sm8Lz56VMnYldFoInkYjLFHHsq2MkHoujVZKJlvjVYXzg2jDmykuNcc7NHZdJCo56IE33jS0P8Nm3jp18Mbu/WdTlYIX4mwjGjy2D9HRkqNbCsNZYOA/iCqYv1T0DXkwqkiJLb6Nnerfcgp4f1U1t/tfpwFuzPhXxQ5y7AQPwlEtvYg1kcZ5Lc9DiVliRDNM2O2WaiixLQiPGfpx1fFOdoM83WZyvOT93ie6aHdrORqaTOZjGBQ6kSQjFxWKOa4VcTKbloQA5W1G1JEB9qBG1R0XtJY8c/W1x/mqXT/arh6f/CmIdpKTaQoXe3OjsXPRODaw2pJcv/HhoFZ0CyT74xkAMc8QRsdmubTnf3Rxbbq9V2IPRn4hydU0ZnxunU5OeUshYRky2dNFdv5b4RmQ+zq0mCyzmgadMKQbjWJj2BRhWT9BKty9bRsD+1if6/ZX7kds96sgB5cp9jLfZD70h1tLKbqo9OxLScii8NrA5oKuDJmcBrVXyGUeGTWe93u3DT1Yftct1upsPa3smxjpd+SGgyI598z0TYWUZaL8OGIModipbqsSRUKiCusdlGiw521ZNAztg6Y1UpXEo9YRYe/TXcf0yFWqXxzndtVnHyXv5g70yJNVLAkCVyFYdf4vuYX5/R81/QDL6DXarT4uxBtSJhUDOCls9WPkuonYL62oD6uEjy6np6fTcTtpYzRCjseT4Z5PijvUJDhQEgZrZRRGNqrYljTIxQAHISiTnbGL0vvY6L6jk2b0/0+ZaFQ21GZUOGXpOH1BWs6vZkNS+WrAjdeSuDwaOTMdL8QsbuE7xfpa/7XwA/38rtKf83nRy6lHoEIHAcBatk62DCDFzZRG3qJsSokTW3llslDxSaWL9jyIgS6wiB/9po7Q42OMNfFvHo3IhwqIyDnlsniFATscdorqFn82qmVELiYFXbEh1zGBfw883anvLYvZo9/2P/FwJZ/ri4XbJM64ft/r8EDi1ELi6he/XLq38B/WBJB+cwAAA="
  },
  "configs/assets-xl-base.json": {
    "sha256": "7c6e217c4f044edf7c648d8c8d743b7b0be08af5fe74d2d203812ee85170e1ef",
    "gzip_base64": "H4sIAAAAAAAC/+1bTXMbuRG9769g+WzJaKABNHJzEqc2h1Q+9phKsRrohkyvRKpEymvv1v739NCk4iSSzRnaqdnaqEoSOR89g9eN1++Bw5++WSyebdtrveHlW73brjbrZ79ZwPNh881G9NrePeOm253eXryFePHu+qLyVp/tj9hu7u+aDof8ufdVW/H14tv7q6vV+mrxBztp8fIvf1zsoyxudMfCO36+uF2t1yqLO327Gq62/RCpr651a4H+bm8Wi5/2f23znd5ulisZrvDyd68uvrO7ePGy6f4FXMb9uYcDP4QbjoSSMnR0SZIL4FPxqUHN6lkapMKtNhb1/l9nD1df881+JH/9Qdfh4tVNVREbyIW7TL99wSIqy93me11vL99s7ToP54phs1rz7nBxg7J9f7tZrXfbF2NDbVc/DreQXX7YxNdXm7vV7vXNEPtqtbuo15t6sX3N8NEdrK7sHoYDasReIERFbKAEDrOSuNiq85k71eZCUZVn+1N/fj5PsNtr3i13enN7zTu9fLNav+HpcH8y2AFwBEgTEU+l9E5iAEBXzETRRy0NeqmUuBS1dCcf3MwR36z76urMwn40yAFhq8kyEWEPVTL4lhMASvM+Um5JtRJyTQiUWhGPcd4I3+idDehy9243HeDHYhzxtRumGCZCHCBgiRGkOIPZpR587Oo6QK2VqsTOTnLLM4d4aDSXW+66M2rd3G3PQPrpUEfAoUAkK78niMOg9jE9BrbDkBVjKxhTqK4ohFYzJzT26M0QKZi52LaAnNVZtbveSJogqQ995ty9vdVBBhz62/KGb89klc8GPCQkwdTq5+ZDc8zsMWC011gwBMdac2mRQokhWysN8wZ+j49BcXcm3k/Feah79CG7OLbqRXtOvTpKJUPLlHIlXwx59skn0EBSJKM6D6H4GGogS0AR0yxYo+NfCPjLL9JJPxPukIqIDieWvAj3VMiRB2TELuKTWJXHmgyLkIzryec8c9TfbhrXM6F+LMYBXwMgUZjKKsbmoSs4bsFGqBC5UQ+RtEdMPodiSrHFVGYD8d9evfz9n15d3shJYD5y9ENZlqlimgNAbNq8QVdN6kl0xsvRdendcGyUNSUq8ynLozGO3/54cX1zAZd5qlUcGelYoh5ypqluMRK6HB2QinKTJM3sIdSgvtXai6OQtDLCrOGeahZHx3rwiokm4u2gaqnBZTGtIWgaO9ugqZVeASN7Suq4V5w33iMb3MkxHpwixYn4gtkWKmDGxQrYJWXvi5mZ0qg4zOKKOGGz5rPGd6RRPDXEr9MnPorOJJs4MtIBblPK1uYhehqrlzsko4YinINGbdqNHnrP5CSCNcbi2bnuYhITzJJLCC3nohJbapKl0qxTcI5JnBTvWP3kEcrUhSjeW5VkU4BdD1Q1OVKXjbDawC/B7LwbZsesoZ9gE8eEOeoSDB5KKGOrPkTuMbXQYzMZwr61mMlwJ6YScs6uEJgoxA4UuMVGzfLAHWsVtH/yy4B++SV66GkeEUzhefPTYxMxsIhrIlRMpphjzwV79CHG5NVkolW+NVhfODVMlFgjMFVq9ls5aomzTsRI33hqiF+nbfz4s8Hd/V3dTFaIn4lwrOgyWX8ndjU2Z1VrWAIG5wvmLtY/HULPKi4nFGqzx/b+bg/k8rB/afsvb9+fBfbnQj6ocwhuIvwlRbaxB7M4YJLc9HgsLQu5ZJodyTLRRaOLM4b/PO14UpyjzTRbT0bOTy3xfaKHdrORueTOZjEdh1IlhGLisCY1w68mUqhoQA5G6rYliA81OW1J0XvJc4d/tb76uEqX+13Ty//EmIe0FBvI1NWtzgDJc2QHrWFs2eq/hxYDGCDks2/sIjiccQK2w96my2F9ZL27vN1NBP0TcY7OiRDMOI0tfs6ZtIiYbOlRqXqj+xaj+TCbTRKAKKBp0+qCcK1NenaaVEzSS7IuW2eD/dgm+v+W+QG3t6yjBciT5xy52E9dkepotJurTyBiWk7Fl+aMBzQX5yOYwGtVPLlZwSer3u+HcMvb97vNXXu9nNb2Rsc6Tv2Q0WQGjV4zEQarSOtl2NCJcndFS/VYMmosTqCx2UaL7mzWR3FE2DpjVSlcSh2RhaceiPuvbHjvmrF+0l6NlCKZqe2tgtcKycVixtXeeoQv8ZHMIzf1iY9pMOPUhZDeCYur1r7AJJspCeMG40/g3s23mOiIPjvxODs8+fZqeXW/ErbeM0UxHFF9Ms4BW2syU0Vy6ADInqBK6iWTJE5YIhfTxzmSSecimNnPDttzXN8R10+RsI+TWbh2riTeg7IGyiX0avPepZSLgIsmCyRhrH2mmH4ht/fvIJ/s9/xk2DWimtHz3Tkka3YhmpqIVBopmz7zxtUJfJ4fTexb1IWzH7jY9P0LPMv0HaE/PfCRpMv+eRmMaXwvNMVmfZ5Tsjsxh92rS13FWiB0QPEOI6g2iQGbQ05FLFHe5eCCdh9mnBT/tZLiRyQFh2W+0e5E1JQHkI27mwWvKaYm7GyaxBq9ACO5jkb5TtGBCeygaXhAB1xgn8c89/A/T0r4WkkJo5JCbrRqVMQatQ2fXbpazZ0Low8hxpx97TUkIoEQnFmeapOqdYkJbOrU6l0vEmecFPxaScGTk0JoYiiNX8QqBdWkZFIYVgtb8UhUQ++dqdrwKSkJD3K++sBo1mbIiGMBLzaV6jyT8jFYl6u16Lvz9NJpQY/fKgHMMPV5wUChm35Cj917b+xl5irl3EyXBjG/OTyeZdNonrD/5+reu+vlcL1zRNTJcY86ym6VpqLfkxYjJGoZpHTVzkrkWnaxIaF5e+1QzJDNDv3zlwuPeM99wdD+/mP/PTbZ/LC+3rAs6/vd/gttwUgyGnlRCt/8/M0/AZIVRDxvNwAA"
  },
  "configs/electric-bass.json": {
    "sha256": "4bacacb4f9fb1a9e66c98958869e2756878e6420910b2af065bea6522cb6ef63",
    "gzip_base64": "H4sIAAAAAAAC/0WQy27bQAxF9/4KQpts7MBpERTorst8QxAYDIeyiMwLQ44DI8i/lyOp6EYY3SPx3OHXAWBSWjjh5cZNpeTpNzwdRyzBjxNHJmtCp3dUnVaQSuA4GBKrcT3dnp4H5g0Hvgnx4NQDblltTLJPn+ZY0H7+2IhktdYTZxvov0SZg3r06i8AZ3++rfm1S8BMfFHCOCy/Hs/7oJkbr8Q7jV+fN1DmIQwXKxeq3XP38UoIq+2dXrRENA4wS776IuweGf7dHUYtcLNhAyQqqWKW0fkR/kBo9yNktN4wQo2dPnyMlczwKbYARcYW74DNhPomQTOkDz2CLqUZ5GIMzW2+QwXM/kEP8u4Neq3cYMGWShbSoVNJ1UnqKr6BtVoUl0kG9Ekt7eFp79v4Kr6RBnOJsXz69cAWhiqYy4Pus/0GJmmwYQ/3jGnopsP34S8y88VlIwIAAA=="
  },
  "configs/electric-context.json": {
    "sha256": "f70cd046214eb82fcc29bc6da216329d6229b13708df12e6aa0552e2cb998854",
    "gzip_base64": "H4sIAAAAAAAC/02QP0/EMAzF934KqzOtDiQQfzYkYGFgYT65qduzmqa92EGHEN8dNy1wU+SXZ7+f/VUAlNyW91CSJ6eRXdWgSOWmoHTS8mIxbMXe4bwXsqoV67jb5c8RY8/hTL/JcoctnYm7+vo265JzeApL5iuLUqAIc6SOYqQW9EA2uVo7QTj0nqrZkCCSJK8P9s5TVLNKakQxKKOHMLEQoMLl1Q622Hql/0DPLf5G/s/uiNoG3QAjKZoBwR3IDTYYe+QgmllWXhOftvvA4wJjwEJaw3PyvpIp9NacwgD9sk4OA1z4CUc4JiPQTxBl7w3/mDgSvLy9A53IpT+zz9ewjeuy+C5+AFLyNqSbAQAA"
  },
  "configs/mac-24gb.json": {
    "sha256": "16fb27fa5f89e4c5965452192ae1b0a3f5878d3d24ad44953521302668b4a429",
    "gzip_base64": "H4sIAAAAAAAC/02MsQ4CIRBEe76CUHuGK2z8FWPICnsJycESdrnG+O/CYWE5783MW2ltEgXczV0b8MiCZTnW2/ICRnMZOuARPQ6fCk8UM0ttCbMM3Ks/zoiBO3rY55lp23aC4IScL62LvsJ52ipIpOwYPeVztNqrnS8CVf5E5+qjvjXuBAWrAAAA"
  },
  "configs/mac-full-song.json": {
    "sha256": "e69fad3eeef49d4fac70c909418b75643abd8c115d98155c6e466580fd4e9b74",
    "gzip_base64": "H4sIAAAAAAAC/y3LQQqAIBCF4b2nENcJtWjTVSLEnBcImdJYm+juabWc739zCSlViIRVDVJZB85I+ux6PVuGamomnN6h9pD4I79x3o+ALVcu098ZIC40ttN7x2VZoyWTo3HpKKF8QdziAZ/359J3AAAA"
  },
  "requirements-context-chunks.txt": {
    "sha256": "f66d99f5111b5adc05fd368747e875bf36c9f3074984e636df275513fa9e8587",
    "gzip_base64": "H4sIAAAAAAAC/yXJQQrDIBBA0b2nGOhWbLXQlcmq+5wh1SkZojNFx0Ju30BXn8e/wNLShl3bqiR8ZVF8ieyA/KUmXJEVhMvh4ClwXiDuupZyVgV0QyiSdsxQJWOBNlipojM86ueYJ+/Cw8a76Yn+9N7GYLoMzm8qOE8354ON3vwAtgHTaYoAAAA="
  },
  "scripts/ace_context_chunks.py": {
    "sha256": "d7e4b17dea0647d859bbeea5d7809099c5582f7ee2b3bde05aec385864aa862c",
    "gzip_base64": "H4sIAAAAAAAC/6U827LbRnLv5ysg7QOAc4YgD3VZm+dAVV5Fu3ElsVW2k0oVw0INgSGJFQjAuJyLFVXlH/KH+ZL0ZQYYgCDlLetBIgYzPd09fZ+GXr58+d37D7OfG1U626LNE5XM4iJv1FPjqKdSVelR5U3tyDxxjkrWbaUS4dTyWGZqJrN0n6vEqZu0iQ9pvg+urn5Sba1qpzkoJz6o+BO83sq6nidVe3Qei+qTqurA+ZjJfF5WxVY5idrJNmucpnDywknznapUHqvg6gelktrJ22P5DDvGKf2DKO7STAlntzuWai+cMmt5O/WUAiL53nn/7//0nVO1eQO4O7uicmYz9aTitgGgL1++vEqPZVE1jqz2paxqZZ4Psj5k6dY8/r0ucvP7KJvD1a4qjk4Jv2CSo198xBf6d31omzTrntot0Berur4yQ0SKI4Gmsptl6MHhesd7EK1BDbyVmdkoLqpKZbJBwtM8iUolP/WQVabipkrjaNdmWQSI7wkeMCNTcCTf/fLBCZ1Xy8VicfXTjz/+Ag+IuBdFuHUU+UGl6iJ7UJ4fAEvwwNe3m6urKzgbIEsu37z1kHB/deXAnyTdq7oJNb8CPcGnd49pc2DgtCAoSpV7brV1fcKoqZQ8MhT8g4ezzYr4E5y7kzaq8jJ53CZyxRMBLZl4t4vl62v8yxdb1/VXvH3Qlgmww6PlvHelmrbKNXrBQT3xL8CMKXmsYIsIz5WQEw8ya5WmCQfCHu+7RiFnZfUc4nOAZEV1u9ulT54bNMfS5S27aQEDR73xcIcggcOuPdpCwIEBT8OlkFlWPEa5zMO/yqxW/o37X/kJpEqVmYwVI6JRz+Rz0TZeUzQyE45W0PDbhQDRrPZpHr4FuZCJCoM338D7VtXhD0UOwqJyVe2f6UFTSijVoaeBCAYgcDVjku4cT+bPXgryUzcSdNF7ENuiyHwHjisvGmfwxkvzRuyyQjZ+NwH1JUhrkFRgivfg00E/4CHz7n4nAXrB7eI+1Ajdh28XCzO+uEe87hlHHORf97f4Wy+YLa/N6MLvZauSaa2c/8D9PlRVUXkuGhSzCHb8v//5X9ipNix8F97WzETE01toxvjI7krB20XtdgwacYHPBRjh9wRdo9rdh/SKaKKBS/j9XLRVrJxjWzcOGEaDoVMrQDoxu//J+R4o2CvgRZGozEnaSjZpkdeAdpy1gL4CW13KJEGD7PygHmCqeoqReDSTlfoVTqBBVsgyIJBH+ZQe22NIxwZHCchoNvmEtNaQJKzQXHnMGHrj3xF3Qg0BjgKmGSbhm/vbr9Nt6DFHA3g5mZIP4EYysJ+AtcyBG7VTMExHtklaaHbEgFIT3uJ+mtUaFzCKsA88eEtBhMUqzTw+KsbTnyM0X8ORpYzTJgXVWOPwDUxAoClKw3ohaJ/Z7YbBEhoo1PS6kvleeTTD3/TUI2Lh7cqGrHHjSeCRYGixuXPKIktjmmDe0G8CAz9B1BzUYL03jJhNRnranNXT5nfoadOrZa+Zi/vmnng2JzeCNDdIM2JBcPDHi7AGN6QSr1aNR28uKuJfUIzA1DnooetO4BkN4bR5CiKKzh5hUugBdKQg2Si/6N302Y8O4JYPyd5aHSUAzfeshjPk+BoOsZvQgCirJjTjNyzg3bLr/uzWKbyf1+3RG46tNn7PtqxA6fM6cFr4Be8+XjxYekjDY5r3S0c7axAsZLPUv9agewAgDVnx7pCuJgxf4ewgdku3EGdsDe8x1qvGFoE1cK7tbZYe08bYHYtdiOghFUhqVggeHHBBIDX2W1A4wlbopTx6sxzRUJRkycI1n0PDr4dCx4Teh4Mp9+Eh3dis0JBWtkSjwhHu+qX4pJ5DDjicZiW3tdfMDDFaJ59DF6K4B5C9JDKcczuYqI6jHUBYU4xL6jCDaNRjwSSG3NwKRHU+By9FirOeZuNmEuUe8BBrjx086C5SzQ9sJhb+TbBYLK8HdM2H/Ca7runEyITXR7mSVcQr3DFYdysztChJhMrb8wIFN5AlhHsJCrHfGzUzyhv5V+PZJNl6+NDmnzorSOotPCm2Ph6+gvhZVRjz/ZaWpCiCtIVUsD8GMHhVIwAwSeBCyBmaehI8dtNbNO09D4z/DIGVyYxWM5u6GZC+KJ0EsBMxS/wuaOnZQBQY0j67GPo9uatUuOgyIgLvrqR+hDnuaitgVtk25iUToMdoBvwlBmdm/XEpCogMSu5qAs0vg/j4s8vpW4S8dFdkoFxiTbSrJBhkd8WMcjVxEdiiyAQhKxM3Xg2xIIvRz9JRpYvxVD+KT8I1ihQZCXFX5hfsSQyEbejfLzr+xUQgIrc/SERkI0VdhfWOMwWK6mX2KJ/raJmEv1TgRZLmuVShS47v1bKP4OrqRUgeDYQM4WAaUypIe16ES+P78rJ3jjjHDyCA974eY/ISkERIZQvIuz7982+DmMUkKgBS04d+GhJkyBKiYudB/J/rXfoXIY4GoJKe2w+6oh8tIYkDhRWfv/in01xMwl2/I79/x5ECRLw8RbiYqcPUCT/yc1uW7JPpdCwg9YqyfOHoxTaZ/SyTh1EVIAL19vp3hl4btTBknFYakk5op+bxvv3J6MQYhyOqM1gxATN/+OYPEctkYS5dE1XW+endPLffL8D80PA5eJQP5jdl764/pK2Th45ItsOey9yISkDFgOzqAIifBj0Y65AFYcFM36sp5xAOZJdgbn5faklVA51fXgv7HLSUMel8zOH0adMMvTnvzdk3D1kFCUHDPMUapvU8Ghw/JSkwnQsXrPVUC4qKTzrP5tlVCuSAlcPCRairFno/mrBHP0cWMyLjGzL8uce22L3hyQHXAQJwmKrytEpx8SeIi/IZuWP4OgbZmx/e/uT9i3CA5iVT86Oe6NBKkFKMNRLn8YDlJD5frIZhhIcCZPIlyKxyQ5mrKUJJ0cnle5kXeRoDWLJYmHqgH0YIGBHPj+kTQi3y7Dlw/sboQ+iOdSvaCY1Q9aASDc5QA2dNOHFJDCGAn4oPulKIsRKCmoHfhHALS4ok3DMUboe0puZUtS+rBVWbe2uXy4Ag57O8qBvIefEnyT1yicbxr9QVIHvmuGFAVjzCYemkd4VJMaxd4vp4JeGfMj5G9e3bTPFa4qW/EVToJMFjHhPnQstn8cQ+tUPd6fJEo1H9WR9VI9E5hFRJAs+V1B6pQTfXJ4fH5SZ/kAOYtewCmN6Ipc0F6RqK/TB4vZCoGahOUsCR6dQxPtDhWUdM9QuUWytnIIKHWCXFY75VElILsd74NyOUwVxhdE7vhlBOkkyOApSJUD2KXyyqqDgY0hmsORqkGTMOxL/xVxgVZir3aIrPb2/M283YZ1AQ4UFUUP9a0b9HOFauQF5fL32ND6pKqAt2FuyJeptAKgTj3q81lc0axOooowdV1RieLkHkZNPWkJGgLqrEFZ2p4FMdmg6hz56Mta3op8I+kBIjIiyzgqMKbcWDjz99+PnDL8Iy+f3PU7iWdbORGRu9IAeui9PJk1SdbALCCPT9oK8OKoX3FBB4zTshC5z3VmFJF8lq7d1qXV7gCtSgcvY9hO4YxWOMxNHoHbnHNG/T5plvRNqaTOWvrcxwrM3htNJdqpJAa4BVczY2l86YHTY5djtOwAHtoU/44e2KDFJ2YUUXVLXW4/M+EpzmO2+3du0hyJxGPhUvfWLQsvASOG1PhgD14GZQJC25whTR5UX6oKKm0Pj2+1I2bF8zvAgNFpf83986tIzlMS4QwKkaDlt7F3AUw4gUNzGFdTCuEZJh5xWjy4Ae0zvSbcsqE3knppjR4ZsDDkvmE6y32aTVs7+GGbHMAmVzS5vxF+HkQVxiXu/mfy/vAFPeRecZkChNhnZdwOpra0H4kjsPOL8coz3IPS9i/ZG2JoCMtUlxnSTdgfbXHIhY4c6pFlmsnNC/0UttRLWwkHI/R2QMbG0UNNJF/Jg20BmDxVZH/hWFo/TAmmsLlFFmC8qEr0e4oZ7Z7XHXhQ282CSBGLpiUthd7tiv2adgZGDUHBwLMtiepE+8OzAjAfYtztCJjzyTWfoH7MnkNlwxAeB0AOtBHWUzLOKcrO2qnn3tBAB5A0hYfdnMpoD7X4GuadjLlMAGfz6dMirc9FSMXgzYPFIkOPsx9edxHsDRBoQBDCUC5Mkc28Vax3vKiylgz7EkOAehwAv4em5wn6PLdo5pTdHi0AjjPqa60+asVLVXFQVrXmTZY1vVtG217HY3+atZZ2+a7mwtjc5op97qsiO+sqoNuzhvsv4a3OiyG/QgAgwWXXMvLm/4XhwHrVtxBIP3cPEnD/8SPPCvP77/l+jDf/639fTDX3rlgHPYg17q7NxsbUY54DiZPPBlNoCBzSHbP3hL+XXt+VyL+PxlcKNv6iXaX+iK3maYbGAVm3IoI6hYJ92AhyXja/Df8eroM097SdNeblaLV8kXd1jGpnW6FtCl/31OZqVIsDVhpylanYSUsYTQO+kDKz1xDQs343hpBBudOS8/deHkvVeTzhgXkPPpnC8R7J7uMfBBvJPtg07mlxUEt55LbTmOiU41dLxKELusrQ/MqDsd3qoBlD85P6m4wFtkSZXNGnGVDTYsNI7cNfgipxC6qtoSC2aGYRz9BsMLjQJrzxhgdkmwLTsGbprr6z/Nt2CfFVtQGH4dXQ9MU5+mkTvVk2wfOnVU2uilLNIgyxjRW/qg39uKMPSZoe0zV5MlhMFpabzs47rr2aHf3m1hw09jke2mmavg0+20Z2DnUT43B0g5KIZFkwo6DOe4k2nmbNUOL69j2Ia6peDscN9nw/rTBKs9bsGiLk5ecKVJHxCoqjmcz7yCldTvbMWKh2/C21OF67hwGdgpCpyWmmNg96Frbx1QQf7vfIwwXf3p74X+oZBA6FazCdNzqr4dhuc0eGB62F6aNcFpSgV+ze5xsi22ME/DcFcPaj+cbD27J2pQ9VgurvMSNGN/u/CwlMKjPF3cqtkt1T+0Pz/WuuoxAWhcPqHnFmvRtERf2MAoTX/72u/hlthmOIB8LPKC6zsMTIKshbf+erX6ZhPIGkF5FigTAmNZBpf6929fU7ALCPMzUPJ61V2UHX5zV1TndrETskjSGLJ8d7UIvnS7z0L8m3fX1b847Lr1CKrQfyV4D4UlfgGB4KFI4GkH0eu6wwa7CTSIOXJAd4t4t6KfAsNS3yqbMZ0UZMWjOKT7g74Zf71YLObLNwu++rTGXr3xCZ6MYUNddsJuQoqCTGshvF0DwBUCRK+MYzcQdT3aWStCoTeUk8KKxeY+XPxu9m2xi5CFAtYSoA3SDIb2LpP7EB0X4l5yBZp8Nk6ixhdYsHkXIojr4Ns3pomnyHdpgu2jYSdscZaWRI3cb+aEo1iIW390JYqoEndgGhtcA+ldGLx5w3HOBC39PHNVSTeejwr4BkbPbkiRaJ/xWDOISEvs8FsIeCrT+ZIbV05F38YRSSngXBAKCLhAZDYIAAz/aNR0bip5jB7hSItHEJZdIypESmRkZ0TF/8QKHbd4TJMu3Nbmkl7M6MV8vrzDG3V6c0NDTFKIcNc0PMumjCLeXU+/YUnfhoSUBlGdBVGdB4HSwLe3kL4RbmJJArk9GZ240/wrNvyBd4dNapDI5wKCgG0BgZiu15rLzEcpHrfh4HCZZ3fbDFkjrx/lzfb6cctVNCkUkIZ20Bf4zxZTjh132JH585S8VvJGba/V1p8vjfxWVS+5bNEmzNg1rKD1W47NUUloR/8d2K8/98KqrzbBjYVg2wkPRNaH0Gjncwy0s5YsAn1r8xHGIaz7tU0V3Qlqa5Y4DZieOuXuGVBGyPDT35TmF6aAWcblGAyltm2aJa1uMQTRbUKPCMa+FG5LIcz8Gw+fvdsZ7uPrxsEKAx20MX3f22KUPuo2D0Ekic8uC6zVqEDP7JRdlNMIGBAl2x28A24oaV37uCSFowlbn9o02M9iawU+CJf260N7XuOuiBgLYlkpMBJYX8jUA4RkXGCjqQR8jpYOTk37TtyqbtwVnz7+9o1F4fb6SFaVfK4pPxXk8mqtr7prZpht3ZF0al/Arwa9GBur9Up7RF7qvwi7W4t6SmM+QAzzrJM807pHFaG8ce17WTRkv6kKbJY3VecDdRwZvFfLvq2PjZO+gswd6vgh9IakG/2nCMBo+++M17SVGDd6cHgx7vS4VAFhLGn3eY7+gLo/RlUPMpdia6JQqxtoI+wxwrev1BAn13K13ehrLDlJy2o7O28g0Q8Mm6rsnknr5C16e3/BGOsuRPN7c5cXRyzwafM9oKdvxVIS7LbpGgYjdRu80TW+Ub/QZv7KH/UCWq103lrvtumr0KdNRBtCMbzYsWf/IaPCXNDgZ4yw0I83/Ng17i0X1lUrZsWNBD5KjIPARjsHme34uxd93WS61tuma16l9nJYoSBxd1Syt5LjpkqPA3Uds4j4YzzFiEPrmLWGmiS7YbKlnQd/Z85qICI3uO8kf9CGxzfs5Tog/j27/IF2IYhB9yVXQQwa0x2pKJVcJ6N+CjjCvCGWOElax7JKLMWhSBEFyLNjGrYELJjm90ZYkU6sw5op3vh2H+XT6mn9arNmE7wZKKx2L5jz1xRuWhLwPbhe7FYQOoDQpw8eI9ZXhm4N59pSby0VSFLsZC1oXq4eSRyCsbIPhXElteYzoaO3M/swjC2QU6MnNmW7Gok58arfbLPezqYEZnLZ5MxpM0b8tE4WqxEDCk3b5gQR+uL+lf81Gqks9XVSpod1A8CrgUhf6sgauomsDKsyPJv98BDeWrsr9yN6dvtWWUc9knrLZvuqAPkyH965X4adx1kpYCNOipmPvtApMpJvGQpqjLpdLhbXFIBCFr/0qnKNKG7mmf5BoZd5IO03U0YBpc4Rmtp0KlBVNkyN3TQmK9TPHIJx9IEc5hAglDyEwRQPbM80uVLsRoSFSDJFavxYlUywCa6YUPr7DKyjzFvAsFIPqXoMPSJjfSak2zj3zuwVpbbb2kw9H9Zt/Hdvz3XpDv6gMaKPKu0QF/mN29AbiOMxcT9DAjcdaBoivMdKASlufJtewj0SP/MXhTtIcWsnKWhvhRUc7OBiUZs36XFbqcD5gC0NsxJb3Pqen32LpYdGQbx3kLWy5DawjDWFG6YDmrjWFxms5Aa0HGsZQLJu7tPOFa/MQg4WBGT2c1zGNxDwgyXxNrAzAV5+jesgC6Dd3RX9I1ysGmPniJIJ0Hfk+7gVTcWTpGcqeLgr/Ptsl7VjFFbzMEnlHvvN0rimVrg7/HZVtk0B+pXGTpzJ9EifKx2KCs41jttKxs/CYfYi42pqeYuf6V5w2EYSuKPIHzvd/sid2PCyiy74sOVm8tJr6mbIm75NmriYZosLseaduWqw486vXgwBoae3LhO3RJO3MDruABjTDSeXL5OmMh19wWlfy/T9EN09/KCkC+8ni7nMGKMVVmsg3cD6HbvMDALE7+yUyhwEC4ZKXJO/6VbYrtY+0c+sl+ip/Ll0fdd1bDppY+4GJPDxAIrPu7jW0Qp9xXI+JT1tzNWXqTv+RNa0Qe3cz50gfuG+NN6BnEXdbrlv/+P7f4uWr91pGO4xfaKmaV4fvLn2gj9f28zlRpMb3YJ3ATSKJ+JDX/t1iIluB1tK/2AP6pABTLvftZhm6fZYvoK4VOGSLY3dfrv8NL0WproTDahEDcbXmOfRia21YbTI4PqITjnw7XpcQtmMUrLTM6Rw/DMtHqiTdaA6ntPlxFcMctx4yW9v9NvNpZPqDp+VmcmrwRjZ0taXEHpFMnWEofERRvPGnxDpz4e8iYT9wvWP9RWcyvfNgTzZN8EQzKAmYQMew5joHhxSjZIwYc10e+HAm33mpfrLGmH3eUSGB+A3rb6QkTdEh0kRB36BqSO5vth2SszU3VoPwyxkNgm7KYiJc1e2B4JTPN8T9cX/w7qJX75Qgzh9/2U1jJ8ywp01/I5R/4dV97wTitj+Rmxdzym3vpc2XcK6KbjzCNPNuV3SItQTc5HKcKGlgxOxT9dbFk6Ya0uATM/wWdvun+2G5WiR5ZUpG5R5eUiHQ/hVrinKlaH5/zuC76o90fgRnyovAV9dpfSJZxhFSRFHkX8HchFCtJkkEUoITaxxZhOCzB4hK0hweyuStnq5YQWt5GWey+2F1AtK41Lv77mzGXscV5Dhwuv3EdTpVQW26J1b0jeGj5eZ/wDBKBODoBBb6P9WJfx2cWZPLi9dXPv2wt6Yu11cHLz55szO2PVukXthE/sLNgil0xhLXcPv1Dbdhvrjo87+o/SRO3fpAxJYgcGb0JoycOjjI8alIDMn+NDJXz4oE4kChBdGK8GsnoDCTgUblKEC/3+Wqa11WwE4Vfo8AkGjqQD2KPdkZy6CAs0TG3d9hbw5/q8AHWvfLsS3C3G7XGzGYnsKh6381MnrixqYXIPOEUtxZe11tw34FGi9C/mDBtf+ar9GMOYTNZqsezroN2iL0BD4+1Qthjw4LJvymH3zoZfilxeWpaTB8fdpKpvAlflqNbxdyMNobZ9b9X1qfX8bwYs+00xzNJyR3P2eljYuvfS9OWd7cmjiSUsOjdodOZr3X+vs7hpuCG/qurncJikuYqQ7dPik6WKv674ZMKZrrqFR/XD+rFDfh/dE1rKVJtVqPu2JmTg4KrlFgleJ6Pwp00QN3ErgR7O4L8/6r4I0YPN/BVHTC6AckReNIiAnitABRhF+XY2O8Or/AW5E1kW9SwAA"
  },
  "scripts/asset_progress.py": {
    "sha256": "857ea80a06c3356a52731a958b2d668da100bcb0bb07c73506ed30806b222ac3",
    "gzip_base64": "H4sIAAAAAAAC/4VSwWrjMBC96ysGQcEOqZN69xRwDntoD0uhh96N6ow3IopkpHG7NPjfV5KdWCltV6fx8ObNvPfMOX/eW0OkcAedElLfEv4leEUrW9kIkkZDZ80fi84B7QXBm7EHB1KDNoQvxhzA9S8e0gSE6anrqeCcM3nsjCUgeUTG2A5bcPId60CfhSrfMPBPjn3YVnC3Ln8uFj/GfngWqbcaWn6KkNUFUZTtAA/yF/+covwvRbkp7jzF45niU9iI+e0xH1cAKodn9ACehLFGCW/AvVT4NPk1XhGU17XUkuo6c6jaJXR74XAJZEiopbeS0L4KVZX5fHcAFhMu1hM41ucJqL6guqbxh5GHhiSKo/GxGS2bLL+AOuvnspYDnCLdsIH1DeyElXCaM4sr8oEvoVW921fPtsecJRIbodRFYmN6TYkebd6+O8H7GyegqhK5YGwcvE10bKtrD+YVUQraBgONj2kNi4l0lXKGKOevGKTHXrPMhsw5eFdOE338MW4gS9wZ9Q6rpDVvyYf8g23ptjQjr5b9A9Srxy+UAwAA"
  },
  "scripts/audit_audio.py": {
    "sha256": "6a6e7ac3df0436319ea840de67a2dac6174313ef6b924021faff92e90e56f5de",
    "gzip_base64": "H4sIAAAAAAAC/5VZ+4/cthH+ff8KRkBh6aLT7tmtE6yzBa5uUgRIncA12h8WC4ErcfeY0yukdI9e93/vN0PqsQ877QLWSeRwyHl9M0MHQXDb5boVmczuVC7KOleFkBiqxVbtaqPEL+//fvNWZHX1oIzVdfVOVLWnK2qZ62ovaiMq9Sh0tVNGVZlKgiCY6bKpTSuk2TfSWDV+G/ncf/xq66p/L2V7N9uZuhQN3gq9FX7iF5rw77bbNqbOlLXDyPPw2nU6n80+/vzzJ7HiVWGa7nSh0jRKjLJ18aDCKMFpVNXa9c1mhrUJbZboyirThotY2NaEzGEugrauCzu3rSpT9SCLIIrc+Zx6/KYfbz99H0N9VV3pTBb638oLYeoHVUloo6e0d/L1n97G4tHoVqUsOlOarupJnBXSX+utm8IBaXc+SE9j1G+dNooHZ7NZrnbCyrKBmLaVrQ3dh42WM4EfLPG+7qoWRqt2usLO87prr+vdtZHVXvmlVjzq9g4TojVKtmRThQFlhLRCirKzJNv1bx0EbJ+FzeAYbGTawrFNM95mJcDF6lzhbdcVRWqxkD6G/fG+4HWNkvf4sOAKcWxX0kzi5uB5AirvoLyqP6MTiH56B3bOYxJtHduQyaORiH7jpl+vxM3RFBwacnZqGDwS44i8lHtMdSyT3Fq/0zDt5SjlU0iv8Ug/0kyEBGtmcMXPgaJX29eryX5/FjdeIXzCUZ/HVCvx5vU3b7+Z0/PbmZPOGaNQ1eAQPGFU25lKvATeZ5gwWLoFsQgGjaV+GeaGMcz7SRIUMyzvkVp7AlPSSraQ/Q2hNSpgPlV0RLY80rwqrBIf6kqd8PX6STvQpuy7kxP6yd9fsjMyawFi45q52xbHmOy/OOFUKWnSUf1TNuPoZU4HH6MMGj5ECXNikXdGEpMJeqx+kFg0hu5fVQagFW0tdsDalsOiVNJ2RpWK7FU3xEEWxTPpOrujyEXciq3R+V69ssAzQHIN/7iH6VlhIBlCF4hdw0sIixICc6DHALAJoCi7T6GmpmvD9RR2wmC3A9VWBRFc4vohwFMZUxt6ubaqUFkLUQEl8AIMyeUiGDUKirv6McX5jVY870hXJGuWVrJUsffOXdn2r9CVirM7WVXQa7zVrYUTGu8DvG+9oz8kS+BwnNQcbWLRqqd29ckgZiMfHGUpqxyCn0lVNmrvhKpq2yK5BRcE1FP+NFDKhiYWSxYUA62n6E0cbWYetwZTj0DVHwcxvQ6upfFrKbEwd5kRz9fBZnZOzSLv3rz2KsiWkv42WZkOg41u1PLGr+7hfuXycMJPCB5E0+mEks/2uVWf9wd/DK9RCEa5lJbUJkfa+GolAqSKFkcYBe25M+dH2YTDYkLzM5Q3UiOA/kko+T3pPtwFH2qRc0jkLp6W4oWMcAhOwK1UrcxlKxGf5OLroHfGzXoBh7i6upgx+1At4GoWiKGaAsoZo/EHDb1TdDkCMgXWtThM+1hfw6aFUE9wSu1iE6anDNrCZ0V2V9eWglMyYFzDSfeU+IdIhF/kOie+lBKR51UeDoUICG0QJfui3oaBhctSkZXmGhkAXqozm17NjaL6TOVzOpU0mkKaYyE6xrLp7149rwpZbnPJNdeSnwkpBXUSNFC2EAWCONYuhIb0TLSUnceTHyXoCaYwV+g/TykSwwiSqBaCYJ8OcokVnGWvKkUhngfHGdyblFm4ym120TfINUZ7wJ/Utq7vxeIb4cyII3dVnoh/dE0DsLy+pqprTmznbT0oD340ojU8oKtC/IuFc/rRE36sbAOM66vm2/lfPAmqJfhIKe+VkG2rqo7D/7qusGeDXbR6tKQ0yQVzDujJ2to8D24w2a2vYTE0KV9jN+gPNI732OBNfwzqIQsrgjPXOLLKAFEj1bGdvjqyE1X9a1Sqjqg2eq+RiNLsrqvuU0SKesISchTQsJ8MbNcBhhCLG+K4vonFYvOFqA++fyJdQ81ftC92cNADhouNBwRdsU2wy2b0WwSiS7v+YLBSyTKF50ec1JM9lpNq6+066L89sk6QTCPSoTTEaDjm+FADEziLRxE3TGcF7JAs+umF+G7c9TsUeYuT2Pi8qqQvqkRW6GZkAmnR5LwZ9ON0UpDaVsI5ye7VS6+h5eJNfkhv0xeWl2QyHUFbsDmkVqncjdMbRl6NFlS2K8h9x24mdJuwyuMzB1NPKutow9TqfQVnMyqIJrVz3ZlM9eHg9uQhmOckALwR/Am05UKyV6dbBH0D2YHg+oHSPocXUbjuLHRE7OuTnVI3G2x+zwDvHSC4VXN/Dqpa9ip/R1BJB2FwQV7A+UZVTCxCPR+1T84wcy8OJTH0gtTQTlzOyEcemqQNBuYpQtM6D54+h1wlj/Ih4OKbkZVqLue63BoQPRGgeij1E78eJgrmoEpk06gKSYpN6o4Ye8Fj3jIeD+et6VArKe8BfKHvwzmrxEicGgmnvvcVsKtGUHWQVxCLwT8DP+oQzKd9boxXODpokPOpfCK7xpw6KYlhcFrkThgfQSAXMtNJPpYNI1fMvxxOc2lA8UUdyHrDzVNLO1EuGu9CxK1PBPCLIQVAdjHJ34ISRMwWoJsGlzVYiTaBD4u+8YZRM9lZ5dgbLoPaGjBCFWWBNHIYYO5/MQuZ21nzHOZiwRh+jnaxG7gI+BO/RJlmcrbJRcqlY8/FvzToFkBf5aTJIaW4CbRaJdWxi2jOVy0Xihm4REt3FrTaSewZ9MPkCB4ylkeNWK+XAXovs0eKBo6R8k7Xsz7H1fC3XrfsE4cJygKEunJLINjH5Zh4xkA5ub9gmKJq/bPdY3S0wN3ZpWgAbt5+cdm06Rzruv5HvdkzG49W+Yjq2x2PhcseMvtxd1i6L+CXz5SdAefAdHpQLJl+UsM41k7pXmqKab4SQDfjIsgPHI72oPTLW68vXGNsqMxccHk23ezzpMuz8/uLnuPl0wuRzdkSOjzdDekqvEkWsbhZXF2F12/mr+HOtMRBMPFlgJne9gy7OondzRrVhcjRgQulF44hl6bJhV6chx3SO2CaqWuH4mccp5eV3i38JqN/rEZHIRlW9KCbrad0oHiziM5Ys+MkXUP9QHhqRc/E77Vin3Lv0TDqnHV15LinZ4tOnBWaW14+yToY2N5r5KucbEv43F8LQmt94/luWrH3qRqpRWojPsgP8x+rXXDSmhDErSchv+mTIm8eTcCQEtTaJ4uByjEYycaL4XAwdeDWJv5aw325Jf8nue+k3JBvc0oYZNLg3vrCZOhwz+vsmHIQGlo0uoXOdIu+xt33IBcVNXzK0w0dDV/+G75wcP8RkNyaPZeRv/BMmCubGc33WKs0zessTaPJykTmeSr9kjDgxg2Stc+NWlFB+EVa6OWcFgRULfkl/IcWWV9DUpnBp7UJ38zvJu+WUzQXlRyvJzcFkyKnZ0F36j0Lfj9l4aqaMHAe7/rOQHzN/5mR0OOP6MXv1NN6+e0mOi53LjepQwXApb+uTr1vDJXGoCcJd8F7AhMhbaHFCxGtX11K2a82B1F2hRyIjhL3q81yf7DLIDpthph2GiPHoerOEAjxEZHIhV6rc8AhJFNldy+rd8DboivFVlspSoWEqe/xZpUbVvcW0rn/pDpqagzHNvWdXz4AURYSzKg6AtxwjxQGH2//FcRYtO7T2iaKw+CHn36+/TTs7TZ1VBey2iaKLuQQr3MhXnjXg7tEX73w5utXk3QClSZv9wfgnhYoCdHZXt/EN5uB9AvX27T09R+w1APXsOYs2cGqQXTZIn+j3NUQ5Fa/ym7pBT1LzZs4+I+f65PzRjj1X4BfH65uj58kbCcrsL4MYEfEn2TVyL6utnreVXmHxio5OefEf4xs4EDiXok7aXUBM+87PEqZ8D0PHDTl2+Y05SuoNCU4TFN/A+WwcfZfhwMaGiIdAAA="
  },
  "scripts/check_playback.py": {
    "sha256": "5f8b4c24f11d03cabd5d059b3c63899958d88503e50f93755a03878bbbecfe84",
    "gzip_base64": "H4sIAAAAAAAC/31W32/bNhB+91/B5YVyJst2unWGAT20XYsBxdCgCPYSBMRZOkmcaVIjKTtu0f99R0qynSZtHiKLvPvu13d3urq6+hu2yJQ5zBTuUc1dt5ltwDnWKjhuoNiywuxasNIZ7VhlzY4VUDRYMuhKaVJ2kL4xnWc7U6JiUldoUReYXV1dTeSuNdYzsDUhOBzf/yWsSYRqwTdKbthwcUuvoxA50lpToHOnk+PpZ9fJcjL5/OnTXR50EiEqqVCIaWbRGbXHZJqRSdTe3S8fJqSZBVOZ1A6tTxap8zYJ6nPujVFu7jzuBO5B8em0dy2GNzr2+c3d+7QAbbQsQMkvmFqEUrTFbojDmj1qoLhHDdfAze+v04OVHsU5YPIkWIpGR1GL/3XSYjzspfAxXIhaSV80okIsYyUGeQUenRfnukwmkxIrtqNKitbiXuLBJbbTKdWl7fx0PWH0RwW5peyg3SPzDZLZoVQpA+9RdwTLlitWvqUDXQ51D3I7Q4lhpmIncoCPEA52yGqQOlY7GCmg9Z3FPEScKQOlS4Incz5czIdnFgR4KBel0eOjTyjvAcDCIX+iQJFBp7yoCM1nB9jzKCer0VZWo0+48+A7x6e/5KNmyZmxQx0SgqWr4eaePwEVvQx/6PPUeyEdsn9AdfjeWmMT/q4n/aBIz4L4blmfYVY0oGuy2MfQH2a7bSltMtAwv7MdpvgoqXJmm38A5XCIeKxD3uvN+enkHO4l90Iw6bl6FLn1+au07Cx4aXS+XKShJvki+6O3EN+Wi+vrZLZczW8W0xcgT3CjE8RJ9GK5KjfRi+fw4V+PdG7VjEqX3F8SOuFVtWux5tOUz7RxvpSa009C5BhSG17iieSxK0+eBAVPF8sFT0+F4TOoeFrxRtZNS0zMq3y1WLf5Tbo3qtth/jW49S3gge0BQ+8GLCjo9CbcFGugBzWvcMvXCi/gg/yYgNHEaiFiLmIapg8p8aDYxnJORyaCPiYKdTIOhRFDU39MiXc315T94AeriJLhlAYlS74rdPos5z/wYfozot4+n9uMfKt941gpKzI4sPQ8mpIz8cKA6Xsz/cqd6WyBY3usL1op5ZF1QmrxVjjqBV06vn6V8pElfE00OVftFKgI5eFroiaV/9FbiAdiY3wjwlQjlHBwjlw0X/h6dYmljUe+5n9KqAOfZMGMVseM/UUas6DSTyx07ADaU8/GiUVczNhHxJZady9pTvd8YZ0eejdjd42kFBlSJBP9SGeuRZqqljLnPKWGIgtDhfIE0rIaNfbhZvzb2Mw0YPQwAU5zWepkqFncgzYfN2L2xtbkhfa38Twp0RVWtrHPhChNQSvtQi+DshQwqCR8NqN246k/thjX4E9FyaVnonTv8kE+PoKGS4ZQOh38jD19XquR7sMpQxpiz9dRcjkFewj6/QJE+GyIEGHgJ3z84hD817Dfs/DvN1rkDT7er1cPPeoPl1wfu5Uh2g9G0XBe85fu3tJuk7regmZP2i9lT7svZS81H7FH16Q6sKel+gcsWofooc34E1PvzAZY+DBhR5IK2xLYDhU1q2QNjYq2MRrZRiogdcp+KeGSxsTLIxDzIs+lS9mmC1434KTqSV1JDeo7o3eyhC2DEsZvMSdTYnnZEVzY9NAFr7sNNCcsRY4FlMmEKiNEmE9C5DkXIlBXCL7uKTz5H6ueHRgvCgAA"
  },
  "scripts/check_source_roundtrip.py": {
    "sha256": "0e3524dbbcceb34eebf1de4ca449b6a8cfed7cb691e4545fc7686dc9ec6afddb",
    "gzip_base64": "H4sIAAAAAAAC/8VZbVPcOBL+zq9QtF/sXeMBkt27mpSvihDYpJYABSS3VxSlEmPNjIJH9kryMByV/37devF4mAmB/XJUAbbU6m716yOZUnopjCWN5Kom2/8iB2efB5+O/ySmbvVIEKFGdSm0walRPZvVCilIKdxwRu6kndatJVXNS6km5L28zCmlW3LW1NoSricN10bE96+mVvG5NltjXc9Iw+20kjckDJ/BayQx9yY+tq0st7bOT08vSeFoEsbGshKMpbkWpq7mIklzECaUNVe711uwNkfWuVRGaJvsZMbqxDEYEGrrujIDY8WMiTmvaJp6ZXhbyjqqMgKbKDnilfyvCLrqei4UV2CZSDMVo1vWNsBc8Flmpnzv19+yOy2tYG63bp1YIDGbVNKOpmwsRHnDR7eRR8Ut+ICBfUF/2S3SrbJyJphQc6lrNYOdxRUbpvwaI2zbsDD9mLq5B2cF5l5fBhsaCWN6lKyqJxMBtt4qxRg2LNCmCUxk4Oemtelwi8APOPmS3wpip4IYDrLeEams0GBM0hpRwpub6/Z606qyEsuI4aULmAmXykUMMgVhoEWBdssxokyCgtFfS9vkOEvR6bxkVixsAr7DxT5ii7AgKF4O/HB+x+dehhwHMflE2IQay21raPqqoD6oS0pqTbwfE78YJv2SK+oHmJ+m194WTnUujSBfeNWKQ61rndAT2DnhirRqNOUKTIoJ1FTCwlPIH7LcFvWb8DbOZ7el1EmI5uJStyITCwkxUt8WR7wywlP3AzToGryEJujtHGKfg2FfZ2WruZW1KnZ3MjR9ses5zYTlJbe8eIgWGXYmpNmjbQ+DdTaKSrPOJv6HRpl0CEKp04RJxd4xA1ZQJYh6vbaGG4hk81hgzF5YNpYTM/BU2zfciBAWa4xUbQXs5cKXM40SIfTbESpEalXdZwRICLAwZCKU8KrmrspxVRKshRoeMGU4ug9qIcQ3KeV4LNA95Pzk95BNJqffnPxl+vdsFKLO6ZlFe6ch7m2rVeeEkHs9ZZM5FxmWCQiHZQJ+Nr38Q4XHkDT29R6xUBiXQTaGeBaQdfC4WtW7vAvZr9pZc0+4IarpD9taj6ZuwFUOUIVp4Ta/LINeFnOCY5oFfXNVytmr4jWmVRwCfzbiauf6VbGLw+gB1eTSjKUCyyVxpzmvqiR9KskuVjYUDOmqDxQiWQ4URgryjKJDnqF9vc7JnQTv3vWkYIXym4beEfbJZkjaI+q5DQySB15+FZqJOWMG5vmohuc0TXMfQ0Bu+WgKDWvUtPDX+Q3++zVeQ8glAZ6bFX3TJmAlbiAsrJy0dWu41vw+misr7X0jCiAJcZBmfm0aPRJ5Bvvvgv33nrLu4aIRI6xX0Ci1qGP4rGYRXYnhKCLE8F2tb4VOHreOYyjuLvvIl/1DcifkZGrN21AOxWY5xjumViIikRDgXRjXJg8NMW8bSCSRPNwO5y7+b7M5dqQNfRPMDuExM0mKFrpFqoR+Ojt+t3/wx+HJeyh+8HJwenL08ff3H8/h9cMR+/D5HTvYP/hw6F//PLwMr+m39JEmV5H+9Ojo+OPJIb0u6C5dI7o83z+5ODo9/3R4frGB9LkJ2h9wUKY/ClZVJWIm5GDGy4zGWgY1TZsco7zq0MB+a+uQWaf1TQVAZ7nGUTJfgCP9XGg5vt/6XivfXAo3dPLvt+iuJfV69OYu9LdadqgmRvHGTGsEd75xgzA7lSZ0Z4hxQ3iFat87rEM7tZ/dp5bqrXa6J9X7hDYnM67kGDF70C5IjxC0iMIBqJa1HuwfHG5fWNFs7+a/+mBahaxJh10pH4nADFMGs087HNd34o+3tuLPK4oBZ/r7kuPA+goQl7FSeXBwnTtkYDDJE+p0bGqAlGYA9XVAH5VeH2qd7qDRRp6ZH/WbajSwAyuKWcutKzyGf+XDfufMiOWq4XiMGRx/GvjEQE7ZuGrN1CExzwyUKtbzw5d+iFIL3oM+lOCZo6fk430BYKlqwG/uKGMYVkQvI7d1Uoq5BEBLoUfQUNtDfwnlPcejS2gXiBsyAC+iMGPnASf6OwjNM6OBD814dcfvDdsrexustZyAHatiradt7EEoP78MCexKT7GsQjmYQxlsCyY/F74HJqhs9uafOzs7aRKFpXmrzF+tEIBnd2JfWWkCxYOHWFbfLyMiePYwoAB0KJxXfVvEp4iEYGLdk46b2yNkVgu+MADco/QX4YGmxj4pa10gJPCRkTgDpLlXhpUA4lfW+PGiWxoQAnSlZ2CEDfa5wnBhISqhg2wGkf3lvgytrnqgQV8HFOiwArUDzPDYIc06ClvSoVcuEtgyWYUiv71J15D52g/18Q5g3QU8dSzgLUbpt44BRLhD2I9CvLcF5haFaH/aQoBC80sfh5lpb3xmHB2f7l9CqEDEclvQf+9/ob14+CG69y/p9+ITzxRdUXhZoG6+1YjM0iWhv82Asz2U/hwaBZy9ddfXRwI7wgc/uizMfnZWLQCvaocSZotuNvAoVhf3AjEQ5FgZ4XdtvF/O1ifXy9s6jTQcqiyzf5Uzfwru9xQ8Q0RKhpCfwUYYKLKG2l1fPfdIMJ7VoV/MhXMM9oVW8TmXFUp766qJRuhktWwQ5rrKjsdGYqBqg+Pi6WMMxxW86+jFymyR+9Nj/riq/EQOgNyd3/jIQt1x6BdB/kAttYms3ZXKzb0jj8WSNLIRFSiTv7xUhbLTGSzYKqQFC0Uo1K6/W4mQ6csr0eqq/08l8qUHU2vNQm4OhHondTCKDjFRf1ijept7ukatWOHlNaoR/LaY8UXi7QHbxxfs3DfG9Wo4jGY/b5xdwJQDgAt3YlpVDLIb8KjBs2wny90jzaRKdvJ/ZPA7QOHuNIUPBA4UgsBwR9/dQeGa3Z0M/92FI+n27vXA7dNpcPdMDX4ieBXCEX5rvPgIAtwFDmr3ligBiJE0cOrkjy+BbE14A5pqwPRyAqmdr2uK5Tu+/Px6DzHLwP3rSFeu4jbfwT0eZRjLQvnJTlR88Ndz+KdX1PGuBC971s2ymt6btUlw7S90JfDWJntKpc/TKrS+cOT2p7WCdpedNPNuCZylmrCOSbYcWxfl6ljoCYyB/xCJM7bWVt8LNeH6lqshWTNsRvqooD/ez8S+IzZ13SDoqK6Aehg9uUYqFiPRWPIOzkKH7tHFoCECm8zwB/Yac6z9NHPExZg+YJYn7i2F7aN3GPs2JA9u6Fsv111D88c2bAtVDxg/H6r465oZ+CS2CvfhRhfxE06+ryctXpicuXE4nZgRNETnMsbKehQ949flvCwZD0sSur2tW5DmKhd+vXmSFFR9LqlYiFFrBRxhXBoU1EC8AEQAn9AnF/qLqZ6YbCqqZrnbi89nZ+eHFxfhQKMnpgic3D/kZZLu3I9vuWdJpHFY5KRWoucIfw/Wo1t+tepnEt6h+WNPqwpHDQ9L0k4WfuVwhXXt21HS/5DgWcDzBhb4EcaxiEd64GkoVoJYnzrUw+gv+P0txz9voPNPxeJquLt3Hc/Va9+Itl6QNVGf4MpennQfo5Ir7KKrn7HAbg77fu9bILTopZeXTTi9zoSaFxsvAjN3Ou8nNJhn7ZR5drx/Qk5Pjv8Ty83yvshI3rwlXVQSYPuVV1xBberuG1YrzNYWbD9md1FQxjAFGaNDn4pb/wMaiPa7oR0AAA=="
  },
  "scripts/compare_bass_tone.py": {
    "sha256": "cc92b244e3f2875518601576e124c0ba4cd9ae0006228b4247e49e3f1d10e9bc",
    "gzip_base64": "H4sIAAAAAAAC/+0b227jxvVdXzHhS6gtTcnrbgNoqwKO4w0W3V272U2LVjWIMTmSWPNWXmwrWwP9h/5hv6TnzJ03WTaaNEXrB4ucy5mZc78MHce5yBjJaN2UNCEsYWFdxuHRNa0qEtKijvMMfrMojmjNCN3QOKtqQkmYp0XCahaRs++/OSVvLk9eEpjEkjhj/mTynu7INSNNkeQ0gkE0yWGZOKtzQjPC7uOqjrMNOcsTek0qVlWwzoIDpSVzfd/3JmuYWp+8DO7y8oaVS1wh+MPFd789/25KStbAJBLXFbllZbyOYYkqb8qQzcomq+OUze5YvNnWlT9xHGcSp0VewrbLDcCvmHoP82Knnv9S5Zl6Tmm9nazLPCUFPCXxNZEdl9ghn6ttU8eJfttV6rFp4kg9/xAX6zhhk8n5u/OzT9+9PQu+Pv34MTg7vfz09uIDWRJ3QuDPeVsBJhCba0ALK6t6lzBNDcKpsWnimpaEhhxLGZwxq31ySqJy52kCFkkT3gCYGtHtCNh3cb0lYcJomewABXUcNmItWtc0vKk8OAruNcuBwiUsShG3QHNCmyi+TpCMBSvJlpZpnsVh5SvIp6SKkQ1I2lRxCMvjRpEDgNLAI1leprLxSO6+ZBsgPQBb50mS3yEP1FtGiphm+ZeVgitWgmMBJXEI7iXaZTTFxZ3JtIvOy+/OP55/Amx+FvOrcMtSGgBrIF85C3LsESeO4MFpcbgDzWkesQR7aMhgZ8XR7fEr7GSOJ4BF7DYOGY4Im4jilKJkYSwhO5JNsR1Fo2yQLtihFqgYiypoWM2vJETARUSzkAUV4Awhf+XP+fQ1Kxlvh33glFfYnK9xiSio8yAsGmj9VDZMQpISCo2D/OVNHiaTScTWIGJ4bhYFsPeK1W6Z5/V0IYA4zvcV42RQo4jEHDyJ8a95N5dhLdJbliBXsPQaziem05SRW5o0DKjkCFqiAAFhUHDEomQG286zdbypZi1i+CiBchJfFKZhk4/rVS4C8kuGiGD3tTudknjNoftxFaCMuVMQGDgICrUfMVbggzvEKFO+CE4X63yxJEPDBHrwr6QxAP49nuy8LPPSdc6VaH6NoinhRPEaKFgRrjna+DyG5xpVXvVajC5vQUhqztmgylBY2J1CuyM2WDKQ6UwCl3SUBJdK0W3rSE91Lwe5wdD7AhYq44iRPAOVIHZaxjSJf4C9Kq1/zdZ5KfhC62saQScQHQiIrB6C/gWFSlND7+u8ySJa7oB4DuhEH1TuLTyrx9Xx4spR6AeFQ0COABRKQ+8wsMKU5KAqWu1+CCvUrlpnitQ73keqS6QQnsKyIBaJmhDkvlo3iTBkINo5YGdHQpYkznRkqxJHZo/YLxt9aIsLlzcnLFNjp+Q3S/Lq+OW+rX4vOAHELC3qnaYEV8+A9XoLthNAkBAUJAXGKiu5w1wRdEnWX375pbZ3QqX5WrMQWmFjoBsmF2W8iTOaXHIywvxWt/8ty1hJcRdiwCRMEJ2K/ZH7RYfbBiR5DVk2AHBxHQQuSMPaIy+ADcDkvHhxc4dPU4MP0bDSSu0KFbp8+aJ80OOqBvSOO/U14C7Iyf4zANj+/ieINkvqOkxXsiIBqJrtPIPyXxDnz5kDP6bveCrFtWR/beKSBevi5CVnhKZk8sSg+hPcygofwByWvAUNpxznb0BPO3wYmJHVFVd4+Co6rsF0syxypmQJolbs6rwMt8EtZc6VYlq3BQlYt26AYVBgHNkTOYQvbA3j9jCI6l3B5FgO2Vd2TpOB893wZGEx5UpoNI2IwGm6EGi2c825xlcmEkcDxxFEw+O0EQmrTad79fh9ITT0kD/LTzOLQCNETB/UUbTdCLZi3Kx5qKbRd42AM5q6aKDlhQeKk4VNzZZvKFgmr8NUyw8g6EYpSzbtWOK2oYmAmcIadDYeE62xWpUcz//593+czI8q2C3YlDCJC62ThzcIvJfSwk1oeh1RUiyEjS6mwOvgjN6CPfXIyNmmQuugUkcLvFrMUVZXoPj4BDTxVQhKsAb6cB2pm+s8T6oZKKU0YOApOFPBrFwdo7eZK08bYg70NblNMkOKEsQuQxWsxp1dvH//9tNHj9yVcc0C7sXr4RALGHjgE0bBX/JrOE5CrUEpDTlBgDh3ajRMDIAjUjDPLWgYWQQsu43LPEM/zxrf7TLzwHo3RSCHdGeA4G7tLaO4BCVT+lrtfsvCmwD7LLA1eEMpOHQ5GjB750m+2bBo0nalBj1AqfH+2qCdiGAUosY15EanGH1Y6cuKqSvp0l5p66ghrBxAMPSArwujeEAWVFv68tWvQJuDoFoDMSDJgs6YfaL6cVgkGJdgYeE5TMlHYOlQEmJuQU+lnYwaYQjIcmTPaoBRopbVV50eqPo80XZ/cIQLsa4U+KkemQp3dY12ywzW3cdz8uul2SM8n8wPd0JR5BW5wUQiN1RGP2Cv9ExhGQjDT+ZE6ArtQ3DR9tMbUDIuTsrqasljDeH/BfmN0GOKazi/gYlWqhhiHmQfVMMiSIKdQKN4gJamkCyLkhVjlxTeFWczGR2Zvw4DLR7lHlhD02+h8dgDq9kCPNph4AMD+rvDcBnPPMKX6tiopdG35xqZvJeBMjrQGAhnIYTkYZkj/bZNdgNyXHFe2DTglmQ1CJrvCNfHKDhXKnDQpxvt24jwyZN0ETSSKgy53Sg0o9P5/npmoQXVMTZM/rYh/wLUvnN0lNL7I5CiI8VRQImTOf4/OlLB7pEIdoU5UIqkEwlfTbtYJghCWt4jaYklDLNdbZmF0NblzkgNSLDc+KIFWqBphamGCLxGxr1No2bdtn52pcWzoLZtOZINzfmit3+j1XXaQOnqN+8uTj/pmKQ/s73Cct94Q+mVfMR0AyAvdLzHgkZFDO13g81+oaBAvHZlVJA2Lq7sB/bIbpcD5k/YF0+glKuRqU0TZPFBuhQlKE4I2t6dfiAXH979cTHiCnkAQjhoRHv6Dircptp2lrO8ekF13YO2o+SJtHauYVDAZma0kLVpKx3R5jg9su+rKr9R+N4YIZrhyhZ1o9pHnFfUMcaB7ZsFx+wO4MNZ+ysCu+gx6zyJOMMNomE+nwenAXoEAfoBcxMRAB81CaLSOFuugOUR7nhxh9dGlJwhhQfxIVpkKDAYRwCkEd9CSPSoS3GIERVLzuIMzj2TuxMZ5lAY5TWNEyCdOYXK63CXAsejeySanKsWp4XCxfpsZe16oifzhdCFqb9+orCvH7mOHEkd1rS6CXg8BXYqYZvcOcxEaqgqn0kTlXckDvjf1wkLRHI3/kHBkiGOU2/j7AZOr5uGYTcVA4NUBwYXCoDqSRlsf6AdXAwwjhumuh5shsJYUqF/5YjclHPF2emGiWQRT1DyAApaPPkaZ5pEPpjZtHLtuHGYc07DugFTbkTD8ILIBA4kAod8BIuZRGKUe+vqDKLJYiXMAvA2qVu4L85PBjLMY2Sr95rW4TaoIIgSQ467AxCr4GhEecqlGUZJB4Tj9kAU4MwZX+oAFPQPLcz3qAqW/TOVadinfAcTLrq3ycItcM9zIx2VVtFgRvzQPdqoJwxADgvcY9HTc2AOerM2uMEBj3L/pYosuK4kCrGgUjBR3OZzy5+0yW4ifGkmgNpah2KKANP6qBMNL6iKQZBw79m/o7dALaXHlib+wkLlcu5/5UHQdR/oASdzS3PzAp6PZQJeP9B7cNL4XkIeWFl12kyHiPSbAmukrjD2S8vOe1oqluoB9iyc2UAy6VL+DqtL2CziQxBn2cKRCX4ESpCI1uHj+6AuKXi1su2V2Da7D1lRI3XYOX9E9QUhM0PiLh45mbSBnhi9XDuf0ca4/A3TshkFXzB4WJDPvOnBRhXykchhYLI4sTz1Z8Q3vLIpS6z+n+LiDZJxgGTcWZGeBQxHOb9HvxhcplLUnpcaytvL4JvzN+9OP51/M+Xp8jLcxreWJkTLgSdEm+EObHCER/sM5I2othEWsH0wWTHXS+5xz6QXI7VmW6rl2XyOeoM4PJ3H/7cKUdJ5loUoWaoXwaROWHn6EsDz8qCPgntC1nIPlJ84gWnLJlD/oHymkOsn5DX/VzKauSw0dfwFRWLuMbSWHHUYZMoHAx0Jc+XoNJBVRlG9jwZz7YHdfBef0U55dUshakdfYJjQv3iw566CFTeMWBAycvcAp9pBgxUfnMwfpo+nZLt1VFOFQPMyw32R+ezszbfkK1VjVcwAtHUMLbTjZsjRsZLKExsN3G0ueFro/mOE7X2Mfa32lzYV3j+C0ArMSDt+5zgCgBIzj0Xs6siBDttHkDAcuA8F7R2QvegdheI5obs9wrosI0Zxz7svEAOuDg43LKJvp5l+e6KOxduzTPv0IHoJeLPx8KZ3jwCVlyShcNh5hkBx/oxnyLkHPRPd3CdQ6BVHcUXPtLP1TmSwJycyOmnvmdV++/dZeodUmTdxTlH+BRsTr4VePUA0W65MWy5BSVaA8EpDbIc8gKbOgnBiiThlrtVNIwGI68qeJ9THwHueV+RziAauIpwqRvohOjgO1LHMXZEhJjO5iNb9huXwva2DOJImiKsd4Tch622s72u+hq3mubzSpa0lR7m62Smop5WvSn0Ehbog8fnR3NHJ/Hm5qj3pqoF808PESuzsR+yTkjzypAO5nj0YPyTTs47vlQoHP6bOw1zdItJKFXWZpaWDTuyrbI+Kf4V9Tq2YEwBjD9nvJEhfYiUbrg6/oGiTqnf70DZEnc6rhx+jkPjksqFGrLgtkZc7mIM+vOqYev3a4qimHAdv26QFGQnKx1TYoqO7estoRzR47JKnvSUlF4tBLdRbZOwm63653ucwjpZJ8Zp570L5a6zinP/OI0XMk4fbeA2BG2Z+oZeiZePeY3XHWOGTMzUd3KYyT1SmkuI14euSmZopv1WE1XaAWeeiskrmeFlaODnGQ9IgInC9yjTO8KJjCMOaCnRmDQR4TX5/eo5XWwuYsFHXy/EIoFXK3d7qrAiS4+o51Vku4t6wx+D8ZMVa5yvR8f8S7I9Ygv151VtBIvnXA+arDzuT+bQCq575762varAHxWm65ZnVVXZP+Z27dpTWxlE7ry3jERG59Zb/d5RaDXEGq62tPQzUXV0z3wrgdONYDDdUaWgt9bQg5NEaw5ne5dPrsuL6ukfouhYI3us8WigbHmAwrytIn8G1VC4m+pZiRR/ewKMkfxMrq1dAuuy33VQxRLW0Kpmu6dMRw7T/qYL6kggpIXd2ZJWY0VaXYcAzk87DXvq1z63KjmNRsxnxKBXN/Wu9RY5VtLgVYGWHqSL0nc1FRH3g/0CR0ETfLU9vKkpmQ+kIdFMOj9cfQ5e6SK8+aNO+SqfEZpwLC03Ijgm9ZonHK0a8RuFq/9Dx2lEI5sp1XQKVu4pEOppgJBjplshbFT0cZrkpa+cz39fDSOUOw8kDanb418qlS5SL4w6u95x6nZVTUpW7QTl5chmvm6xqgd1Xzguu83o7VNNTHf9lhb39rvEjhT1uEdEA/0hFvf7mTIg1UtRrN+vRutY3UtM7vCj4M6v/pcB6rgTHv6lFG6u+r/VPyw1PM1zyHjdiopqGNAmCKA+DYGrN9GkUBVROcSHaQF8WjoXMt8T6HorZmsIp+JsbcF0EMEzRz5d5htXx1X7IzNwV1OD3zrB052ETrCDNOgB+Lbl80jcePDY+nX1tgtUM6Kl9nJHVgYgH71QGBDAeHFykDagg8FBAuTTKocXvqfCyjADAfxAEeDQ6MQdvvjpzK+jjPSZ+xZgcW9RxOnGIWIGZD+EMJrl0CrRYOTgPIVoUnRlaia+AhuyjLiLzrcCb3pbKGENsVTnw4HauD+AXXvhlt4//fgkct2X3q8Xxy6uuXbFcfldD9tp4MsVxFbDz7lbUzlr3r2QM18epbj0Er2caJ8oJqmwU8s9RDRpf95Euss0Wkz8PtUaBPwWt6s5BF6tWob+FiUOwLOJf7lNGTVpUrljNA0MQgZgsX07tcc4FB7jQCRj8GCzGbxyFaeTf4gUBKscgcAQlhKac/Aua2y7AakEAAA=="
  },
  "scripts/compare_decoders.py": {
    "sha256": "93191a8ee5eb12b7132d5e8a5b735070103994129cca8a3532d598a996bd14f4",
    "gzip_base64": "H4sIAAAAAAAC/6VZ21LjSBJ95ytqax5GmpFlLnPZcIcfgIGdju0GAujZ3XAQikIq22pkSaGSAC/Bv+/JukiybJje3X5orFJdsjJPZp5Mcc4vc8liES9lwk5YmpdNHbACY0k6nzcqLXJWCqUCVi9lzhI5F01WM5En7PTqC57jIknzBSvmLE1kXqexyFgmavxUIed8L12VRYUF1aIUlZLu+asqcvd7Jeql+12ovXlVrHBmvczSe2aHr3pT1LKp06x9Wiv3s05X7f5NkyZ7e9eXl7dsqld7UTRPMxlFflhJVWSP0vNDiESCzg7u9rBPSIeGaa5kVXv7AVN15ekdxozXRZGpsarlKpKPIuO+b+QUTZIWTsrr49uzAMrMi5z0kP5b2stUxaPMRR5LN/P08vPnj7c3mLyU8UPUlDhLihXOXIrDn38J2FOV1jLSWtJbVE3u1hpjRV+L+06EOtoQhAyg6qiSZSbWZlZc5LV8rqO4WOHWKTZu98PJOEub3sxdiTh6KqqHeVY8tbO0oKsikVkEYUjXAUlFG64Ah1ZMehPJ/DGtinwF7boNdrwya5Ssm9LtOZxdruul04HRUQRtxlKp3swoKxYLCXvvAZ/QtiS7engBJDc1ruVP9hj+AY83uSjVsgDQ8mwNwD+l9ZKgzYoqXaQ5wKukTMaQqQas1QeWy0dZMdw7na/NxCxh8rmUVUpX0BinvXunOcRhqAe2wAxagbpxvZo2nWqnCLNCJMqjxYS7zlwhveW0UCQRmdLzzVoAAWuxw4zjp+J3s/07/QKKIsi7l/aRm5fpnEbDhaw9rmpRN4r77C9TxhcylxUAkXDohDY3c5x+onjZ5A9ASyKfzYoDO2/GkwYLETH4HY0f7Ruta+2IVEn2h8gaeVZVReXxM6gwximMbphJ+pUXtbwvige2/ysz0A2MfY72RwqBJqcINU8rVXPf3cFeasY1Ms3BXAAftSxHjwc/j+6Fkvom7UwysDIzZ1psGuB3dxuTEAXqqiELm5l6Yn/wvbudmnCKNWNnA4qmsrJyz6F5gGrKjJHn3+/v70fH0cvWIa8RyfbSk/L1e4M2qSgMT3vBwDO7BnRs0NlWPsu4IaNEKl3ksHMluQWOKpoqlg6t5hA9xO+G+ISm7ZGpYheUHqAs2MvuEaYKwQZRJ32UUV1o5NMME808M8lv9WieI/P2m1R5Yg8aWynipcjh8VafxqfC1UOSVp6N6dPbqkGEks8pAmHxMD0XmZK+9QsdIMjj+rHPuiZ5nZvBA3tuAAwGDKAVce3SJMyIG4qVHJl8Z5Ih7Nr5LNbHRbmOJK3TErXqtKppZcGpVpdP4pH/T7qCklx0M3BTVj8mY4YkCoHRIgVHGm3iEMprJji0QQwCYVp0EtFLI5SFng67U/bi4sYECstEnmt9tWFm4pwJY23wADDwghIrQSRoL2P+DS472amBje20/Jhofmzt5/KqTlIpTbSJd0YxQu8FbiNr1R1p7eLyPqLOPF2osZmmo4kNxFuHUfgiXRChyuUTYGJDKZzvQ48dKfEIe1uOxBqFB0JSS6dkqH/DYkSsANvCmRPZqGBxJtIV8S2BWPgc8lctR8cXejAeJo/AGs8ZEsEgt0M2dxoBorhsvGHm/IS8ZLImZcEyJXuzP47P4AwkCiywZC41Ez00W+nJsSgp7nSXBvnAFdvcWajQsoKwKRPM8V4e5HrySNjWusFTYJ5w2A4iASqH66+U55NrYTbN8/jnq08nx6d/P7v4jQf0cHp5cf7xb799vMbj7+fR719OotPj09/PzOM/z27to//qD+SaufmX5+efPl6cISkgzxzwrWm318cXN+eX15/Prm92TrasJW9W5ZoJxfKyP1wXVbzsDwD8eaIzCOaquUkfRIYMPQcoQp35Wjp03NSFzHUguizuM5C2bo1hbwbKbj7YDZhNN+dRSLgV0CZ7dNXCApAklrURBfqc5W3k7aAtFhRvb2Hej+3fN/fRqYmEmQ18Weftb3ZnSlf2qJkLbIZNOPTy92LvZ5Gnc7AOl5loO1csWU94ImsXdesNHYkR+bp38xwpxQ914iJAE/5pyEAanhmBlotaR2TEL3tIZJNPlGGZzN1bmr7jje8P73IO057pI13iRQjq1QoiI8WvTT5FIELeAvgeIZUpBgFFRAEKex03thfcLHFaU6AiSopqfHx6NrohsnYQ/kwSU1w260ywiByF3QKGqzLzcm1cy6nizRXthFaBg2zcO1Jn4BYQ9kWLrI7b9A/dXLNx2jeSHRsnHZPQPEON7U02Sc/KIW7Tgf4c6dsuRBCrcGSVEMjcxjNOd1J9kbWv0Ty6nUKFZOl+CIepakVkHRAle5dFCtuMEU3GvAc2+mcizrs40NRkxzmBHTZiU6iikh0aeGez8UAebjkgnuG2ctWIWqcxJb4KRFWde7XJjg4/9JogdZqIBzx/FaA6DyIHPBcpADvPGrXsUTucAHm2o3BI0RW1q6xhd6ROj0iQuwGKw6wAN9AoUhFlWbNlCDKdyMc0llPyZR4k9bqUU50mQislYgVyoyXqhlxMkVY0IPqIDuDEKOijMo0fMtknxOQAtC7ME1ALKt00wPUQcFtKVJRtpdcfPtTDuEVo4Ba6WKOBGxFgc+Smb8L87CS4DU7vrNszfYAjPiY1wUoboUGZewpFvY100RSNElUl1p4REYrOVVko6e0Hh8GBb3WHFU5xRmMEXa0zYhUhgtccdBEcQ1ZWp08odqE3PYhp1K4KY5lmnotARhmjg7vxLz/5NhSRHpLILCVB912Isu8886rnGznMTiAYLm4nDDf9ERZpXxpAz3XchqImVmr2Mlj1On7p3+eVB0MEmytTW0SjLM0tGdDNH2/gzJZEEgTsvSw2Ce2a5NiL6uoDAccPUTKh4Mf0GrUdmBuRTd/YBH/NGqt42BGxaQX19RmI03tgR6mLRj2JeYU0qaYHh39FEYMok4ly6qxs15s2gtptbDZyYHBe4QSwFj7QcD8kLxi8GZlXbex3r61MfxL2r76MN7kCMoBtpSjNS6wfWXf4AK1TVEJ1UBesFC4jlBIRamqClweYr8Qz/RH3ynPy+Dbif8c+izpeMpeVv1cU9kG9i2rN4DP1yJIWveeiEVWii4+saJKcGH6OzajDqYUMN601Jh959g5ClMy03tIAYi5bmXmDzhh/nIe6mNEB0i1CgHTbIxaFt0GPq2k4RdSyogRBU0Q95f84/oOq9+ZeOz0//3R5fOuihyZCUF/ktExenebe0f4PpoWrXRxywTpvm5T9oBu+uMZuWXwfr2hGm2OJxgW6ta3JnDckbwjvfTYReH3+xoNWGT0f7DebPdq5V797mjb+yPgG9QtabE23FRGwBbLTdD/8lZTw3I5Pj/b71b8r0wxPnnKLWt6aZWr/BrpBEm3Qp+kOsjUspi1gpi/cZD8+MemP6xiOp40MiPF+BOATCgHchoB28Ohw1ymmdt8MD3yy+RxwgnF0T/aXkfYGPqGhtzc0RmjbCvbKHZzNBfjE3COKICvRjCh63bVnZygyT3Rf1Etjo20L7jCq/190CPoNAsuRThAYsB04z4QH3dL3Co9hSrE7/SbzxWCXNwqUjR1Ma2KFm3tdO+IKNIzdrx3p/8BGI9PqlNQgUPoTVtd/MV+vBFPUbQPuKEWOdDOD8qVtW7T9CP2livqz7qtVeFwtdEv2Sr8BJVNxlZZa31GUFHEU+b2VoUiSSNglHh+NqO0V6EBErdZ3p0I13zrV3pcHItaSoGwlfNZQGn93oVH4SHtUe1SwlFk5bW988+Xq6vrs5sZshGFKmnY//Yd2VF17mJ7Crn1EfWIqdqlXPBkwGN1eGszv+s0dCTHcwn1dMeZQof4SNu/97g5iyJFy8+OX1+8Quy3w1J3XbobR7c1cYUGYorLE4w6tnetEHEGWPjiG9N9PIDFL+TxDELobdEz7X6Zse63vHh91V/nEtgofJs4bt1ugb/jXuenrlnJ1b3y2dbWtFbg0XVRf3OJoMuCS/OrT8QW7vPj0r7BzrgkCet1Yz1IppP2xrVZ1BxMqcRR+jXKV4r8IWC3yUrAmTxpQmXA35+yZu67WQ2l0jZuAHCpvx6dIb/N7IVAU0BeqvJ4e+jsPs18tzTcY9w1T16St0tvvZcGuDwPbgdA2owOrqTcOBGmfwQWRXEdfi/uRyzgBP9rHf6ORTZyjNusTDdpqZXD/rtNb+wHUs4cEMn+c7myTBlp3A9G+Y7fLje/9tosrn1PUWPqDGxXemRSac9KgVj5byRXxRZMXma08GNWdmBfuEnBmW/8DW9HoWx/p/WAQsTqF+Hf/xU3lcyxLuJdQ8kz/pKsK3JJIeIc2Ez3+n8Zmt8uQJs0FlS9ACB05nfMXCr6efvLBAYiugQBM2IseeuW9euybkrc51d8sNZA+4exudzadgg9GlEyjyHY0TWbd+w9X7eNBDCIAAA=="
  },
  "scripts/compare_guidance.py": {
    "sha256": "e61a94bb5adcff4625d9e6838495851d1b64a008a1c2551144a3a2f55d237f8d",
    "gzip_base64": "H4sIAAAAAAAC/51YW2/bOBZ+z69Q80IZlWWn3W4AB3poB+1ggJm2mB0ssDACgZYom2NdvCKVxJvNf9/v8CLJlyTdyUMskYfk4Xdu39Hl5eVPTbXjrQgynm1EHvz05efgOriXehM0tTCvV8GnYC1q0XItmzoKdq1Qor2T9TqQ9a7TMyVEPlNa7FR8eXl5Iatd0+qAt2tsrIR//1M19UXRNlWw43pTylXgJr7j1QupTadl2b/tlX/sOplfXPz+7dsfCcmHaVrIUqTpJIY2TXknwklM96i1Wl7dXmBlTMfEsoauOpxHSrchLZ8x3TSlIn2rVNzxkk0mVi3e5bLxSmW8bmqZ8VL+Rzit2+ZO1LzORNAry999+Ht030ot0uF6bVcPuxCo6Z/Nyk5lTa3Fg04zA7rEkl6yFRy7GECtrHigiXRdSp1t0gIYr3i29fIlpNV4I7uo4ll637Tbomzu+603ItumVZOLMoVuWlYiwi+trXid90rTRCrqO9k2dQUg/fozU3aNErrb+S2PpXd7vfFaAXvBqxQIZkKpkWRaNuu1gGEvclGQY5EJQ0xETaeBxGRxEeAPXvWPmu/UptHwRegcSK0CunspNJx2xZUoZS1ugrwJagi1wjmscA7dtBjrFLSEYyrr1sZZafvhPOtbeB+5VWTGnD7DsFnZtHIta14mZP24bHiuQlo+Y4NhYppjtJLnKVk/nNjFMLBoJcF5uhxw8K7UKY7Dz2wQfXY3gEsRkXiVlsyNsFszL4te23gtdMiU5rpTbPImwWkZvCNnBNPoqGMxj6kVLEUdDsJLBidX7BZyV9ZoBlkugfk/edmJz23btCH7jBUZ2Wywnj29DUZB0eefT71tmb0mTklOT13O7SWLpsROyRkI2ay4nM/n6cf0EUuWDHlBtx3twW6fUspgdpyeMGIdw5+dDHEc2iMiPEbHUIkHkXWUI1Ml1zVgawVz1gH6frNAquAr7vYSSr9JpSi/DiBZNz4CA3DxSiV+lLTXGusIEWYnnfWFgz15ZJqrbar3O8EWrBTrhkUAoxBInZlITRZniw/ziK2RcbkZQxaE8HWMwbyzVYAt3s/x3l/B/jlUMytRXP7sg1BvRPDo/PEI+0C3lNboDjlMbkRtJnbJcnF5ckzGd/6IXxCO5JLPbs8z41e1sRM5CDIjQUvnqKZrkc43vK2aeh8FSFw0RQkm39e8khkq2snpeiPrLeTY4gsvlYgYqsIKtahu2orqhQPITRqHWox86xnQSD0Uo8UfbYdV5b7F4bDQ8pfR7C1sRTlS1mRlGIu3mi3IEONhUePA6dXJOUh/yPmoGR49p6EfrwRc42S05PW642vhJp68O4c93qaysFskCI70Dgea3l19mJJBTZ7o5ej6iuSWIzRuey3HomMTvklOApZkzWDvjpB6Px9vxet9aCPAxOZW7JGZ7ijE4ANtgPfIvsm6D44YdbxSSKiTl2Lzk49jH2xBLgvET2BKHblVIR9cgnPpDPfSTdaULmyt29ks5Woe+JMZjO/5HfMQW34R2hmoP6R2O5RaAYvHsbDF+FDuxWtZMge2AIPDMpPjqrFkXOHKqt/tTeLOtLQK4VrItZpZKWN/V6xezHTkPKAttSzgO/70ANkPgXmU7mwNjqttLtvQkb3ExIt4kCBDzTYxPuoLokU2GVMrV8cH3FlkMYreI4g+P/CMGIYRvUE2KvegDf/uyKvzwCdEp6QKdBNcsShrdvtU0Eqjy+TIer2B2cjCf8k+ylMg62/Ke5OhzDGpQcHjCtTMY+erONiuYQORebZaRB4NL5xmxfrazllmJIipoWg4FrBgO+SDmmDr6cXCPUTDLn0ZWpwrTYeJ6QiDxTlcxls7MYfuCxc4OqaX8GZ0FQ1EP8eAHk9c0cTZovhKNmcgn1RXvyE91OL+oHG66SPJJYBR3zRzORmAa6RxpCLXea32O4onZcsiyIAsJBatuDbdWu+SqxY/mzj43ZFxKmHQ3fAqZdYRm+B1cMfb/Y3hyDBbg+guAk7Rti6xSbeOmU3wQ0/TQ3zMaCPrHN5RwHZq5y+Ozle4SeiC33SCbeJ7wvhjuza5/LsZD3OhslYaCJI0zZsMjd1oXczzPOVuScimU6ROFhGHMYT9RVGo/6Oilr8hRLhhMAm8vmlFirLjMxCkVeJWmx9ar1wzAKXogirGw9AnUCrwo4FAdjpt3cJxerNb4PnMFhi1W9iMix0Vm4UDUxu2TNlb6phj+vc3tMYb8bBcXL27PQjrM/2Wy5ySAPliEkmwE9WKYqReL5gTi4qyU5tRunMrLEnNoYLtfG+cF1ei4lsux58LPswDYhZbvul9f493clYes5MDXK+ajPpWU3acE46aneg0w/ceXKyvfDsQOWNbuN3LwVlvkyV8ouIPU4T7VKFJqUFfIvZ+jn/TqQd9aulxxK6GRovCa7zvKK/3PW/ojolMez666gjP779+/Bp8+/rrvxagpjnfBjznLqMoierYdqdIDdFoXnW7Xxxta5rNvKt2KjzzaSA8bODBhiJZ59SkvpucP+z0SqK+S858NUAvfe6yz/TBZ6z2I33w600i/YFBgPI17T455xx/pVc0nuNrybhh7I8yPSN1foOi8JV+je8Licz1g+dbuuVxcwYuhpq1OKhFpxTCBuM9V9ZBM93xEgxnaOsPFDtXiN8kPt6eZS7PHD3c0rBMR2EsZz6ieabzFmDpoj/tlFkcN7gmOxcaNeYV7AZjWaqZPKIX8D0BdQP26BhvSOz/NXu6F9PF29mhp7AC/v3pwLT2hOnjsbnAI9osNV0ue3rB3pZSj2E/C4QXe80IH/NcUqYFAxloydDGOHVvxp2L5yMGIMQBNUrUbx14yutc7NwFftBz+nbLtwbo+fynEavkSBnSsuQrUUb08ZesGYYU2dfsJaobGRlQ+T5UZ71BXqLP4z7RhX//zTgkBfyhodHpLUtLtCmidqf65hWda7QGysk8vo5Qb9LRxDinE4BxtyOtQsvHx5/kooHIeoueD4WxoLHBgRhdtKfdVl36pkD6patGb0jJkzr1yfKDLfgloTq6JsiAGbsaj+2ojNk2BYkg4JqrMyVfPGRih24M9vpsHslZkbsE+cXiFVwKDmMBFCOcFOyRGGBo3iZxmtYclSl9WgSPZuhp5ELGA+3HROLq5ah8/j+k+OICgeEPShKWpkSH05QtLC2++B9WY/mhexkAAA=="
  },
  "scripts/context_comparison.py": {
    "sha256": "365715e1fe735b7e92d817d0229d146aaf439912693120052dd78c3a71e31e8b",
    "gzip_base64": "H4sIAAAAAAAC/41XS3PbNhC+61egOFEZirLjSQ92dHA77i1xpsnkotFwIHIpISYJFQBluxn/9+7iwYf8SH3wCMDu4ttvH1hyzr9oOAgNrFCt1aquoWSVBrNfHLQqwJglHcCDRYEGBaVRrUEJ1TB4kMbKdscK0apWFqJmeLhjoiulyjjnM9kclLbsB27PnMpB2H0ttywcfMFlFDL7zsq6Xz2a2ezv29tvbOWkkjyvZA15Ps8QnKqPkMwzwt1asz7fzFA+I+OZbA1om5ylzFidOAtLxq1StVkaC00OR1Hz+dzjcVAjmr+vv92kTIMo80PRpOxeSwv0M2DX6gitaAtgPWTx/sPvUdB5OZuVULECjeCObA+dTUqpobBKPyIk1ekCUlZ2Wlip2pTVYgt1itweHnN4EIVd/SVqA/PLGcO/XjVr7vB3Mve7QLQ7A8hOL0N++guye3HkTlRWI9PeJv15rjM6IlaTHtdg2V8FiGWkJppDDQYvjSQFzXkv0nOWjGylUXN9+f6dVl1bJpGBd0T6fBMckztUQvOe2OQZHEzFUtKSsu7E9fFZRrHwDAyxScYSKfvJTbGHRnwHbXCPX7LzlHEN/3R4K65+8ogRF/HnU9p7+uIf32LgDSqsN2isVPftZKPYK12G1S8M3QtNOJ0wv4mV5ulG+juDdXov7V51lonS+4UFuBOy5SGtNk/zUwomjB185ZeeLWSkhxSoyY8Tbl7gA3d9aHNcAx64GhrZcXhzH0/ScxFOp9EajkPcx4fzkbV+37j4XOP/ccY/je0ORr2gv3kUP17JByjzgNDxhm5m2Dh4qyzQFfy6pkR/HBqc/JfaY1fXC9fpXIFfsVYxA3Qz/tINSTlyls5oCIIG2+l2SNnQKUIM8qG5JrpDWjGsaBobw75r77CRlPCweh+6wkggdkfcGjXG1G96EX+9LLFTSvuIChTtrFaiNAnpUSbEU58JZAnLm5p+MvfqztsXVWMSLUnkVX3sQ0gqO2MfV2OX2EdWQ5uQ6pq7fcM386HhaCENsO+i7uBGa6UT/hVq5A+D4KSZNExsDYL3L5Ld44bDiuh4f3V0b82FMWBNzI0N+63vNfGlwDBWcmeWXnKxFQaCV2/B+qRKqFkjWllRBytlVWHpRFDAlJY7SfWJuK7YHcBhug0PB9CyIUcM8UnFFFqe83PFphytRyRuQoRcZ1ixk7hwXDjhNadWz700pXD+msry9BUJNYKZi+0L5VuKbFhVSrPwU7YBJV2EIIl6f7Tm9Dbjzmo1qdg+QiEI4TWhsATMfaTwmiA0wu4kJ+gGjbfC9afA/lZGzvwQUOxFu4PyioqdcrURd3g0HouGGg3AcSIgV5GRIcPCXuBZVlFozRvKEZ9zXOBkZeGwOJ5/cCnGyT8qhV7aAJRUDCR+/pYvN5g6riL6ZPoDDbqY0vvAVAuMjPHYh0xXWzdMGZdWGnWpmLNdrbZJxX+OUuvy7KJ8yq/zd0uvFiqhDxshHhv8NdrPQEgRUuFD4LVdFlFBmEl5B8i9Y5MGNL53fbZ53nOiWk5jH+o+0/DjI2Z+lMx2YBPupkQXQ3rdcOFTddRfo/unesJ2hjsO+A5aoCouXWQpnyZwMmkwYWt8Jo6QW+X69yjFJ7LOYNxZe3j/J8lvI22+hQSmsUHKljK5BksNKOZ95Nq9GWHcDPP16pvucDh0436u7sKEOntpkJwA7x8p7C39gc+ugdbTCeUFlXHupT0Tb2iGIooaYekV3LcNZsNkRB9U3bEHyYd5PTZQLRrX2JYnc86zvwrLEiduPyAwNaRKCMa4yJ74ZP4nssM4HIYsRNvINrmgbxrXX62ySMsJmn5Cdh9rrzoYBPKLM4P3jnrp6KvkrdGU37YQb0lp8vmh8JPrilVS48PXHRZWLS7OwjxkyHWYDq99H3KfUCscz4ZAdzSD0Xcb1YObl3uW8GC0OoX4PL+EtnkAEVWpdmjbEfdqFCcp4NG4nXnakxe2w2p0kL80Ip/ax4E5hOXZ4OuTc/odN6dH8+Q1fGYzvjqXMdnDFEtD7Ge4Z+FbHjBQdP2QjvQyLHHSoQCZjH368hUfvkfXsLbAttIu/NOGs2/Gn14vueFtjFXnAzwZfv3W7D9T7w/XdRAAAA=="
  },
  "scripts/cuda_precision.py": {
    "sha256": "57f060f17bb4248a87a14f47724b2c63a336c08ea1988009a5cc6195aa7ae7aa",
    "gzip_base64": "H4sIAAAAAAAC/4VUwW7bMAy96ys47VC7SJw2xS4BcsjaFBs6tEGbbRiCwFBkuhZiS54kp8vfj7KdpC0wzIc4oiny8fGRnPOFNRKdG5ZGihJuF1djqE2p5B5yY8EXCLXSGjOYXc+hEDor0YLRUFsczqoaLcL195tZwjln7CMsTxcs7pRT5Fqh9vR2IcTTcr5Ib5a/FnNQGoQGtJbyFEp72DQeNO4ovkWROVA+oYhftfOiLEEcs7tmI0vhHGyQMCLYRif1HlRVG+spi8TPQm5RZwncG2hq5yleBbkq0VFASicLlNvahKSFcAU6sgj9jANClLVFG4JhVYYghdbGAzopaqRPysGLsVu0Cbv99jBbXo3Tnw+Pd/NHmII9OztjHYwAqt4PwO0dy62poBa+KNWmRwkLOjInrao9XQyniFwTYZ93q8t1nFh0ptxhFLNgDpdXk4s1ua6omqi7SGZL1MaU5b3NrcZrGAHfEQvGjoj44ZPHeniZfOLx+oDRGyuLw0GQDMglObAsXDCl/ZGxjvLb0gh/Nf7SWaNXHgnxHnL0n+IJA3oyzCGlO1laCaXTymRYpoGQ9NSDyGGZD+Ccinf0Ot++hH99gPCoHDpb8ow+4hkJSyKP4cMUuGwywU+u4bFCOYTHhlRX4TzoK+KtsINQ++6RxH43iliGLtq0jRMf4wRISeb31PNpx1OSd6UfXWobsPPVwqJshb6Gh4Nq+hFoE4bq0U7gRi1HHv/40Y/ZfNoH4wPIy8YV06Vt8JTdom+sJqHTfEVx8l8C31PHTtwTY+lOYNrW0jPdl3xvNL5iuU/6tlj27w4TMW/VwA4Kpi8nMU/WrJ2FhH7TuhX6Ua0k3WDVosIpT7sKU+pCmCPGAv4eRypNVVGWqH/3sGnp3CHW7cQ6CkI7xaOl1US/A6A5sHvoxjzMNeFpwi5y7eqBnLpfdBuvn+iww14xseqTrS7WA+BDSc16O/LE+MGFylyzv7aMZFNPBQAA"
  },
  "scripts/cuda_reference.py": {
    "sha256": "8d3006a79e47f83efbad738143d934bf42fd607868f885af7c6c3dd3382e621e",
    "gzip_base64": "H4sIAAAAAAAC/51aW3fbuBF+969A3AeQDU0pSdNslfLBuW3Ts0l8ssn2bF0dHJqEJMQUyRKkHa3r/94ZDABeJDvZzUMskoPBYK7fADg+Pv4lLVSetnImv9ZV07KqlEx+TbOWvWCNXMlGlplkaZmzpitZu8GvSreqXLPTl69Zmqd1KxsYxl5+fnUKf4tdfHx8fKS2hl3arOu00dI9b1K9KdSFe1SV+/VFV+XRqqm2rE5bJGH2wxk8OiK96VpV+Kfuom6qTGrt3+z8z1Zu65Uq/MRdp3L3+zdFn44+fvjwKcEJAiHwjRBh3EhdFVcyCGMQXJatPn+0PALGMcoVq1LLpg3mkW6bAIfPeFtVhZ5pmFDIq7TgYUjrSLtcVW4VWVpWpcpA2b9Ju8ymupJlitq1NC8/vHv39tPPkd6kj5/+NbpuVCtFrxfUv+eWbWQuvlQXUV2k9ntWbVFisYaVDtnWjcT3RERWFutCtdlGrKTML9Ls0pEW4Ai6FcRJ+Zm3aSauq+ZyVVTXjhSkQboteIYXr1VbKWR5pZqq3ILqBrTTTzRGy7arhf08pa537cZJANqW6VZYcw9lKKr1WloRsi5PgUZmSqvKKwukTtsnj720R0e5XLGyauVFVV0KXXVNJkWu1rD0AEIhDRdHDP6BF79dl1Uj2T87kAWcXH6VGTggsNYtaMpEBUZEXmUdLkrmLG92JyZQQKxCRhBCbapKjBaah4HhskttQgQnQREStHEMUuaa5jdfMlkUOjm/4fhDtLta8gX+PB+8WEaG1P3jNAdfBJzHXypVBjTAvl6GTK0Y6KYE8cFBJl+jAuI6ZLLQkk3GQVCAn8EI/urjr+Lj5/csYZ+aTvJo8OJNCiN5eDsUaVU1hhdTpVkqCa/5cmmoQDtdU7qkEJPjB0YdebettZFQRxpd9lLudIKTRrLUHfh5qjOlEjNrGEOWqnII2jDeyK/WlqE19ZVs1GonLroyL2SQK3CQtmp2vZl/MQRg3raBxAdWxFyg2YVcofWNugoIDTR85b0ATIq2ryF8UvBAZ19dprXeVK23sJ+PEk0/fZ9pDB34plqB2ENv6KlnnMSP8SvHsWkuWpAYlmyGg2Udh3gt24Br8LRtKmDxGAw8fJA8WnjTNKkCK0Pu7+TrpqmagH8GpdYYL7AUk8lpPkZsOE2C5gSBqiZHg7r5zrnRF1/2/DFXJgPpadA5x/fkT8OlW/khJBklWS0aiQq/kqKtBipD9Q+pcN4gXIyCYH9pL2glSMy2SoP/r2fZJi3RasCw6lqtcrBzq3tj8V4ycIkuLZI7EoYRxVjjAnKEBnPgUviUmlybo9qsAk2cWY9HJv2EkKMlumFClGTOOxhGTrP2eTnSKIn+IHEcF3cqJ6sgfUHGtnqxy7fx6Qxt48nWEBtPkO2itMk2YKxB4iyzosuR7XarWvSpi0bla3yR93mTcAYVM4Abqqy79jm8pbEQc2wL5AW7lmq9afukaaejgHJzT3zqT2ytWkfpeGpWVFCDffJH28OcOmLKZHoDcuAJs3DEMngBOlGQYYzIeXVdYlwCmZFLx2YmaxAAFUmPSGKT5wXxD845CAPJ8uQl98AhjLiVDj+cQGht0zYBLvD4j9enr5wpG3llytnvZQ7jTgz08vwizBcmhYYxEKraqupatRuPl+JPEstm2uxeuVgIQpZqQ2De9+F2kWprBP+xdz8PHhMkm3H/zAck6EiJBSjGkzzVyI+JEJwc8ApfPkjm3wr5N6rRLeWxHsRSFGjjelWj1qoEV0CWbG6WUqhS8nAiXdzVCI+DrrYYhFw6sWDtnKemdpJ7u4/OZuPy3P9LNSAfbWM4sUmA0CQE4kqt9YxITlAum/N7yXpkGPjV4UiH2mhARAsYDEM7W+wb/1vVbzB7qip+gYnr7Yegd+TQWJyex6ouQV2QmW6gvIADtXpmMJeXIq534G4uvPRs/oOYEKh6V17ww4oZDvwb6LJIL/ZH3q2haCyUB4Io1AiVuGJDi4FCAqEFvKwC4jLdSsRCkMsXe3Lu+9pZB4kKfapPduR4XVnKZuZzjYUTlD2hBg1czco0wUFWHCwuwX1+MYQ8D77Xm75nZf8CzI84h7KwqwJQJFdgE80M5h4v3KGfydoMPkhuVhZRzm5Qxbd8MVwhvgoNwsBfWCj3zOHtBm8A8eUaPTrgsyHoRAZYTqnSOt9p1kV1EfA/T9cNDCdYwoh6jnnUVvYeh6D2wnCZ7JV8z9JjuJsp/Fo8ivgoR/CFTxJ8klv4YpJcDgQLH8OAhc+Qo9fLiOddY5ArXzyZR5RAF+N0GnHXMQoNxRHah2fx/OCcBPMW0JMYHLdAO0QOeSwm3ms6maFr3nrjRviNLIx4MzB8Y8hpW8RPy7Exvwtv+joxhJkDF7HJ45DX2yF34KvlXbiP/GQ84QB82dJu9xDi7SUgy8BuKNguBrdRBHR/piDfn6Qtt4h/5RGmeYA6BhJ4urdn4tXrNz+dfnr9yqRuggbj1e5pf6T2BQ2JTW1B9/ek41Ceko36kmjQujlrRapEFJU8Dh/y/5RjYGnXZXEloRrM3QH1/kM8qbF2m2wDFQ73KSwIc9sFzzHaO0hgL88+z96d/cxWgB/N3obNuwQiPYjzYBIRaQIPg32oCFBPtjnCpIk7SfgQo1iYJtKrVEFdwlwRMW4SfVf6l8ZNx2KZVbEfzz4joAX0icE4yxtYdsOP6mTAHXC+yCEpgJMB0KthbgXZZR4e1Y0q22FbfMPNML6g0cKlGSGgPJp6S7M7AvvZTAPhXnd8UccUvVcN5B2Tx/BdW0GrK7ZyC7APWKV1eqEK1e74wmTgw8L2ZCBseAtAxekW/LQr2iFyBcECyq5kYYCtGfp0LpcR8GmxsScXoyjxkBW6/qvkwEbSoP+l2WLyLWQ57Xc/0nBb3YztRsaytlqBMaFXAmd9GFieus1l0zjUzExach9AXPchHHn3oJE/TG39HrexfKnqEV1Eeohov0HanQ4fEh9KSTjjS3VBWcNj2he2dM4wwc9evvkRdydr/ZxdSlkzDA4Hd2mDpgeOveHGQhDK78F532xF5gtRTXswJ9BoS+MexHp4W8NxgWo4hM3YB3wn1jFdvuVC3fS04gJuGtfc+7ZKJm3FGB3NXFmftNJkkWSwetv25DOLc67TK2qNnHGGxO6dyFbrZz0pqGcEFmEdvbomlQy1YIkdtxG5n8INuE8HzsWQ6cihxqv2PGGl6VYfnE22iIahnGMZRSq+dGubDD+fIhVwgWcowR6dKq3mhPF8JHw6P0jp8RGQPJnft+TXdhOlj7Mn8xMNAKDMIwZBxp5F7Cm8gQmn7SRkPyy2Sa6yNuh1YF8DBKNMmpjkbcfYQnsfcqCMcDRopQF3tmnbQS7neDBQgiGiKVS800GiPRdY3OMdk63nCbgdh9v4Y/g7ES9v5H87CC2ErjecVAULNLqKuAk/qF2ky3P7vHRY17/HR3Sy+SHAC9TGTxZP52PIfDsRxTeVoq4KlUFh5G/OnjzG8y/4dHK6hbJt83LQ10X2d/ZDPA+fM7duBnk/hYrAKkA0zbXCXZqjvVYYmPss7/QfQQfE9DUk8tlWtikCtFlRlesTs3tmw0/H7JVRE8vxnIC2rz6+/9Geo2jIVjvbxj0HTKWAoMCDvwxcBrdX8WgKxlBkd+vYts+DfQfyzhmfdPf7Gw/21CUZnBeZfB1Zw0QHMiKPHPu1hBaajOHroP07Yv8wOecnJ9v06wlUQxuUGpzjydxsrjl7n5C9I/7MvLaA4ySXiBYa2j7zK6OPPKRs1A43vtTKngJMdkdcY7VJm/waVgLtwwDUjg+2gnAMrHtvSfbZUBgN8Nje3sHA2cCc78Gad3X4B1CQwz+5yk2DTQIYYABrHPB+7lwEnBHxuE5XkvmY2Jeq7WpAyv348O/BD9H8QBvm3GRyWBfYv+GBvo101Ack+GwDeQaVzi0bPhqGO+6L38cI9Tga8UdCwMzijyrdku4GtJHxmUlfaLc+Drod9Qf87KfT9+zD+59+XWCOMIb1ddANhNhaFZ3eTLgPUCtJPjqMaBQKNkJx+wE66ynvRHP4DwI06UnPOTxTWh50/QVEY3IgB8xWx/P5XJyKGxiFVR4C1py88uWtwARP72lf4/Z4sJVr+pD+2DygOSI8QUcbT3abkdqFEaIG4jqult9AWsTk3JUrIKeC9a2dawsucfCMUro5vlTUONreZBBo1N1G6aoFlU3QTeSEuBNjmdAjxJbcXMqd6QfwL57XGs4xnrsG4f/MBPbBQDP6inkJXgKQJAL3fDtUqJ3hBNBJkwlzL4LfDkH5HXCQED3q7tBC3NdvafRz6Y6/WO9KzKhB4qm+LZnPnd3pyB5PgUFCRRsP4+RmS8PBgLDfZvbvvaFwlWLzbI758Q81YymdlrvxNvHjoXl0Tif4+GRxVZpdyhLcIUk4VBXszgUwHRjXH57QgANJDse6dGlwQrCfnIeyGHwlDK7Atol2ImLPwZ7Q+qXhi7TcBQc3/PuV3MlvqBPPNPzmme9HBxmZgWd4ewVMPalv5pNZz8xBgPBob7vM7SYHsPZaCw/tI99FRYEFo5RVZrbbp2N4vBdkUL7BmFvTu4XTBQyvB5mNZ4eAzDbcQy5w+0WWNDhyABUalmgNvprM42cRgB8x+HDXORb1B4nLqlgQXFAdiLLI6kU4l7d/7zrYgnXk5jqZO9vyLFENvnWg1cAUAsUXF4CCcQ29zHTRK86qemf25Q+UG017oLbeOoLRW+Inv2aybtkLsNZr8xPDP9VMoqcsvqEmm28jQ5ys+A36aWCewlgItI4Qtwt2Y17dDo8P0Sfp1gSmuGIAH38viKBtoi2oyl12MAe7TeJu18WnzdrUwTPzPsglnYShKwiRV5kQ4WBcnOa5SO2QAIAwnUnxKDWZLgFrQW4XLd7wuXccwBceoUbM/lC0kUWd8JeAGguJofcuzZh1oMEmk4ksO+O93P3xuJ/hfmH8AfN3DgDdfy+pRU/3aAiodWJHmz843p0N4VUMeIpp1YshKkzMB/jR7515crxIZq6J7N3LC/ZOGoiPu+8AGqZNMbzPwGfBxL8Ef4j3IWP87y+BOaU5Xzx6vHzIY7yA0DO3yPJzjVWOOkV7L6it2A5QD8FMWDSUkwX2aHddDdnj+b6yO2ceo87c3jzETokdtZkPnChm9paKv8Zhb4SYetWV/fnj9GJHPDrPR3w7ske/ied6JmtBSVVk4FRIgfsQCm+JVHQTdrB27g0djDnbE6Dp/a3+wHF8O+3g4OGGENkZfv9RG9v9otGe83hWt/FsXXbQbVvTfaDTJZd1MUHBwl02BDwhBCYrIfiCktbR/wEwg3q/aSwAAA=="
  },
  "scripts/diagnose.py": {
    "sha256": "d18af5f78a60e548a54d4f9ac7ba5d1ac1111d2a1a6677637ccf896b277bc14e",
    "gzip_base64": "H4sIAAAAAAAC/51X247bNhB991cQepICrfaStA8JXCBtekmRJgtsEgRYLARaGtvsSqRCUva6Qf69Z0jJlrPZNq0fbJkazu3MHA6TJLmSSxKNqWQj1tLWW2lJWOqM9YV4TRuyorNKe4cfU5FzgvRGWaNb0l4YKypLNR6VbFyRJMlMtbxXxJ9GLYreq2Zc/dMZPT53jfRLY9vZ0ppWdNKvIT3sE5f4Owq69VSF6xeDJ/uVnZvN8FWwjkJpR9anZ7lw3qasJy3LpWqoLLPCkjPNhtIMshZeu+vzG3EqEm9M406dp7akjWySLIteQSuvhdeja4OOsDibzWpailrJlTaOUmuMz57OBD61crdiPjhf8L+yd3I1yASRmGYIfQp/+ZMYlzzdp6aI9tMsF4m01Vp5qnxvaSrSSqxrhJQflHQ7vzZ6KhVXSsDplNFB4bBEd1T1Xi4a1spZPCxMNAb/l5aoXKkFBK3pdZ3yasGryOH52cWTR48e5+J86slKeUgvkKp0yMR2rap1Gl5k7MZy2Xa0GoWmyU3Hd9lU43IJ+Bf00Ib4MmhGnQKYctX1nNPrm4kSb5DNEqXiZdNQjafylqymBoLHlVssla5L11GVxl1JJpQT2njx2ughRZ/Dt+w6VJlTjaqMBqoR3muG9EbM5yJ5Ie1W6URIXe9fHsEaxaRtv3+STCoEUlPVLHZs7EjW9asVoWzqsqaNqoJWkbSdS4RafuEkNY5gsPfm2B6kS7mRquEyQG7H7LCqX9DpFKS/VPd0n2Bvd4c//GmpNXbH/bBv36JaU3Vbmt53vU+vk9Pe2VO3UPoUVVj5JgGEJ5q/19sC+536C7HkwtOdn7+1PeFRtYT98/Oz7MjaGEav1VIhEdF6KF2OIBYvSC2NLwowherSLDsu471Kuquo8yJ9c/Wztcbm4r1sehqeJwFd7R/Du+zpf3GKq2nM6nGrxDo+ca1KJior07ZcSHNxPZXgpJ187MnuTlD4cy1byocovUG157VVYIGRCaI8U4T088ptcm3WJGuyeICjHqU7OwTg+sYfY2h7nQ6e5KKSHZfxAOmI0X24LiZwIdqoF9yMzboyNXEXnB0nDx6KBjQnlB7lna+hrHBdozy/cukXCedPiH+ovlzE2BFBCvr3I+xBOS+wctZU2KA0TXKk5yLL7mkdoZwyzE2BXiDU1aeEjYJIou1kY2VbtoE1DyXH/BS9wXJ8+DwpOHTY14tnsAje8r2LnX0AXyzRsVQ/gx0mLC/8msaYEdrO9BZw2FZpHHFH/V71tfy2hh93PMygNwfPhxMzyE4RDwsFGy3UhGgABVc0c+tXM3y/JMDNdBdqQmocrRO9kfvKClXs0dn3KwMqEdtkx4r8wJglqrvDEKFQU8HC/60ALBf3q4BXQy8OFCBOR9q5mFZFcsgD1cmkOobprO9q6YeYx3aex3jK8X9ZoisfwHb+IAr5vXAHEv/6qTAPB/FaOukxcUWtC1ndIhUujydPBPb4VYEXXxgeWm1/WLTyLk2RrutD+m4C7gxeYIKvoIDWwkwmQRHzs+yBk7EFy8TiTmTFS93J5vy7k7vmZCEdhYNycOKHubh4Up6dnY1n5UQ8yB5ZAGTxvE2TD694TpBa9B14hmQrPvYSrLIDTepaMXZ5KHUJU9KBN2uB6UCTLcRP7148F0NeVNjT9s6LBU/m4dSELMLn9g6RTGfyInkAvUNMMZLXRrx+//LFy+fi18t3SBkPIVTDeKM6DPsERpQeNSS2xt66Z7CGeIJnHHcg461qmtiuvRYrgu9xx45webgihpw3DXOuM8FjyPJBwGngiGSNUwN2k+xfZopvRvEAywPQPGcD4mqYgQ6Rv0Na2EOFkRLw1OIyjMjicXF+PiR6dH6r/FpEtpijjp+Jh7Iexs3L3VsufaGJIr0t+CAbqHNE8vceEzmYOjYV8uf5SAqGfkREucCAIBw05P9sjDttqKgR8AWhaYinMqVX4sOrQvxxeYX7EcNn6WOvcKTuQwN1YECpApITVKasPMzlN3z/O16P4/fNfdRkXysTL1KlrFj3MJfK6s0VZnlL2zElIqp/xgd9yMGYkd9MS0GQL3roLYQ0dgOPsBhh4HbT7IqxKXmgGDzATQ1BlCXTcVmGObssW4ljq0yit+Gam/INtah7YJrub3V8iSyqbZ3ytYIPBI0ZBlz1N6tLNLo+DwAA"
  },
  "scripts/diagnose_download.py": {
    "sha256": "60f5ae98fd232037825872cccf1f402b23dec0a23d0f8eb6b002bf4ca5b444f7",
    "gzip_base64": "H4sIAAAAAAAC/51Xy3LjNhbd6yswmA2VyJSdqs5CiRadTLtqFpPuSnqnUrEgEpQwJgEWAKrbcbkqX5MPmy+Zcy9IWpLtzkMLm8Tjvu85l1LKn7Wqrpxt7kVl1N66EE0ZRO28iActQqdLU5tS/KCCFp+02R+iiF7ZUGv/nbBOlKrEudo0Ogjltah0o6OucinlzLSd8xHL+075oMf3/wZnZ7V3rehUPDRmJ4aND3gdD4VDH00zvfW7zrtShzCuRNPq2ezn9+8/ijVfzIqCrCiKee51cM1RZ/McerWNYXOznc1mla5ZYyjgXuF16XyVeefiQqSX+Wom8EsurQVtiaWQue8tqbvijeWh3++N3deqpOed5Dtedy6Y6Pw9LsrWIQzh6kqKrwfRG0knClPJLczrGlzO5FIuhMSpOYvYNW6Hy0n78lQibKDNIHk5iavMXocot3z1EGNXXNyXtCj/oiCvY++teODLBYuqWLBcPSlZ8NHxl44i0NGo5vRY/snEQ2FVq7OnNXpFUGRubOnajmpFzl8S2LjyDtLOvFnmtBqeOZVd+sIaWMKl7Ihi1BGCx+Qeta2cX7798d3VL1F3Vzf5m7PoQJ6xKhpn5fZC1nl4ODJCPgXiIgZ/2v1Lz/+G049DtQerunBwMeOyH8ob3dE3EZXy8Mjv1Otk14KbQxibmiQ3UbchGy7RL/r71ZmpISqSQ8dzes7mZ9tJ0YZkb0md1J9NiAG+ffQ91Enu12B+1cXuPmraICkQxYvnYbn8we1wV6gGDqtIWbiQsOOoia/Em5tvhKnFQQUVo89oH6qnE3Iu0Kpa/OTsH2lEVwMLocueKmoJGrD0OF3Wn0vdRXEL735y8db1tnrnvfOrPxmdWwWDHk/7MZ0dISwBoQ4FF5frtGXk4yQP6YLDFvXNmUlys7mgPLsRWfNPB1MeMtkEV8uTJA8aNwkP/ineEzUMKsW//wWQtxU81GUf1a7RXDkjX5iAHVVGwfgnKL6JI1zb0jX0Xt8SHuezZ/UEFztnA+HuE9gT8mabZCNBpeW/t12J/wHJZI+3C/RKB6t14frY9XHN5fXFXA4FrT8Ph5lOcHt9c51qeMhh9v4XTt3i1KiP6ey7z53xuno1dlOe4NKwRGFqjNXUZKPDqKEK0vLQNSbS5lnPIZG0Ru3lY6CEZ7JDzVI4aWNzs9rmJgADTDy9d2ZArjoUSZU9yA4EtIL2mI2X549PTaubl9SVg7pJ3CtaNlc32418Kg1JZT2qOa3m6cZQ0MPgoZmL10TpCzJR+6NqKCFJHyrI1AA6CKUBAkinqpBlI46XztZmH5YKgmO42mFeyemcpHFAVQXlOpvPBzsIOSHI0uLwRrkZHpGdUduGQSoM9Hj+Q7BepAmxxggA8C7vOgc/YBPcJW453rxhw5bcH3lQNfDfBufDMAIw8A6I+vqUkqYFjW3ulnOMH0qPUlzL/+hWe3MXlGh1VJWKiic1EXSjWiUexiiv9o8Y3KJBu0ZTqTsct/uD6vqwpKd+pw6Jj/732+9ovbrpw4EbJ6mj5slDo3WXjRLThqrx9oqNRGaeaUgSZWKcYlp+kXcFygqTKOH8GV2Mp2nxkp4B2KMxRcA5W4VU+rwEkYQqhToq03C1grCda7KX0HHOvA7LcejhTt+vJvBBbTUw8qiL6DhJ8zkXEg69QqePz6xMmST9/ABdHDYs8P9n58uDsntEAkDqDYeBSWTi8UklFWiSObDMP9ZJZHp9Hq8LZiH8TjM1cQz5TvdWXyagkITPz8zJ0lCzOJ3tXogE2Cn5I2+5SpHV5cDwqIMlM61I7gfUayi92eGLA4OAQPWYo4mIOlGc1fGT83eC6oTM3XvCbfk6J8i3zFapO5gwMeELZ5PwykEf86l3RzgUBZiu7D19W4AdD7qpvizcCgrSiQpT4SoNE5ADbEXPJUNTXBejMleDVTFjqYmBoTjEvrz7I3267eITbzdwiK7S1GlLdK8hN4D1KG9a7+3UBzQlqHLUhbihyEr+oPuCwildwM1YHih2Y8Om5SmCw7ePYHgMfSuOACh8YHKGc7m9GHsIIsASMLXgKbooGFyLolXGFoVcDbCJr0vCmfFLM387TBofeCdLtdKRjnVRVK7EV+LJzVxVVTEOJxm+yEaUQMXG+06va5AN8BdkpTCHTUMCriS0ZiH8j8SEYQ4ehrBr8T1bBggYxNLCt9dPNDoI0DRpZBNqibZH1lDgO1SzRvlcMw1/ey1GODsFe+bEqm+7kE18OpHomfY5sSvV3/ob8OH/AVs95IcDEAAA"
  },
  "scripts/drum_probe.py": {
    "sha256": "1ace27dcf94b2ca7846fa6a3aa31114c9a768848a4c2383fd526c513707357dc",
    "gzip_base64": "H4sIAAAAAAAC/7VZW2/bOBZ+969Q80JpVpadBJ1indVDZpouip2dFr3MAhsEBCPRNseypBUlJ2nR/77fIambL21nB+uHxKIOD8/9fIc+Ozt7WTVbr6yKe6m9ovKWTZZNdZGvPK3yVSanpdDaa+jBq9fSK1Wey9S7/vlm+r6Wpffzx5fXXtXktdrK6OzsbKK2ZVHVnqhWpai0bJ9/B8/JsipwlqjXmbr33Iu3eGyJdHMPSRKpdbvySZVLlclJ+ywzmdSVSjjJyY2cQjtRJ5O3727e33zwYu/zxMOHqZQtmEiKRtcqmabQVE935yxk2yKVmXknNbTA4vPpvdASr1K5U4nEu6RJBQsto7KSidKqyLG+zApRX16AVOW6Bk+Z11g23LGopUw1W9zO79zeVaNSkSeS60RkYPwimtPWpaykWcX5oH+OxWJJvFNeFzwpG7b4UDXSMUlEWdvjX+siEzVc0Orl0cneRsHmSVJsS5ErEiny4BdoVwkFh4VeLuqmEpm3qopiJ70HVa+xKdmEns5FJT2Rp16SFRqc12q6FnXoLYssKx56z4u88EpZLYtqSwp58Dlehl65roQJEOKRPuViqxIdeT9nUlRWurWqdejVAsrCcR5cmmlDrUu4wLuX9YOUuWUksZWCEuGYZ08QvPB2BUxnwnMrsyKFzr3pdQQnfZlMJqlcIraSjVhJv2jqsoEGukkomoKFsSIEk7FxFF/CS/cgjhBgzLw0BnHhFv1bla/w3/GZ0caQPbCQ7AujUiTEHe3rt/zlzatfrj/cvAwoGkWVrNVO2jPpA4PBFfAaViG5d2tlMEkXUWIgbMAW+SLTmcpxYru6krmsBHl+Jh9heutZ+7Jjf/AZbtNW2iMM5/M5v+YmbDkF7XwGzZrsO9hTMDaVnLn/LW/LCorslHyItuUlJZp6HK3cLUZ81dJrbdwaKIiU5mRYP1g4S0YPlarlPmHY7Zg4Vs7b/RFf86n1gVOZouBPOPh7nWwsFD2InbWM+bZnEfp8r9o25p1LpY/ErkNdNFUiQ5cAP4S6FlUdz8O0sW6PL+ehfJRJU8v4FZJKutw43BzfUmn2yyCCQYpsB4cYJUvS7shhwZ1hZGtxJOpaJGtDZv1ja3+F2pOb4tGW/LW4eP5jaFTlpkl0xK6rcJnvVFXkFPrtriOv+n2oDFIYy1M0DLbwrFitZNpTbkXCH4pqg7L7MKRDFKDCDQhNXa0kR4/QKM95J76ry4aLrOCb/zQKdMvy8mKwG52Edy2k3frqlzfXHy4v+L/evPvHzbuR2j33ZC1T/ntxH5aZsKZJinypVjEZdsbsg56Ne5wNtzYrLE2fVabuEkVEzUb77j1shs4jH2s/CJ7FtpP2oYkuoqX3m8gaeVNVReUzhxuklrWXqiXamb6yzxUSgBpGkaVeX7XMsQ2YCC+XD27nbIdt1NdsjNhAirabVFU+GRzlPTZdUD4qXfNi42LWhqwkI8WfGWK8btBDXRFFAJ5qzvZUtrDqjWucAQWgtQHMCfe0SIGj/WSak3cJLwBVnO/tzYua9r5BZMCetcobOMT2PnqGWa+okRnAkqybfEPtEskEp9VI9nwVeb9d3yAgn9BVMyogtaxynPoUeb8WHsUd9O2rN+01GAItA00RxLOf3v4TDVwo+J99sS2tS6px0RvWJGtEa9C6euo9DqYiFbWIXUK7BuWPcn7W9S3mKo3521eb9kvQR5I5MGrKlGqWdV3MKLxz4mLaX9we3m9DILeLt84pLW929zdUtXEZdRkcD7LZlizn+UPx25XexqwrlO5/cOyIv8S3bDrdiscp0nSqJdydUuBczvFnOm3R39Siv5C9MMuudU5T0KeyIutVnZPcS+Yq6sAETpDDluGcZA1TPtVr+Lyvvccsc+u+EFBl0wStb1TK/HF5cna77YDoXRD+0HI4X9yNJe1rre9oQlTq+EjVtkKGKHPJxmR6L7DMTincqUBGO6l5YGxqehg3pY/zQSsLyAtW1VNYBwTEyPrGyoklOGnoreD/orne07gPyvhInF71313tHLu9L8Ix5Zl/GPw0CLX4hIUIiVMM2rxFvFNb4i7e4x/ngBhmcuqrZOxGKPDmxj1xG0R2/Lr7CsY0n/E8ZBvBeJqKX4TEKx4V7bHsgyI4AL/sAE73C6Oig8LeBWElkZa5q2CTI9YddFX/0E8HED4YNd3JKM87ypWs/bbDoTG3/OA16gGZzP2e+JbBJxq5+Sw+HwfQYf++wa6E5sjC9KxtmUl6Mi0LXAZWxFN8eAaC5AoDIorXkZA8HC7YoAEQ5I57bONbNiHBnF8hzcgSltpT2qNXngHZBrIbu7hhnexiYo3eg419OUAARGCD21HcMtvHuMWf7O5Z3LeXvVffsqRBQlYSO725QcoDuFFLldiGvRQoQenArEg9sdVhi+fMFpwta6CHFRmYWQrkybGXdh+7G8UhORSAqLstOKja9n6CLebh4b3E8VuJIznKus67ON2TQ4Zxf8Prp5KAUSZXxQiUiWx4v7HH35Il7sbj7y7iDag0XgRSwexOqMjEr3khmlQVLdhaHK3pDKPHPapEThcYmfrkdDCQEuIChhEs6xYAVoEdat5Z064fY9ySkjn0AQPU3FUjVtK9+DKMb5E/+dbPJmg3iNSdGbM24Y4GrdarEarYVqNMLA4D8DqpG5HZ1G1DxGFyNkomB/VtSTH3VDgOiUxJMXx3LwBKuVafTGqd778mvSq0tmJrcpsFlJ5UKI1630qXj3mrUyuxTGfmxE74gdQOCx0trUfvIcZF9Wo4kfmOcGQTm+UO1gbDMlBUaqWAwbtCYAC7Jd9voTO730z0wZ+oJa+pfmBCQMSQfRoaZqyZBv2+l9+BHxgUHYmbFHDVdNbVDMSNgT9IyeEU4O4hekl3SImULxGKklrI95zRXWJ0nLuFb/OlINdVEqa6DlcCoU6XJv7ojuTIjVL0Igj9/piDC6bwPNgzcn+pHAGG+bdsudyWkkbEaV7oOlV0MzMlXpJ8YNbpjxqBckhK8E8sWbhkuyJDEYs/k9hf2NdhDLD1AiCIZeoe8mWwAjG/N2vnf73YjE6BLYK7o5jw+NjUg4GwzZ0jnSR0UwZvk8n9PyW4CRkbsfFevPfeCUIy/XGq1j3BqRMGjcYByGFXP3h7mlEpxYab+xXMykVCpuD3T7UcMzxJxQaw6w9My1d798z7rtqHifIxkWXt/YR+dWO+EiAQ2jMBt/iGjx1wCA1xvGSfqaX65imIOKe7ac6/LLzPZukL+5806vbQHcCedoMrl4EynSILAt3jiyJ7maXM/cXij0gxmaAitzrFMeN8ixTjnFkjlXH7s1J0Xa0MjnhLT5WfSp1UysgTc54WCYY9uyUSacqFo/a7mY6MGNNwiORYCgKkpybFyF1E3Z7fnWJpi/yAKSLkgMgMjv2xrjGlgzxfVUVTxnbntqGenj1xmDtrtNpJbl77A9KDIzr0Zc8xv1R1+l3Ov7a1+92P5jVjRnSOAo0TQEyyU4q3A/SRLeFaZgN/vf/4lsDo+5MmpHujo3LPjxnTjWRfEVbAkOZo2qf97kcCEVmhF11fM5zaaw8RUXz03g9FBMcNnu0Vohtlwx7QtgOGPWg8wptjbZC0k8yijKRt+W340Js2KIZjAiBA7Rv4kzbbUvvdfb8VNWw5W0ndNRwWxxdxZnoycvS/nJKQWGhpuusu0RolgH4p6XaBOvlf2JbzRyweAAA="
  },
  "scripts/electric_full_song.py": {
    "sha256": "5629f87436e2168a80fe5e34327d41a12ebc3ffc1ac23d914c54ba21d2c55c25",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAji4UKZVzfQd+oZ9kn6XXWABgpTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ4yqCkaIqpAgKEYg4TaSosjgNIhmJ0zQOZmIp40zm3sHB50IWolxKIb+oosR5mUoSGJdXSalWsEQSCZksFKxxcn0xFA+qXKZVKVRZiEgFiySFaaE4fX/xXXFwPB69hjOFQeaJK5hRpFUeShGmSSm/lEOBJ1mlkYxhSBz/KJIUhsCe4bJK7nHzNBewXBku4cM7cBznQK2yNC9FkC+yIC+k+Q7TbGN+/w5rmN+roFya32lxMM/TlcigLVYzoZuvrSFFNcvyNJRFUbds6p94f16h3GQAJt18A3/H8jJYySILwvpEVaUi8/shWNftX1U2V7E8OPh48ulMTMXrV+Px+OD649nN2Sf4fDwQ8McpwqVcBf5a5oVKE2cijofCURH8cKTG8NEMMOxAM0EQe2D3opTZ0fr4DXZKZ8iLRXKtQokjwioKcEqWy1DplZ05kEL5+hW2q6Qo82olkxI7zAaFlFEBDbfjO73iAi4XJKH0C0AcrvxXb0zT5zKX1A7nwClvsDmd4xaRX6Z+mFXQ+imvpF4JaKPU57gogBZLoLU5YBsuXm5ioENDz3gYAfuWQS6CMExXWZAoPKknTkSUb4YiCcoqD2KRxVV4D8uUSF4ObWP9cZBiRRjLII83QEdArRVvG5RlEN4D0xRLRFSSllLksH+APIF0HwAnzWJkHmAWsQzyVZqosPB6NjkRBZGFWFWFAhjR8WPkGpUADyZpvtKNR/pOuVwAx8G68zSO0wekfuTDTAVJ+l3RswXvD/cGusTReMJokwQrPBKg/ung4CCSc+DKIkyBkPw8TUsX6WIoAP+AqVxG00sA0mDCuHCcc4XXFLNcRQvg06UM75G5DZMjSGZpei8WcToL4gL5EyipAiJdS4AmUEOS4FkMU8MKcBaHD49bA4Ujv9ExBl4uAeVr6Q6ofxXk94B2GOI6ICLmalGMAECyLIiaPeRrJMcyTeNiBLBa+XIdxCMQTV62IUINc5WV0CfLKvO1yMI+3gChARNU5KII0LfGPzmMzwEvcexSlxgBMa3ggKrwkVndAWAlpzbEnz4oL6rmDTSFKhBGAoHarE4r6nvXQwee/AIUHIFYzt0uJPS6uFTveenMgQJwfuQ7nuV5mrtz5+PV1SexCQADkcpUrJZiVt0HSYNJxuxEPOKKT86gCwJspjbAZAQ8XkrEB8BTnwzBAAQaAZ0CICIVlh5KxHu5KVxNW4QrEM7AmgPrzDiTIAHzXF5iaJYaAepINTh20w1guPhVJjIPUBMdfUCiKrRQ2/pzyBM9JE3XQTWyCjIfBZ5/ONKrD4adYSAxfa2NfB7jHzqDDqRVm2os0HhBxK0N/hhMuQyiDQDulm5sX72Z3L4GbGJIz1nLJErz0cnp2dENyvNj783Ig8b1aKaSUbYBZkwcizjbK6Ec2LsUEUOWqgQ4pasyRsS5XhHMZSmTIs0Le6M7Jo1lCuqE6ALkpIxcvi1csrlczRuxTFw9YSB+FscW020TsHMSgfCRM8B4FnSJ1hMXhRJI4f67i8tPwM3JAki7COJgCf8vKwNjNdmWlvaf7wXKCu93gIALNOKCyhuaSw1qBBI76FZQfHgXc3GgQkk8riUswcxnvLgoZTUBcQvACdv2ItaBXte50eKL54GAkA5umxYeSZ7pVDigmHl7xyaFA0tg6Mk1zvYDXH9pgT2TcbUSASAhw//99gGh6Yl/BhAnKEYKGEOytTYHwQoQs3QG++4FuROpWnkUo+MxMB2Ynb5R7v68imMfedZT2SaZDUUcxJUAvQyWpliQCCiU57RwwxfVKCDdvbSBD4abhwRxOwHkAScCll2Dh44GQblgd2s14gC98+rzHK07WwqTEYFmnZdmQOE4m3qBsPKZMyBLG6yFdGLLc5eavIUsaUEQa4Cfoahbk3AJ+hPQ22otglX2oCJQHoOB+AdQjmg4DsUr+K9XJfwGoDPoPfuSAYRBMaFlIVMwNu/ffRXXpx+Of9iW/c059H0H+vrA1mjogAz8Kn1wHxYuy0owPcAOlyVCDwypKdh6UYXCOk2mx/CxCFQyHXt/HYLG/OLXXeCnNBYHnDBWoSrRLVkcaVkM3gfb0AUcPU3AsphtwBoC7fqQour9EcTKIgg3IgbTBxwP2H4DpAokUNbWBhnqQASIZMK3sddz+fdK5ZIaaeQaAUZmB12juQTfoH36mtfcINm4YEODxYyWsLsGzQcrDlAOIh+2u1w4Heg2tLQH9RB0ToBRwdxVQAlrNjLWqCX4SG3BTr4QnE/8JMZmhTF8mJOJn6atk+I3gNoeitfB5uMtqWATDfiAgs/EG/5sI/anqb3HT4hM0ji0NKr28fD4TtNW4015IC/cWxvyrjOfrzK5QN5zjtBrBC8V5fLRGv8v8Sj0yY3KYRZlwhv02wDOEfkrNA4PTkuXuqVGIDYGc7Q0nHUK8k5OH/HwT86uRYNcL4GMx9NDPBR5TEfhhD2qcOUXxz/EUo9lzhjcDVmNTdHlYagAkVZg+0+NVNEjQfDg+oa8glnh6pFHNfBJg+qBezA4d97KMEXvvqaFlSqA2sLlhMgfdX4kHk3vExA6UMgj7/fUFrPcqCUBmK9o2dX8D9o5q3r53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnRa7PcctQZdfGpVcY8ByDdhEsw7XCJxdx29GtG6wT+fWUmP7ig096e7jMYqFjuR4Vki0kE3CCz3Wh2UaowCJkT7yBjYmRgUWBIj5qNAwSnO1UEkQ+8ugQDeJycIwOoMxmafQgyTiYSChcC3JQuzlMyFuixgYNZMsYzpyBZz8Bx98lFzJnT6FM0c/vZyai9LEFGWHo51RSyYBq6PuYk7XrL4OFNg5Mbq9hDEXr3Krl3Xubp165bsB8DmRj23MtdFWL0d4axb/CfEHLa5NbDWhNeN+Rqx64zd7EXtDRtkWtkQ6ZwPhR0IyGmyEYzgiaGfYCQ4LQhxPslJJBS7AC3igOdr3Ag9mabO9R+QwIqxGkRqJdAd2KJ6rkW7ySyjzrDRURvThre4jhZYaOqcFYWrIkU4/vZ+eB2BY83AMD6HhztPQJjRk6oHRxwZGmK5WeAmwLf9ExbatxO72gXFSH+N7OEePtrtrnZUHkfYzyg/sMzrIN2k5gpPeo6Pt9Zbb+o986Ub98RI20QMJHZKKQznEo7Upc/ic5qsVH5EqBt9KJM4domcvaYL+5b1Z3Xa0MFLrS86hqXCLVms61UdhF9LwGpiivPf3Zu9jsA8sAwHcMwrqTzsTD814ms5o4bCGDkI1BM3tDTn3eDSMGsulMUww7Ay09sDhD3o4nlJGdiAMvz3wIIEDg1Xhuq/Ij2GHBiPFl1eXZ0hniAoM8eZATQijRtE9LFHozOI0vBeTaX0iD2MQmqKIgCdmNzIvuCcPHlya2YqyaVjunOD8+9gBwLoG5ofiH/V5EOYKKYJR8aij9r+1gvYa0/D12Ej7SRtvGLpEXod2+vtphyk6k0FJgXjQNU6UPiSthhAILNJf/fMfghzPSmOcE2SRI/Ku7ItQFkYlYKyhFOXY8uhebka0vFjJMoiCMvCcuyemm9p8cxvishc0cVu7bbB7qjYxIzPtsT8XshOU9c2bQxQE/BPMMVhU/zS0hvhsdtQDW1YIE/ZT39Jwl2aqnrQPDIMnze8ajEQ2tQlUr9QyivCcyIlbG2n+3MK1vuXW+A6bYmAhlj6GIWAMM6FDgsNn8odW+uzZwQK/LSVNNsrfQ+kyiXzNS80+uqHvLjZjNPwkI3uPxjw2d0eXDjpAu+0hU5WgUaAJzaCk5f2YRu3/yC8yBNvG5zAEBYwaJ+iQ8iog3cgextyrSaZNOZXWRD0+SkzHosECg3SIRmdVwUCB73jDqVWhVeoRiK0AMyqULRudX79+NdJBEszt5CqyEi18Ckw5cFaRUxTYZOxCCiFyW+1K1adl3wNc+ToBGEEHHB9tTJ6EUSLXzhZSZKr53qca/4VgZ+9Haprc044Nx5tpFWmH+Pb5lewQVBksL4NVv3+pw5dg/69VniZ0CuNmbnfRPHvotG+UfTIrWeQ67879d59/8U9PTt+RcoPvfz371Hx/uH7/y8np384u3+qv06vL84tf3158dCxLPS08vdktLo0xTWtzbjtoD/SqDIPxrj7A1fn5+4vLs6lzDPt8+nhyeXN+9fHD2ccbu0drthYM3T3xazxygKmVVtDVA+zKvET9bgdW++ZrXWpy7GkeLm1jkBo8TONgSLt2GJ6La59+fnsiqqRxMDBtRzFujTlSc6fXn0cfrm/EPIjjWRDeO62z6NyIB/ZaFAPFYkwXJIv+tEfiXLhaYbfBPjiDufpAexWYwz7ndPs7Xsa1lvROQolw0V3WFVH2+JRDX4Fc81mkIiX7TTLHLWQ8BykU5AsQs4eH9w/4azuXxe3MwboygLiXiwMmveC0a020KNE+TyEQ1E47EoEn8SKskgAqZQzqIoPWsCzHYzu316Ya4U7YG71Vn0boUY9+OzmbNkUK87gqlpY/0YliF1VGGdVnwdUFVBvYAB5/HUifbqFBy9DSoS2z3/b1dmMUoNHGfjsCoskIRhmCwvm/8G+LgkyNj+5xuwt0CcfH8IHvv4hAavCZSTvB1Do6G/SUKgbU85ePoGsNpxAim/77T4F/uOO2rhBBice0Z7VtzdJY6Rxs/y26Z8aN6EuXLWzhAgZ0cEAj/yJOQZCgM6DtBBfcoHWqwDEz9Urf2VVSbHxTkdQR20ec0PC0wkHJ4dnJF9h3KxdjND4ZzS0PzzSyUbDQifS0HddomrvDdLCkLzwClofMlVaF+pxZHCSu2VIrBdsGGmL5UHe60VCYU/g9nRkrEbMKht382hyYmvIlZDDi6amhBy6B2uH2AJm2CpB03KddwzTFLBWsMnUKEkF+RhZb13a0oAjAsy6ijcimRRstQGwAo06NmGs063S/ZtWkN9XAbIQXW5j1bXcDRHABDcjBRM0BivWGewpstJjT0D7oAFFDDyho2gBjWFPgtCGBPqTWq3Whj/VjyNVVDjKXgif51HKkuEfjo8w3jbjQ5Gc4jvm8iwYMw2Ql1kTKM/qJCETtjOrNWmuLMMHrKKti6szBjgBOGvKM6dx5JL1AXygs0fzy/SdwH6mpVVmD+pTNQpRIsXX2byQs44ugneFTpGsrIRumFbIcZWCP7TzMizIr/x+JuzoJF4AYXU3RK2SHdPpIh38adrJz+2OU1rp/OCOn5t1cHJhDdIh9fswVU2cEzi/VXSDn5Wupq2eDsKyVEK/OKzoGfSAH7oOF1N7oH0YRuhhtpJOz4dbloX6M5YQJhTqGTdWoD+dcK/ngrbLXiELXWakvVH9hRuJ3a5Clm/8PCKNm8r2p3RZmYzWDg8UAU1ppxq3H//TqvrvmbpyDlAK4omzmX1QtU7ht/01XYWjI1kUqoJKKKi69rypzEII1bNFf3gJ4Gw8taA/23Nbeby5lhLpge0cbUb24s5BH8QRdfuz9m8rOsT6oBhXf1/lCkTqOvGJqrx5/ce2/PTt/D4z3lstbwOxVa9m22+ZUEBqDHFuz83uo4YxmJ4IS7GY9keOszf5m2rD+ZTiljaCJHZ5xbW1dF4gaFTRqxXcsU2e0JV9bt7BHFgyHvkXG47F/Qjj3sUJ6PNJkYYZqpTXSf+sAoCkpIoVSY7YtBhzH+ZuUmQhEsQKNITDUi7qXPFkGhwjmmEy4uvowwqVg/aYWfxUUVD2PpTFhUIFmryND3SzADhprFZShP09WYOE+R0049k8hIhM96cN9t2yyRcJWteJOUmNSp9EaGxqrUkf0uuUNh7sqHIbG0tfWsSbOnkUwVRNkLgitGZbXTXQ5sFX5C3zdN3Hwx6Jfp1cfPlx8uhnuLa/4hvAXayYyWn2tAqwp4IksFjJq7VDH47AKPEJL0KYqHYi3kMUaFsnoMi3PQVdGpppF53tx2hzbwcri6cbMYquXVrKqLY2J23owwVxoVW6biVatN+b/rGIDexCmmXwMSuhiPI6w7jMTWg9yTPAkUvM5lrrzaxIwGAo6JDhwJsvRKCKtYyMOJv7ZhfGTbsG5CdaZbQfbOLKvRxXnJs/aejikHxHoNz8c1k7zzbcl5LXQoyQbmeSYzGE5D3uZVzCU4bHcN9YFGEKe0AshLwKpij9cxtuW+nWMY+Zj+lrhPM1Qt+SDYZqt41FtZVme9bC2d+UAFb6BKhp3Vye6sGwW74Xvp7A8UiVVWlFwrHlHRRm7SGZgRCMrN2+okBrM8ynx28kZVWuUiuKhoD7IHfHEuyCPHgAF4KCURPzm0QpsUJD1Em7wHRZookRicp3r0kpJiPacPRmWHvXMqOzx5KycWH+dWSuoMRSdak27nhV36LpwGKBIaCKZBdN2vudbb2BxjlYFk54Y59y5fn9yOaEHb49m49vv2rmy7+6eCoNTCk//ev1Z1E+qno988rEOujtfXZ6J65Obm4m9c7PnxHs9f9LVysNnDqcj6EyhzXO9msZ2nNGqlEHrfLtcXtdfk0r0Sdj4vqUZyaE44pwbWfxHOMtpcgs8AHDWsv8HTXSQSIXCYyz4ZD73yR3DWGdDN7Uia6pXQA9OdyZ7tt2KhvxuHT6wCX9Q/LJnd1P2pbXnVnjNUkRuX9huy5YdtJSUTaTWSIr+a0nK0X9jAkUOpvnw2UYz/NYB7V1gjRqMPH5x0XlKIgt9eipBt0L7TEgqWachxx6bc8JOGFTc2vt2fGe9JaI3Qv1xzC273LEwgwY6RVGNTVI/RYLfQ0qOtkCmZ5i8KaljMvJ3pE9wBKx020nB3+GImrW6nc8B1E6K6APxzuzgoM2g5oohqR0LC6Acyqb6C9BH5PDiEnAKWZZUDXJ36/Ag1G99vTzVirZLg2JUyM0TTtaqVnhePx7FRPyw79Xorjej/eUrdlFBDctOvQFuWYL/Q4F81JuxXKTtp61B3HqCurUNGNTgZ/n8RlN9NVuSNYKrg6RD98pqAhcLGLr0G0hwT//6ZjTeoehZBV+9VMFCmq4nmyDx1QEji0jwXm6I+ujBAJmL0DLUn2AwGkR5Ch/TuYNnX4ycsG5vvcag/YCH80Ibrj12K9tZI1Pt1GIiJh8tc+gtMZ0ZX1XhozGrd4ZFAH6hvmqmOu4OQBjlIJjTFfE2jNKFewSp5y73OakJ14q34kIj2lnURG8pL/bed0rifi+/XwaThYdqEH8QtugHPQvk2XRJGgY0ewsCF8EHnwY8lA+AW+NbMNCgmHvENKXFmugRt1ZrSXndw0K+NUzz0ZZE675B6ZtEvMZzWulQx9SX0426KyEph1qQ7l0h7LkI2zx4EQIjwxA2Gbz8TVSjmrCchnXSSOceRDvjYADbLk5vC/VuFdfd1uudbnlYHaeyqtIGezXFy5ds1St3Fm1Vk909C7ALUjL4Nm3BT0rQu7O1uZVDHNgqoklP1I96a+2CTioaeiiwm+P3RVKbc7dq0+52bLV/sZ7eVnx212bNY7bnbogx2K3NOjH256/U78I0NtqwFlXTbYWNKTyiYl9T8VT/vSvabOlgnauzrZytXmdn2DqTwT2/uQZeRNsOjKzZBhzF1oI7R71kYZ1m2b9ue9DuZYkMmRGmW7y0TT9g8CMmd01ox/Z37cn0htWP1vtImFnmIN1N25uhIS+mjenzJPOt3msnEXWw26t8YRa1n3b/FxlUHeDeZe93HSBbYuuprRhW2z3jT/LOLAVv5vWq8W+FNEY7dgX+LalCQK4BjLWxIGPFX8RHIAHFz1zqzGLNmyOW/GzRSDPZ+4NZ6BddRz96h7OYiDyVtqA/ZspcvJN8Qbb2NfW4keQIJMbNfT9KQ/DvrZn47yj4gZ7iNv49Usj0mnIKsGUAjD7dFSPQ/+JEcXt8t3fl+t+ZqNfeO5xjCtY5TFjU8vl37USFy3oymTPNLcZ7Z1rPzezJ+I80TZ2rlSo7b+1MlGwVfFGraoVPZYwFu2MHHaqiWhpCCijkFPO4cKn9M+soTM9EfcSaCG4+X6NHeNMqhdHr0l+4ssm5om+DdYm8gV2T0Sq5pjFIHnbihBoBVd1/hkO2fAKzQ1E/H+MXVPpEkq0dQyLYb5DtdBUz3KNOGdUnGtrLN4cyuSPubMcsubGpXjfZJGrWH4NORM+5IYk6qTny1ljG6Pv+h+C6BOw2J0CWVVgHyDKWnAefSyN99KWJkw/+Bzrm61BTTAAA"
  },
  "scripts/export_glitch_feedback.py": {
    "sha256": "b0157370725c3f0b8079858b087c8fe118443b2b64453522e64ccd90ed2995ea",
    "gzip_base64": "H4sIAAAAAAAC/41YWW8jNxJ+16/g8qmVcFqevaFEASYTDzbAJmPMOHlYwyDobrbEVR8CyZblDOa/78erD1k21g8Wz2KdX1U1pfT6dOi0JVa2RNRq28qSGFl0bWlIV5FCFDu30vW6kKtSVqKv7er9zW9E9KXqSNVpsq2VLXZEy6OSj4w8KrvrektUW0kt20LmlNKFavwzQm8PQhuZ5v81XZvGjbC7RaW7hhwwqtUDiRs3biOOzZNJw75XZRr/oQ6VquVi8enjx1uy8Vcyzt0a58tcS9PVR5ktc7wuW2vu3t4vQCp3L+WqNVLb7IoRY3XmKawItV1Xm5WxsuHyKGq6XAbmguDx3U/vbq8ZtNR2rSqgvz9klEB3R9kKSJ9Omp3489/+DvVoZSX3ci8WN+8+vfvl+vb602fwnFErzJ7bp4OkjFBwZXVfWNW1blqIQxqGnQZygC3My16LcS9qnYPzg6FsQWZ/dAutCb8Pfv1DRsrS/dqdaveq3bpxbyQvOssnr6alRoLN6UIt2m0vtpIuIRFchNTCSoO7XQNtK0iaLdeeDTjC+13XGUkEcbu1tM65RCPfuDutJSVcr5SajHcZaeURK1ar7Ra/ULXd4XcrWxnE9v7lyMMMJYTD49CmgdJlmQ3m1H1r6DLf1t1DRuMzExb5N6txkjvz0OW57sa/vXza1KJ5KIX31bX/nxsrLFwMojdWNZK5kJBw9s2t7uXSE3Px4s4iOibsroeHVOVDIq87UZrMU9VSlNzKE0iDfWkz6t7pIQzZbEgUpaTrGbNa2l63ga3g8n5bCwXl/y7qXl5r3emM/tpNLPFc+2C4b8ucfJa1LCx58wZqJCtHdmW71QU1nk6nwQ+khxZewb8eRLHPcJcRQMOhty7UhLabvzCSvHfz9mp0k9+lVtUTKChj4ZIx6KCwSJR0bf1E4AgAK8cZmFetlRqB6gEobMGxSKVO2NwKNfrJhI8EFViaoAQLi+HIZH0RTZSJ9ilTxsWhC6Xs6BTKHgAYSwIDw0PJs90M/LEKdrXL4VDj0cdUqgUmhHPLmRmdu/hl5y+ZVxlL+gpk/Br5nlwlmleYpCPk+w0ZtHrZ/r8ZpyL3fiT1wwYknKIHIu7pK/b26p4G5rT0FthMPTXzfgGUOo+hmfvGKD3AN+UL98PmKv6+SOQAJuDW5WUqaXeVBi/SgTGDOGeR9adJZDnNJn6en4o70wC8EGUIgVmc+Tzq1exyaYq74HCGDrwFk3w72uIH0qg2S2LdjdB/zxKPd/RRHCU8p+GVRgQYeg+dDJsICnDBcU3S+1dd4/OzyJKnAoIYH1yQpgFDITAHNiLKIe06DP5CQ+GQgyO6Js/MM9llwQx38QYP6RIcnoEwjbrjUWdnpJP/pFM+4MIDgwZmey8/VBz6lx7BzoRw4hzL9P6OBiOOdL/OgJ85AEtqDYrKfYSbbDlLA+G+TwHe0dK1M5x/ZrX3oWILdil2yM2y/A5OXvXGAantEoIKuJKpERBueYxbehYY3mMkD/l5kMpxNCj0bO81n/KVYztGwCTTlF0AzhK0HPYPCB7IR76CdvNmXyqdxWLO51fmcwXv9psPojYRR4OoHibu7gc7tKDKvGYZX45mAAY2MytMy7pwPCaNVSARUtgclTdpwFzK2Vzl/xgRPXGTi8NBtmX2hbqH6doTo1F962j42VPLr1F6FEGqRSRuZrCVljkyFbRK2Zd43gMADqcDA365tOz2olah/y0PYLOJl1a+tgX2hGX4wySl+SPzlCahdPJr18pAUFqXt83zt+PGyKOrJ1EFiYAXcN5GcFc2OThZv2WRW9XyH3lsTKAjr/MB/caNQfvPSjc6yjihM6wx6uzFRV37pgF7MB2boeXaF/vPCcfzybrsUsCsXwumCzRjbNH1F5Sa62n2wXzpvRgDn5pTOmK4VNfutwFm1cCuoyrkOA19xcWSljb1iR8FUDdCvrsVl+K9SZDPprYcJ0hNk4lAIfj1gmgIWtGYKFlyh+Acccu5xkVZx3bpEmGHv4HqDJFf0NqgniRgsevbfUqZjKJ/07U4jAu208XuskjALZgqQW/IYB7m7GN3ntxZALWUU1e+MkXgkFJVDqOBhJ3rdrAMxeTkdqcMQo/UQDfplggYL12HxzxeCpeDbYdSUhWgIbZtZ5TJaUg7Y7c5AgptRKsqdGihKmIpAEM4Ckipjh40hgupgs/RZocS2tfYsenO/6MOH/CbxauMnpBtAexAIyfPZjj38w3/6frDvxFGPy2JMOQBvUUtR8BN4Owt9I2PK3bG7XKe/wKF3Is5h0wGbtzvxuPnYtIURTZjk9JA0ZP+9AZiopdNHz5CGnXNxnekP7gq09tCVtZlUpei0AXrobPw3zY0dJe+c+Tv9NZ36jd+JyulKbTyPfWG87IrOF9ObuaiLLmIVzLq2y3KnINuXEfy6lFI//8e9YgXD3v8ZjEfox/byfqwoR+rCqHpzQxb/BhczYn7uOtq6fMFffWJ8ZvEhVfQk0Rf8xkikvA/joiJnZYrt7wqTe6GSEHj2HiOXL4JmefCN4dJvZDI+O9S1WR8TiZUeNng8ZwiA7oPTbn791d09zt5ulv/8/48WC71urHV9a+FJOOHQwMXOxnlNHYj9tC3d7c1ZZHu7MTPRq3JobNQPWLe9M2D1GxAl6jacQGVFiY4OcQPnZG7VaXYEwR++lBnFCOIph4lIyOAgx6z7VZgSrqALyYn/xJG1VB2I4A1B6mt29+LNvdNP1TLuYs3zv23Cc5ddHEee6MQaov/AaMGdER1FAAA"
  },
  "scripts/local_clip.py": {
    "sha256": "1c7fbc0c5788363fb2df16b10082809ef5741834f67d3faa5d781a3f1fab9108",
    "gzip_base64": "H4sIAAAAAAAC/51US2/cNhC+61ewPEnAWl63PRnQIUBSoIcmRmrkEhjEWJxds5BEdUh6sw3y3zukRK3Uxj5kD1rO4xt+8+BIKe8IRyAUIDrbQieOwXig69HAYAVha0mb4bgTdvTGDtB1ZxEcawTBSfxph+MfMIrWDtpEOxtqKWVh+tGSF0BHDu4wy385OxQHsr0YwT915lHMhjsWs5M7u3z02I8H0y34EIwuio8fPtyLJmFKpaJdqaomdLZ7xrKqYz6Dd59vHgqOVcerajM4JF/ud8J5KlOEayG9tZ2T8eT4KoXP0MmqmhhC0MZmfi1XYzBcH/MPzgnMdZsdZrEoCo2HqZSq7cxYOhuoRa5f8GPw8Xog3zAPHQhiyZobFtYFbN7bAavbQvDPHDamSTkbUiTxUyP2F3X8ERiH4hN0Ad8RWSplbtPEpT7BszBOQEcI+iw8mb5Hfct9xUxPnIx/Evgl5oZ6w0FWy22EPtCQc19SXXvnxCdQipqbWt9jrB3Q+a3hQfOWzmUlwAmdxUteiyr3fVFc2Kx7tHC5AGOTl/xlbkT6rpqRDztxBMO9uYTv0YMGD8zgq3TtE/bwCcmxr7wVNzshCf8O6DxLXzf9kEazLo5dHN86fn4tq2oX+QzH39++ZIzMki0y3EbMLNl8ISx5xj2Fnmef9fIRnJMxDmK8f//tPzHSfP/GfYjOm8pIB/3Y4Ufw0fbLz/s9T6h8RPCO5c8PLGh7GlaKbeT2iZfG4noCioOQZPneLjvDm57V18lZAG+Ws+OhdGEcO4O6Fm8ivys78MLZrpeHb//riYpvnBuz6fYGFTeP/D6uPpHxqDx+8WV0q3XoR1dmn+rFeX95tjbxL0+gKPjZKjVAzxtLNI2QSvU8ZkrJadTTsiROJC/O+g0dU0fvkqXU6FoyaRU3Smnb8uZbIWvQWsEMKeXV1cSKGfnziM1dYhPnlJnr5p4Cvopm3j8KnaZ3Bh86C/GR4QFCx9vlVeQy298H37yO3myqFfcJxJ6Oqztj019Eu3Iyr9Z21NZ5iSTBxu09qaedkc6X55fE9fVV8S+N1xg/XwcAAA=="
  },
  "scripts/mac_workflow.py": {
    "sha256": "c9ba747bed5932c5c75b3db30ff6cea14354a6e6c287651777d21ce187bb20f0",
    "gzip_base64": "H4sIAAAAAAAC/3VWS2/bRhC+61dM92KypagqSS8OdEhdpwjQ2IbtFAUEgViRQ2krcpfYh2LD8H/v7C5JyYqqC8mdmW9mvnmsGGNXqu0atAhfeQnvPsCfv8Pd7RUoXW7RWM2tUBK22HSozUdoVYUN4BOWLgiM5c8GhIS7Z7ul7/f5fJ4zxiai7ZS28K9RclJr1ULH7bYRa+gFd/Q5KBm37rQq0Zjx5NlMopl20ooWC5R7oZVsUdoB4oxoMrm/vX2ERYBPiqIWDRZFmms0qtljkuYd16RnlvPV5NO3P77cFtf/PF7fPHy5vXkgsxeWf+d7lgHL2+59fH7g4Vk3vAwvfHiKenjGF7XZsNfJZFJhDeVWKYOFUU6XmFRCY2mVfs6Iuq4RpbCLGyUxvZwA/UbxEPh4kAa5qEczEAaksuCto7H/RTeD9aCb5vTGZeUM6uSIg9GMcD1WtM6FCXQl6QHX/zQXBuEzSW6U/aycrK61Vjqp2T3ueMslWFHxHSVhsXU7Li/hJSK+soMrjdZp2bsKpyVFRoYWDcVtqJ5YJb5HoFY6NIvvqpGInNA1fR1Ff/qjbLzZIQ8gD/HIuLoWT3mjvnsmPPBp7SNsp7FGrbGikA5FmQETsnM2tMZQkIMqoR1yuTzNeNQ74Z8gGpTJwTKFxQLmP9gfFJa/rn7E8PU76/1/qvaGvZr9hZbvqGZguHXkMhZ0IyzXs05wqYgGeBmpeM2g4Y2De0cMNg19bEQO7A0o+yJpKTTciBl5dVvCW2PjWkJqXcNJ/zH0S4st33EBVBhKFMEIadFSe9NUux1sUKImkPyAHpOOmf1NYWBMiX2qOLlYk3rHxxwobqKEz3K4E43YwsPtt/ura2oHUibScK3UDkjBwcZJ7imQZAdjoXNq3mGUsdwVYfEV/cpJurDt+kkpSUQNc+CWHS2/jFZZZv02/XhEFOs0pZt4hbxybWeSl4uIeXEJjTA2IbN8TyuXlmwhZK2Wl+9WaQYXAYqUwjMvikGnyI7hLwiSlLac6mp1EpXXvNyhrEwWpHE83kpo6Rk/P3zPRcPXfohe0zTi9uyjcY31Ezvu7JxISZZ0VQysUCRTvyI9Lyt68M4XuFDOEreLR+0wA4tPwyut78WZVZ74TU5Ygf6gOkQQuF0EdmmkeWWSGFVubEVO6KFFl6TjjESLJYvhsRX8tIDl+wzm8xVdc2GEBhXKn61Op+g+BjeuvfjZX4Wxt43gHe29CPNKPYeaxuToToQK5Yaa7OvdA/CdFXXOhnTCmEfLvuOIjaJULbVxlWilbEZbRPmd5l/Q32BVBpHOrL+LcfGZN2a4ULzRcBv49367xXCoU4Oc1tqeiq707NPV9fTBYjed57/NqOvkfrYWctbzlcb7J3gBJCcR1zdoPPWNkvaTEIImF2/7wX8MPq1SjZkZuiwK3PNmRsnm3TNLs9PFzqbTIVsWIYbP0GHTvmn91ctLZOfsBa0i7Xw3sZHDJTs6XQUkg1gZ0vi55V1CsiPdKFqdjY68TkMTHIPHg4hb4V74yA7S/iSKqYJ9YrGW6epwufT6qq59hxdWFWXnjluzpzrnXUckJB4u6rKj/wyhZj/YwC9UHzLo5SwE4zoKBHnL3lbrTIecKdXIiDFozZSciJr+PJ5glUrWYmNmvdaaG8z9ELM+8eHGi1FO/gOjuo8fnAoAAA=="
  },
  "scripts/model_assets.py": {
    "sha256": "10aff4ff4942bf29f39454c19a2c04b001f2b6bc2beb049b31157affe52229c1",
    "gzip_base64": "H4sIAAAAAAAC/71YzXLcNhK+z1NgkYNJe0xJ3iQHeecg23KUWstSxXLWVV4VC0OCQ2RIgguAkiZaVeVp9sH2SbYbAP9mqHHsw+rgMclGo9H99dfdoJS+kbdVIVlK+B1LDCllygui+I3QQlaasColN1yJbEPqZlkInXNFmDIiQ+mcwbMmS55JxUmjeUQpnYmylsqA1KpmSvP2GYULsWwff9Oyav8v9SxTsiQ1MyhC/OtLeGxFdN4YUcycnGoqI0oe8+pGKFmVvDLtmolPbg3Tmpu4VnKluNat+FtR8Ev/bk60+J3Hht+Z2eyXi4srsrAmBHGcgVgchxGIyeKGB2EERwPV+vPR9Ww2S3lGUrHi2gR4hDlhxUoqYfJyTtodF+9lxcPjGYE/kfUSpJJgfEXuqc7Zix9+pHNCV8I8XxZy+RxeHdEHtwj/FBOak19Z0fBTpaQK6MdKNzUehad9XJwt/R40tBpucB0cykcichsG4diexYK0phBewHYD8aMg3D0ALhhb3Btst4yaOmWGBxlFEXKPPoq0YQb8qE2MXn/45yGNeJUA/ILQ7QGOS8BvcK4FObRvbmE7i5FI1rwKqFrSEOJKtFGclf2mgEaS5E21Rr8Kw1VQsHKZsmMvCWFkaXB0+OJ78pTgTzgnS0rDXsOO6VZdOPre2/dsQQqwZ0IG/NQjTttQIwzGG3llVirotDpFiptGVd6WnN95lIUedC41PegUT6RKH0Ucbm6dJ7TFM8Qd/DQRC/KXhdf1meIzvd4DwIyeC61FtUJlt5Bxq+dWR4vFYxfuB9oBZ5Qo7T49VK/7A4RDS9yyL9hy0mZAkvNkrZuSlEKXzCR5a0dE3kjrCmArYnIISmtpRFuveqrA1FdSmjlJYWdRMQOU6P2JHwGX9js5GAn0JDHlecULELsBmpF2cS8d7kvz7mCDnQjXCauBf00O5Fs7bIN7wWNGqg0dIQj398fzEAtKVokM1M27xXBSXw4Wbxmk/pwYxSqN9LKguTF1myPtUnAB8niES3qFNr8sj7a5bKRhRbzcGLB2QSAuwRhgNmXdK8zZVtFnikDV9NppSWRZFxyYrtN06JlCVCbInpzb4nXfr7bVjF4/QPAxQXfVPhD839zZR+47/g8GBocPT+YkKxqdL65Uw3sce1f1YYOXnb96Wrd+A1K/42ZI5dNx7teXDbh3CewLyzG3cHlPLsPKt5gqegEWsF7+O/KhKxNnbwdmaiBVzA52I0VKPnHzBGjKItQJZdxxhI2QzZfLi9fRQO+5txNSpm0EXG1FTsib1Qp+Abk8zpslOsTqyAD1eedBAMTSg7LXLHXkj/OZnr2Nzz6+it/8/OHk1bvT+NPpFUAGqs4RHfscK5F1tytc9JBOqIvAUkgD1hQm6DRf/OP9u4uTN/HVz+enFx+vMF4/Hg78vX/56dXJT8OlL4ZLPTrpVWsmoLEz+eElKblhUGMYwRDKxizuB0d/MrHDk+sH6FXoVg3J6NnV1SXB1NurafuoqI3uAtyDBoLqMJEwoNTdACP1YGCTQgDoXkIirXl1YJQFhahSkMH6zDYQemgdoRinj4YYwhq/Pnl9dmqDO8Dy9te+0GNjt4Mx325m+BS3GBsc6u+c14BpOADk/CeELXQl+qCQUDHATIM0i32vSwnFG+1PKiAVwL8FSDJMJMsi/XHarWLnK988bp8DY+DPEULl8HjdB+MvqpkmoE7L8VbbgnCkr62NFjLQINVCs/ylPZAfAYAvV2tWw0cD/6aiZmsm5h4G6DQjUraGDzmrGx09iqB3nAGVQKUrLYN0KW8VHRhe1s6N2Mfd8qLAX1EBYooCPIyMD00MNNrRIOgcnesGgigVeh03mq140FaxMEKRTr70/cm31J2dPq3zM9L7oE/oK2jXr/SFGkLUN12jaNmz/K0z8Sl5QZ6RH3xX+vTpX6fKxS+O7H3BeA928Eo2K6BTcAXJIf/R28dgiOYKafxWJLzbwtpMXP2sIWKw20/ilacr9AYkLb+bD5zCq6bkCpvgHffMydGga2nL8Od7q+Ph4LGie03uJ730QP49rMHjUE2WYdtbwKDJsQp+RThGGWPX9wEa+/w7cnontOW6tlHElpG7DpNDu11x6MJhdITDmmJDLPcBdMGVEj7cAucZXkXj0cK17W7nvnEfDqMB/RWFxJpp4fJyw8AG3aQsJ1Av6HyrRw/Hk8d2r/Rsu6efIIWMEnKC8DAInHb342FMttSGEORHmqZHGMHZBgiuGj6Mwm435bzf3U+AgRJoGcYoWVj2FasKapFnkjnycwWa6w2RFYSBZXAIO7o6b4vExj6a4kJCzoHtmipt8gMg2BUMi6IgKVPC893Z25f9HNe1RYaVNcj9Bl5Cj8HMJth///jPnoP7qcES+lZ96tCueC1jkWJyta8w/BUruXvnrmYWvbx7AR8fYauJP3usGEaFBeRJMC5c4RaSJmfMR6FqVXwJnEO/b1gBJGMRPoIdWXPgoyIFR9uKtN+zPondrUxUruFogb+isdJzwjGNY7meWOxLCYLHUoA7qlPZS35t0mo8GADWDSF7PPLnU/X/k6Ytj7uzQIY9Ojz1jIiVY2Lw2qJsN3/CPA6awTu/9yUldj4f8LYfaXBG9pRMKbWtWwtXvPFCytdoMVI0J7dIAt0orDdVkkPTBBtBUwFyrrW53JhcIlek7r7yywOttW53qh3auB9zdprua+yejqPnPy0blfBvrGtTRXFo7p9U8A0JBVzuLI90k2Xizg3BkWYZ4LHSUuG0QaPa2B+YEbaHYqM2u/djMCpANq0Dp3o3Oe1MfJfw2pCLD7Y12tWBlu00jSOcRb4faw8wuhY7nqTXqZ7sNauwPcyZStFo6GvFKjfuMh16W3RLguOS69h0DZOLhQVrEWXrGA13Ntwmqse8gWPD8exrV/qboqFHcECxjU4tgRM0hQwGL8YxVqM4tiNGHJdMVHFM24sxKIQKgeev/6MTtWpwbLm0XwLAW6JEjXhbxFD+kjgOBysjlqYx80sC+vx5myEAF7Op+eLS16F/NQKq/5C2pte3KfOt61u2gfXAdGg11QYajxgaPU73Lu1mMVib5BLacL0IRvdBId4t2ruE9n7N6gMleLvl1dofVKyD0c14gK+i/hbPPvYMYR9b6/1jZ1E4+x/6kcsY/BkAAA=="
  },
  "scripts/run_song.py": {
    "sha256": "e69f02b9d047310ab368c807d23ce4bded8756466c24fe131345968e64bd8e14",
    "gzip_base64": "H4sIAAAAAAAC/61YWW/cRhJ+n1/Ry4cViYwoKV4HCxl8kA1lYSS2jNgJFhAGRItszrSH17KbOqDov+9XfQyPGSlZZ/Ugid3VVdV1ftVBEPxL1KLjWjDOir4s2Q1X6iTv+orpjmdbJmt20/R1LnJ2dnry6vRYiaypc4ZfWtxrtWR6I2qWdY1SBc9FHATBQlZt02nGu3XLOyX894arTSlv/OdX1dSLomsq1nJNG8xtfMKnJ1IPyv/b9zK39PqhFcpTf8bfUnzklVAtz8Ri8cvV1ReWGC5hmhayFGkaxZ1QTXkrwiiGTqLW6vpstQD3mITHslai0+HpkindhYbDCQt005TqRGlRpeKWl0EUWfm8z2Xj5ZcSBLWs12kl7919uuZW1LzOhCd6d/Xhw/svn8F+w79//cOS3XVSi3QwQdfXnjbj2Ubk6dfmZsnEvch6LSwNiNfpRHbbCbpNSjukutTZxn1AX5nDsWm26ettqoTW0FEtFotcFJaVzGEHqR9C1fRdJpakN5lryRAEgnzrz5LLVXIG8+BiXcnbYS06XzD8PCMunHDYOx6Zs7NF+E7WOpyTMlmwomz4/k4sVYoTYi26MGKiVGLO0ojphO67mj0GCvateAoSJZs6OGdnSxZYG6TWP1iz/zjTRCBwxsGWN5PhOv4JrN32mNhlYtK3iC/BqzRrqkpqkLjIuA4Qu8HqAM+JBXFgZtFgdlVQzFaexj5vS17v/N30uu31X/K7jyE47Vti6pmIUHJdc3hLgK2rGrGzJaVMnPdVq0IvC5GPZEi34kElX7peRLGosyZHrkfxRtznci2UDl2smSvH1TaXXegqgTlEuYZMTput5WGooVIh11DCHqOa4IXGpEhgbVA4wtiwUAhCjhJpNEXA5sgCuw3P5ymVTWjG/pbsjHe+83rHJaL3N1724rLrmi4MfkFhgK4i0033wG5ECSsrphssFoUg/W05Oqlw4/LEJ94b2LlpFNX1Wtw5/QN7q6H2OMWWO02c+SEDlx5Xl1nMkCXcdh78OZ+inlFqXz8GtCi1Tb3gAscDlF+NngMdhvS6Hq+uKEMFZI226VsFq+vTeda4XKb8QzEPD2iMD6PydWBSejVqD9GcWVYKXqf/V5bzWuMOum+6a96jKVsLea4ddbhgBTnkD1Bz6nspNe9g3wKaIyPMmRH/0SIJMTU7dYx3VJPV1dOEcdF0loxwgVXDfIJwNauxGvlLXAOqOLWJksB2M1wr3SU4WdT/DwoKEizRn+mVxhaxgncLdJW/XCRn0upGk2rBeyCfVtSUHZanYndSbwa8g0SreiUzXhpIJOueiqFUDBzYuucdR2MSeRw8LY3evhYTpiDkoUJfhMV9KzpJ0e5KK9GYjLGmJdtLaHK/JPOQA0SN5CD/h8PZa2vCVTTUlKF6jOpYcfRomJ2fvsqf0ov0EcemGfeUUoLZdZN6q6ejoU4Jgz+SEVgJd3KMguMLXR/0/CrasUMFdRxhuY9NLc4n7nBRRRu7dWOdmLfknUE0bmYZUbTDxC4Xx7FpDjo3VFzWoTOVgaodruRha3zRrY0tPpmdEM7OOtnSLZI0zZsMsHJ0MuZ5nnJ3JAyOj13JWBq4mhAcXUKH//TQNR/1mGdOw1MvHF2yjSjbJPgVFR6FQIycjNaAygM+hMztptVk1xuCFwV7nDMRDmPxvtSJR8a2b6iTimfHNDccU2zblvgic4dnwZxnxpBwU4MWg6gTL5802Xfs89cpB9RHzaeRmVBJSDjl1Wk0aHt2+rKJbQmYMzUgc8TE8sBBSkbHyvwhZsrhCmc1UIzaPu3HbmfS/M2RvlZ9S7GK2SoxEEbkIXwUuhMR+zsKacZNxAUGgYpMKvexxjREE0aqUHuEbaIGD9CSFq0KniKPTkaShsRyNxEWZvzonUiTCEZCJHZNJQwJVj5QJCHQIB/qsUKKMlcojuw7RnLjrw2yaCQj2gOx5gfUMaOAFSVCtZNZSpFj0EXcPpj6ZiLWbMMml46MvQV0ZIR7jt3U6TSJg90Nd6DAoCB0SaArQtRkiePbs9fHmGtFwCCiFHU4hxAGjJ39sWk+Xb1jvcLoCY2EwXjiHnEMA6EyMVMkXbAYrAuvzvPFbhhtRgBSdyMIOJTNpQdiA3I3EeWhmPloDgJ4szVrgpb8EDQT95lo9Qh2gg8zt3/GJgSCzH+Rj36a2IrgZ77u2aPpz0e+Px+tzuPviycEzCPZftQVjqhVHa2iJ9tal8w4D6Ws6jkmf7EFEmE3suSM59zhDk1Dm+KIu6Ls1WZcR40OwTvfmRlkrDe4ktrymurgTV+vef0GA0VXNbU8WXcNrAE3AhbZ8sKgWtlDgRpC5QEZiDSTFGRJV8oGC41gtfeMQYjwm/X1pMcvps3NsrdTfZHVujQLZZNt4f8Jv3htn2ykKTPZNojiBk0wDPh3Lvom4WSYxQVRhvRr6VZ+vnr3U3r5b/b7+Pvj20lAvKUD6Bjvry6nwUDrcVZivgij2eyCeUXjhm56uYC5Nqgmg84Uq8hLxZq7GnPMBg3fl5037I5LbQqBwTrwdddDDb3heneskJ3Shy7q4dIIWg0JcsjwsnBnDmIO3yzBcfbCFPohftcNb4FBmu7k4t3l8WcqOGfxazj7hmdbbCRmsqc2JbJti1KpE5K2P+mDLDUZkMyrmc/rtOK1LFDUEp/nubhF7xsO2O9D7whNQX0tH0jdQqqbNGt7EgJLJeOaYiebZBLMu5lvT4AFHbtZeg/vJX+IB/d5VvyeoOXu1WG/rE17jcvJ0DvvsOO/KVqI0IJDb4zw4KBKbc40tviO3wbD+UreT48HWBkIhzrbwYVNr2bEg0qjvjGK4uHpAakDzrvvOZJ2uH2EUrzE6fOEaapuY8fbvK09TmdCEIaWLSoTCrCb/CKWoAW7vKf5mNql0VQqA8vdKwnpulvZzwk65F7icNYwnQqD+9zcHI2pwXaPmAzuaWd2GTexD6LiWy7p4UmWrKSWhvrab9gDR50qG5Q9m8yIdzQ1UuHpQK/YH2Cmnth76flR1pgi3YRmTc5ueu2HSTSxvs420EHQ8zuVJEJJlHalfmMhUYd+JrWxQz97+xkmhFFQQnWYh3cP6YHwtsYllEcv7zH9+kdoXtSuz/9p4jzWVTsL84HjXsBbB/yP/EZv2qF/5UbmLmeqDwcmz/Hhodp1YvGTETTns5xe4DlLIVVqtINwKnpy1JNMuIyyuG/puTy02ZKMEsXeMCGIZdjbbxe4yTgZlmyI6GQU99FiHtKfgakVlz5aGSYHNik/++HrIQCQ6qX5l3r3Pix8/ka7/k3JjyKByQVlkkYWCx2X7CfxcNPwLn/vKV2BCQqOakCWMITJHG2+ALbmZfIZyEW5Zx9VKOfKPwuXfv0YPQ+BvkmjxQKWSVOKlDQ1FTNN6VEiTQOrk32hWPwX9w0l7KobAAA="
  },
  "scripts/runtime_environment.py": {
    "sha256": "ba615414d3e8acaca8e00d6dc268708e1ee6c110fcc1106acb46ab760dec2955",
    "gzip_base64": "H4sIAAAAAAAC/31UwW7bMAy9+ysI7eIAmYtdC+SQpumStU2CLR02FIUh20wsxJYMiU5bDPv3UbLbukUwHQxRenyPFEkLIa4RG3i0imRWIdhWk6oRHElCUBqoRMhLzA+mJXgsFWMaiw7tUek9yJbv2SOXpIwGqQsg2zpKhBCRqhtjCYyLdtbU0EgqK5VBf7xhM4q+r9dbmAQjTtMd06fpKGEBUx0xHiWNtMzv7r88RFFU4O4lwBT1UVmja76NrTE08UxjjrhEzgWLycpoHJ1HwMvfv4j4/UAg3OeSM2RAwJ2BSHqRz+FCBIyP3jHmT7D8Enc/09l0tpinl8vv4rxnYff2eNY5jt9hN7+3i/UqXa5+bKc3NyecmmcqjT7tdbFcnfDI1Dv4Zrk5HVGjmiFucZUu7i466BBWtvs9V3Unczwr2+yDy6/59v8uT0hDl9vNzWy9ulp+/RBMLampDHEr9Oi/4TsoKD9zoXKKjUv6U1C7t9KCcuCrC1g5fDvuarkzFrSsccwUFnMy9tm3cShfwrDaxX1X+PWKSeoD7+O+3SZb2zIDPilHqTkEc/TqNIj03ks9cLyObPxK1kE/wa0psALXZo01OTqHzrcv8CQ5VWAYLW0IM2MOYXQK420o5RHh7ZVSpSulMelJ7zhn6edTF9IWUKIsKuaGTOYHZJJHxV3Es5qXUvvSBJkuL/jWco+hhQNajVXy8d3vfc0uprPr+epS+KzEdL8Xve5lF1ygRVisb+dj8I0UNmRYG7gdC1Z0Y/5FmKdncEjU2VyU2bT/NXQjidRaPRSP/gG0RpBAjAQAAA=="
  },
  "scripts/seam_diagnostics.py": {
    "sha256": "dd5a2b55cca766a433cf3ef47bc18c1d9f88d0e66bc2c041d702d1924f6e43c3",
    "gzip_base64": "H4sIAAAAAAAC/5VY3Y/bNhJ/91/B471IjlbedXpNYZwLBLkUfWqLNrgXwyBoibLZ1VdJatd7C//vNzOkZNnyJncGkpVEzvfMb4bknH8+to1xLJPZQeXMKlmx1qgnrZ4tkzV80ftalqxS0nZGVap2lj1rd2g6x/aqVkY6Xe9ZY5hRbSkzfJFdrpuUcz7TFXGXZt9KY9X53ciX/uVP29T9cyXdYVaYBpSAp1LvWFj4DRfCsz10TpfD24vtH7tO57PZ77/++oWtiSISotClEiJOjbJN+aSiOAVN0IrNw3YGtCkKSnVtlXHRfcKsMxFxWDDumqa0C+tUJdSTLHkce93Ivl6z3z9++ZyA8TIXbVYl7Nlop/Ax2GGaJ1XLOlNsUF8u//F9v5Gsn81yVQRXC+uks5GVVVsqG69mDH5Z09UOjCpVPazQQqvkI3yv5DGK5M5GoGanYlZAPOiR6Zr1BAkDKbIr3fo+BvPeLz98/wMxMZUlHuAI+xe4wXZVVJSNdIHdfL68zRG4kGYDO6aLoKsqrWL3nr1ynanZK0dlRb4rLF+x5f2cBJbN/uE+wpUYickeov2lqVVC9P2Pg563yWGBqNGQt4iLrgTfZhLSoTAyc7qpgQ+aOvLbj2uy48PXzb228koSBEPI/E+ZQZqJ7CDrvQJJQ4gCt43e3g2Pd8utD5pGeQZJomXifXszbqeQM5C4rcqcMF0dwb/E67V+H/IGPvWlAI+jKqBVSL09LGMKgiNlbiPchZkPEIBlki9wS4obOBJDjjt1dFHs6cENuL7h3g4BWKD4lv1tTUWxGrxipAY3/Rvd+dmYxkSIOqA1AE4m66bWEBUwiz3+/B+vkxcvKUZeFMBN22EFeAUjjkglci33dWOdzqzg7B0BQIr/fQd1flDHzeqH7Zg+rR5zbYL1RmWNyTHzN1v6gP7PDl39SDEnw+jV8u1mRX7dnm3Sda6OQEs7Npxe+XZYhuQEQEXmFmpe5ejYdF82u6jgr7R5df8+P4mPYr6AoEBwg5fjs4SCyj1witGtD6uLTMOM90ZcfsffDqL1ePF1EoWC/6IgBBJyBsOH4fCqjDzhdT3xs1phy0XaBB0399tpkgyKkpF75SB2gHCd5WQSD01E5Zz6zU2DLo1BQAb5Y6ke0yExLsUAcCP+84TRS/osn8DDV1XwfygIbqkbRwqk2gqjSsjRJyVcQ+WF6x7dI9xC5J7txivjF/l29a3AfBo7n2kL6dYHCUqcICVHaZDWVueKuQPsARVGcQrYgiUTepNX6jq/BnADZZds3ic0YGSl7DdV5Z/C4IAxIZqAjLkuCmUsoyZYyVoXyrqReormDqBc97W9YIWvNzEukJ2CZFTQFDUEmwJ4tpDmgDRr2heMMpmXDIxHGSufKQ1Gpd6XOyz5Yqcapbj6VPK1yuchZaZ1J5/TGq3V2NiG5IJEq/SRHk/TmsyaGoalTs0uP7Yv33ICWvCKwRpMiU+XvrjlD9icEPv4KoaBRyrbVtV5hFvImHGRUxX2O14Dvq087GE5ATzipGKcsLC3zrEph9TxnykZoBss/IQ0ccbQLvPOI/2I0UVeAtov58gDGiEPsyns6eNMmn+FPZpbNbkqRT/XAvHgA0IGmLdwFLg1fp28T/7OftLGOuaeG7aD/M6l0cquCDkhXyCBOpMpnB4Bw2QJ0wfbSesn6ELTAK2PTMLkkENyUb+jRuc0FFnquw9tu9mNwVRlNM7et1oxpuPbO9VRW7An9qPK62nWIyjZ3w8HJNyDFN8SR/pyxlEeGAQ5aJzAieqchUDTYyOxRoCEcacpOwdIu7pA7152UHv4dA3NE57nKf4G/H5t4PiJvEsohQM/UzAJtoAI7gDBU6UfRs4A6mBCfES8gNqm0GLOTEYjXOjLHj2CSdSr6pHgTAYvZ4edhnmjlDtVJoz8AVDi5aZwMKjs2GnBEb1fqbyvQGkCLojvyJ2t10G9czqOGmToSbE/7fnuRbHAhjDKi/+1eQVXU/r3nUoWTpnhuNh8o02R9K+YcakAwTjBFY3Ob81D/bzmN0wHtsGN1HnX1/MfbZ/uJkhpOisAJTGfb1HdPWyvMfEde3MjHDp0LvruS6XoVR8dqMa/GkDAA/EbSr+biu95vntg//RYe0EXSt1/dI0DUOz1mYgvVeHC+fMW+icXHoon5EbvD0Sv65v04Knw+dIvycjuKddwmTFKqaG79uHv+8h2Sm3VHgEU3RmOZ8s5mrlazkndqRMa+B7Q24sGighJ7m7ZFAOjiDjdXp7y75Xu6h1EK1c5QN8ey2gbEOpGL0bxC3834TcA4WiZ5C/e7sw8DNfCGyT8TKKOcHAuX4B+1zRlNHhqPbjAH8Mpqj/65BjO46dRCwBdYVKjmvkw91piCiyX4WU8ACwWy3OQQtwuB6ZXAoiT8I7wNooP0BbEK8ggK1f7q4FpuKaJAstkFG7UxSuJWQvxgj/b/tRIlzjod2wXK7owwu4Dfg4FtOpR6Nq1fDSBbFo/PyJ8taMZNBxUL8fP7YQTdAPsSfwPfz2HQ4umczDLG2oV4WisWNVZmjf2paaIpuxTGFo81C2ZH7GZH7Hx/g7v96gz+pmELgV3L6wECaqGDSnEE1SG+V7hvZa/+5PasGdJp5A0NLjzDde5ALl3oB9PkuDOeHxP5Lf2S7PZDDJKCHSWENQEhKikroUIjYBuFg1EpL9lTD+afYeJ+RutRDk0e6NbbDxrIfImEyIeUaYyz4UMJBG/u8O4Jsy9tGqNwxFq8lenoeWvv5gudCbYjz0rcKA/yMNG4+uG5Jwt43sa3JdiyngljAaxNPXlXdXayNMkBNG1Wy/j8T7+r+HKY7inXYG2XmA8+y8qG/g/0hUAAA=="
  },
  "scripts/sequence_diagnostics.py": {
    "sha256": "69fbc6e8e86f395722526fce18ce95abe91a553be117e7037a9db5c9a05736a4",
    "gzip_base64": "H4sIAAAAAAAC/51YbW/bOBL+7l/B036ovGsryRbdD+npQ6/oAQds26DtHbAwDIKRKFuNLKkk5cQI/N/3Gb5IsuK0uxegTURyhvP6zAyjKPok20ocmNlKpsVOskxkW5mzsm47o/GLKbmXSkvWqFwqZhqWl9qU9aYr9ZZlTW1kbVihmh1Iq2rpjsmikJnRSRRFs3LXNsowoTatAKPwvRV6W5W34fOrbuqZZdMKQxvMb9zgMxzS286UVf910I7CHFqpw/nP+F3JD9BFtyKTs9mnjx+/sNTyiTkvykpyPk+U1E21l/E8gVRQQa+u1jNwTOj6pKy1VCa+XDBtVGw5XLDINE2lL7SROy73oormc3e/6PKyCfd/evPl3QJWEzlvs51XSTV7WYs6k+HU24/v3//ny2fw34pfX/22YPeqNJIPVlBdHc46l/Cvze2CyQeZdUa6M1qKHc9LsakbuCTrTaDLTS0qro0wejab5bKABJL05Fp+6yTkiMF/wZrOwM3z6xnDD3z1P1GVuTCSCfbycqkl3JsziLRxMjBBn7Vo9baBWNuuvmNXFDq1/7hk96XZgisTlZEKUYKwKTclpHGxQPeMbg5uwdLIIwu36IUb1i11mcNZpTmAlIyVVI3IdUwcyENhN6G9iGjhBiMfTDx35FaZs6TeQvkFHXmWHqf2ZdPp8zzkQwutd5DheQHIN6ZTEhx8CiQuBmLLMO92rY6DHogPOJTfyYNOv6hOzhM4r8lhjHmylQ95uZHaxEG0hoIbfAP1KvJr0drZrmDxsGddxp2TdbRm/0jhdPjL2uiZbcuGfnBs4NS1yBKKxazZ7UrjDvsIX0VIQn//E0KhtTSaO/0dmbdFSDncXpQbfeFOLm+Flt60Y47BK8lGmjhyOVI2Ne+NHc0t7/B5SmtNtIp2sGvlhCCZkebtcn/1yt4Z0clK1nF/WktJZrF8r3wG2egWJcASidTJd0o1Ko7eISgyA0wNqcD+BY6jBKPgobxhO2GyLSWNFeUCGhPQaq8rvOc8A5wGwnGFRHXiEuQEAUfOs+L9k/06yO6stIoAJUH2MzTfU+YD1GbCgExow8x9EwrGoI7n44RWWFO5XjC6EqG5Wi/wz24VFAp1Lh+oysRXC3Y5utkyAUxUFvdAeCrjyhKC11Qnv9Hz0U2nMsqJaZpH+LDcVpHLkCnQhISpAXWOS1JqrmQlTLmX3DRPOM5t8rj4dRTWwv6WEOQ9cx9/k22i8Fpj0XIZ8mM4XigqcCEtf6YImDLuueQdQgXpMM7x4Qo41G1MQ3t1uX6WJeqjUR0B3YR2vDFy5/lgejvqNfpoh4ai3sj8NSKrJuPvxB0VJGo1VFNVIFC2ZxlBAGRCHVRws/NJ8eLRxsH15cv8yN/wx7PyHTlp+nhG7+OLIRCdjOmoCsf+uj48FwOynMRO6KU0+9DU8gfWKHxuQVFK7xPAcNXV6XQc6U2tCFUir/6FvxGqUI8yQn4vECV76ExCiNoY/c2FUECK4Qz4nJz4kUs/BpEdd9sssV0HrLhtgHC3Eh2MyEx1oDgMxWVQyMNFItpW1nn8GAUTcFeOrAWia4cbC+YTBAvUpnmFaBm3hkXSYHEi9fgHRwUKbJDkuk9Hu2yzDBF+4bu6CBDRVYaOWUMfT0KwFzsvMxMPsWHFSk8kzICfKE9nds4pnNr/fQPxE3u7ldkdOqwqdOm3ElgqKVGoeIj6YGNIqBKgiXYdRjWNOrxmNbXyDK2ost0mTlIEY2k4ntg7XOuV7O5AHPv+2PYf1ICi/efNXfpv9HQ+3AnJ20aXBDIL70MCdVkj06hMxd6vI0SwwC3vEb2+FUTWRj7HHt3x1YtzxnixtlkdTTl5aUfJYYeFJGvaA2WClyGAahSKB2S4CIGU3It99Bc4UHhN6LE0oR46+nh00EWQ62GCrVYhrtajTBBkm1XrjEvWpE5Q5rFrmE8kCTNMsqmATtHPTpA5ZXyb1DTVEY6CxeMgJ4J5Vz7YP48DRtBlcNqtBLjRIDTxorifYPrUQkQztksRgYY/OpbHiYF6JfxMwkMMIePSPqCeHvdFkaYbe3I87gzQ9Yxzn17vg85C5t9h2nt8YPkT+yQzm523B4ORVBSYgvqJybjszFkH0AyL2trcVbSFawPrBllmlGAbAfPbXdvygTwZo7lvNZ5RlECbQGkVDLk+20+cYdIrRiymEeprS2Dxl0p8KOnsfksDytgep0V8JGzStTSJxiOMfEbRM2j6Xd/D1jQkpzYbULQJ9SOYv67RCSEtbkV2BxSnVRpdsGJby2sroK0BFsuwEJrbSXUZiha6EV+GaMJdsH4euw4dE6WheKBCMapBLy+fsEQCE1X0mbLZwX7fMS3CG03uH2lWV4vLNaUuug6A/j1dhoFGJ+hDmOhMg0GjzNyzxfJbh6Ef87R/Ryhx6v3NZxSTA9sLdUii42yCZj1gR6O60QMaGTcYGh1R7Zf8OwTsMLxBOEY9Uf8M8QlNHI0WdmawKREUsYMR5uY6r6R6TUCjpdpL5qJT2+wqRFnRfL2TRiCERP/wMAzI7sLpeNx3oOn0ESkOE24a5tI9IqRRF2/evlt+pkHxKnkF7X3opD5wLBS0TVmblBrAwanY5laVdDp9LpifiXeiLgsMoekPJ+EFyxHtiPyel/sGs2HOLQp6qRiO+AWMMTxrO7oXzkiDQ8JI0y9MYjR9eelcbNRhQAD/NhWP43KIBvmQydbY2fed/RNZDmWZJKAYjZyWpM9+m6BpRD612WlPp0X0SG9+sf2aJ5xTkeP8eM0e7dK4R7aY5NoUyslqJPDfjGkXwTuAcjzE6o0zFfXtBHXLpq5O2q9bZJYsBKLzNTKykIoiHyy/dejLNFsuvdWGKKVHUhplwoNp8kZt7MRyY3fiXOpMldZ+Ked5k3E+H1EmIs+58CRxtFwSCC3sG2l6Y8uzvzx3L0rfI4VV/l9SrxbI0e6TqABbNKkc05f03sFxyjXPwP4iFjqe4PSTt0s6lNhnRPsXpBz60KH7DEke8Ho9bj6Rk5i4/qs6gya47zefdCIv1r9cHf1QgGCtSvaD3vSIXOkqMTp3MmCged0cNYxSVJ3ejqyIcm518WYb5cMYMIO647fSkxxDSz7VMrr5/c0H9vHD73/4qYUe8AH59WuPpqbMxR0Wd50wDHjZ4e+vAvXwTtQTSWczyBmyjaUpizinfOA8cte65Jj9CV6i/hhYGAAA"
  },
  "scripts/setup_runtime.py": {
    "sha256": "ba2a61d143bdc7442c613a6f04412659e51379f413103856783b09396cf20d6a",
    "gzip_base64": "H4sIAAAAAAAC/61WS2/jNhC+61ew7CEyYMv2uptNUuiQ9bptFnUS5LGXIBBoahwRkUmBpJR1f32HpCQ7WbfdFvXFojjv+eYbUUrnGpgFwkglpIR8SIRRJb7JSV0Zi5cbomtpxQbIi7CFqi3hBZNPQj4RWwD5XFdbC5o8g5ZQJpTSSGwqpS1h+qli2kC01mpDKmaLUqxIe3mNx07Q1KtKKw7G9G+2/WMDsomCiTaODN8IreQGpO3MHbiKopurqzuSeldxlq1FCVk2SDRggg3EgwSjQznzMH2M5lfL5YUTppxN4eT9Gk5/mk2mH05PZlM4XvHjFXw4nfDZ7N30dMJmJ8fHNLq/vr27WZwvUct7GhOKweZKU/d4Pl+Mbi1Uo2nynkZRlMO6jxIrVigZD84igj8NttaS9OZQOXFZOzMxveVaVNaMg04CX4ESsXYVSirs01rpDUkx7hchZ+8ogdIAoSshWwU6aH3zAvgzdq/zijaksr1XNCyMNd2t+/VXoU7J5jkXOvZymXpO73QNg15418MEs4wf6JOwdIjlLJUE9zAaSTXqgnAvCmsrczYeo2BRrxKuNmPGYWSwZuP94iXBEmIx7iIaPA5DPt8ZxGj+1oCLbC+W0SgHy3iBzwEJBxwUwHJs9Z4PL5GhiQrL+vfONDQjPwxO4rfF+SeKHix8tcFBgvKiioMjY5mtzX92FdRDVjgbHEomZDgi/DTjz5CP3DCYVKrXYXTA8Kn+kLa1IEq3Me2woZlAnN0EPC+0VjqmC4cMRws9ceRivQZtSCAAzy+El8AkYr4RRij5M6lwHkE3QLaq1uRF6WfCJJKPcZxkAIvm6AmRB9wqvaWDQzPTYnyllEXXrMrqJt6jghbUyE0XEjMpyy6YuiFCWoWeVohxYpUqPeGg+47fAq8dGQJdepWohn52QJpaAx496zkPfQB7nJD0L4NMmEsU2An/P3O+N9WtEWE86+0PtcsuWcjmYy3KHHTsOD3DFNJfGFocJNyvg7iPLZj90SVNEE8hdlNXjnUNscjx4ArRl9qQl0LwguTKB1Iw7KzTFaHwkCetwRvAFhtfYl97X/YS1pastogA1xfQuq5wE43XDLPId/UmzFrYVLhi0B2UZbD5dvpd6bCMvLZsVYIbgI0bA9RvhyMULkxQOAyGfaH2f7SN3unVTZpOkukkmbjZwaDTvey/ZY0WqId7XTf/os9103N5G07X1x21++P+ekwPbcbYQfMb2Qd6/yW7vrn6vJjfZYvLLxc3V5fLxeUdfXRUtEcyuwUVbOAQpX89eodbg+bqxrPVVvLQDiSJP0C+aQ2dJdOpKzR/ydMugO8o+yGHb7evcz/yztuPCOQXXjhGQuzFR/50djQMr5Msa5DKkLLwG6KXmd9/OiesQXg6iO2EeZ0zN339FXrrlZbXt691CmYQ0DoOuivH0DI3Q3K0qczRwNPh66sEL96a/0c0RhHCLMskwxJkHmJZtsHVkGU0wMjvJ4297D7cknP9VDtD1/4mzsF44GIR0izLFcdS7GkmLM8z1qrE2MXd1DDulXA3KQ2ZxXjwZQFllb5l5OXvJIcKkwTJBWARKmXRnECRLa4D7AErya8fW+ihN7coW//+z0Vg4p4N3SlpA9mxYD9AYQhwxnZ3oUu7mRpEfwJ2k+TPJgsAAA=="
  },
  "scripts/song_audio.py": {
    "sha256": "637781e09561622637fa3ae542b9fa662df5cf302d35b9aa787902ae07670bd6",
    "gzip_base64": "H4sIAAAAAAAC/51YXW/bNhR996/g9CSltmKnaAe4dYBhy/a2Fm23F8MQGImyuUqURlJx0iL/fffyQxJlJymWh9imyMv7cXjPoaIo+kzrtmILWvG9YAUpu6paqEbsSStZSyXVvBGEioI0d0xWtCVKc50fOMwoG0lum04UsC5vhGb3WqVRFM143TZSEyolffA//gGj/ntN9WFWyqYmLXyr+C1xDz7iA/ddHTrNq/5Xd9vKJmdK9SMP/dcjvWOz2acPH76QjbERZ1nJK5ZlSSqZaqo7FicpRMOEVtvVbgZrU9w65UIxqePlHMKSsbFwSSLdNJW6VJrVGbujVZQk1lvaFbzxvn765cvNnEhGi6zN6zk5Sq4ZfnWRSUiYoCJnpA+IXr156yeafJiZ4AzuZDb1cyX7t+OSmcHZbFawkuSHTnzNlKZSq7iUtGZZDsnXc/fEDKnNakkunG+uYv4JDibrGYE/XpKRBfJ+Q5YEiikaDV/eTxbCwHgHawH/JOWKkb9p1bEbKRsZR38ygIJoBKtb/eDShdihYoBPTauKSaIPMKgPzNqOEmPVRgdF3C53ZuB4gDK64e1itSOvAl/AtVEcg2d2QUrblokirrmInzaxmIQ7D1KzCCYn1k3JdCeF28WVxx4XluHZiVXTyZxBBTrddn2FFINTUmCJhtr0Y64y4UoPZzs6AvPcjttZ1ikAKi8oIMtvpjUcUhUHe59snIx2zQ5UHWBLC1S/qQeM3Stl91wBAJMh2TUVvGQKnUVQp1VDCxU73/A4YUpSfBRhBHBesFPELpfOeuytbKNG8j0XtMqsH9FuToaHQTTho0lk0S4hP21IPIptUojTZAxBncf3DQbPJ83xllUQoCK6IQUvS4ZdxmL/0hfhHenAEiWCHWGOZLlu5EM0JAA7KYw2siBcjELCJgaBhG5BtlyF+hTbtdsIe5qL2w/5JIY2zof30WK4cCc3hyO6Z8U7sFV2CsOGEAGCXc1GvrvD4J024w4r9VcINnZ9d/NFdg5NWj4M7nBRNiF0hl6f5geWf82suXg77opxVJYw6xZcmZNocRfBf4Zh4JeFOjTHDHaVHPI3D0KPINlAQJuis/Uz85sSPwxGLRU48APAEKzW9yFmvxb8LsFnHWMQW2c52m2j3vguQDk22Nrwjiq5ABqI/cRk3H9789CY3y6X6fLNS8D8vaftjx9+BbJskUXU0IotGhCKpGuxjEAT0BY7DekZPMwrRjGm8dHFNKRAsNHQXIfyyE6cVqVu2d4WRTRKw6bRmQLZQR5me36C0b5oC2oqdNdUgL7NMv3ZrKfSGTDcZkZyfHBlnuZrip9AyZlava2Ym2uixMoacI1gaYIzcggpyDO7mz+cVUsbG1IxoBU7PSGXl+RqXGnPT45Z3c9rLKfj55dK+huUDJVVXzrCFbHVBIPsPgeuVU/UccxvEAlqtJAHnA/JhDFh8qnMUKHCmDt7k97pLY4SxW0it99tX1oHcILCuNa09u3MJvpxF8ZhTOyCVslBct7Prc/YMJkAUMCRYY7mJ31cgNt4Uh15fDfL18vXxWMIbJMHtq+xf288ErZXF8bo+uqi1xETCeEUg0p2gSlDyFkIF2t9AhcHmWD+U5pr4ifQsUaJY6R2av7H0QGSCwIqgZLEVxfxRO2Mt0lG5TJyy2vYgVoweXO/XTi74HtL+1M6wjXhVIMFr8d6PFjbAw6swcdwqQXBsNZUDyY7EESmIjYehBL+wkYxihKGxz+f7jKGHfySsMKGin/E5ZEa+h4paDE1zeCkKCSDNVnB2qm+WZNAokQWeBniGZ4ZKX/WY7imaLDSO9y72pOPH4OS2BvBRD6tJ3ro/DZTXbWeyiaTnXtWZC6OPeW4NzRpv2O/lc+k8Rc/H2ch9pCCz0rHQek9Lz2wMbaa3JgPSMLoUmAulKmstWQskM59A3Za/n9K6f5uFXNQS4BEuPxN19wCSRquD9sxMj80shivBq+Xg1fY7AZbJ3nvrZn1z0wEgQKnwmiVJLB+qkimUZ27GHq334dhjA0/SxKwcPXjk6+fpbDkuQvpX/B7tVzgHq+XC7vS08qR6wNochv23PX7heLfWPHybdUCxb4LsXc+j0LkJVZDzqET7ZkeUGEIAB8ZfY6/JtcavLI8F8wNOPVgPSB1By3mQO/wVrFnwrBfYXaG+whwJBve0zhh0N8on+YK+De4FHQXJ2PxjALfgLIuwNBy5gnZ+GQDRxh/4+2Z2FxiRiGeEVs4JZDMgcqCvKGTxuLWt+rdS0rqM7pl09Yra3tTU8S8gcHCtv7uwwWkKZooI5BA5nXAxu89Zp3d3I8GrLMbh2F1w/U4gy+5/QfgD5JpHde8ZhUX44uXx+gGinaPr7DG1VnYHcMrpnEMbUq82MUmpIlWOjK+P2CcVvtBD/bHEkyuEvMiwP2+JivCKnB6mb5Jes07dAn7cBXqAAMWKgSrTL8Dr1fJqbox3A5OgHrxgssYT4zyMstP1lh8b83SXa96g9GLeLWwAaKhQeBZx3vTuws3KeCmAfeY7tGIF6KvLEb6FybjRT9tnjxZz534z+YM9xfyomGWKnJMsYEtiDKOh73pz/mg4WwD8u+Sktl/bzaIL+4VAAA="
  },
  "scripts/stream_process.py": {
    "sha256": "d48ca0b293dd949600f01918b00fae195963bb253d09ce4447c9fe63c2111741",
    "gzip_base64": "H4sIAAAAAAAC/41XbW/bNhD+nl9B6MukQdWSFRkGFxrQLE7rLknd2BkwFIFLS2eJtUSyfKmTBQH2a/bD9kt21IttRWoaATEo6ni8e+65h4zneTOjgJYkyVmRkkJkmjBuBHln5Z0B9YMm0zuTC06ENdIaQnlK1gCSfLEMDJE51aDJV6bZsoDI87wDVkqhDBG6HX2xYKF90SzjtNi+2aVUIgG9NTY5hpMynm0nWAkHBwcprIiyfIERZpD6iShLDCUkP4Yk2aTxpeAQEuBfm1GSQ7KO58riOAeqzBKoWWhIBE91fHQcjA4IPriAKcFL4IbEJGWJ8YWOmlnCVs6AME2cUwKFBjcRPF4aWZlSA/70r/nb95fXlyfXZ2fjq/Fp7B15IXl7tnh7fbI4ncxen5yPF9Or92+uxrPZ4uT11Sz2Dr2wcjfwzD+cXiwuJpeTy/n46s/X57H3s9ffWwrpe5VpswNu6cKtLbWhGWBqHG6N73KT1OTREovGaQm+NsqnKgsCshKK4AiLTxpsPx6NbhwErQ3CkuoNM7nvRfLOC4KQDPprlx/eBME2CGUgxTBcMaNScGEEZ4lffy6oNouGXnFrXH2RCrnoexdQ2oIyfBVItlctFVNmaClZsaYcffBMW57hJFnSDc3/++dfBGJVWJ1XNAgahxXZ3DZb5kVTIYHvGOXohH81m/aQDjG0FLeO95dOpuNv1q99cBkotb9sNj99fz0PicGqNCQFnghH+9izZvXiV4wd1wilY0+BLGgC3nf3WdqVZn9DfBTWIC44bJDyWjPBYyyVqxCJY+JJodmtF+xBvPLmNKcyvq/48vCKTCen8X0brmTpwwCYBePgoKz6O/rgfpuKVs2KbdxU1W+6zT1G3Y06iTjiOU+Oee2GNdCjXsLVjpHz6EbB9vuKoaYUjxzvjHft4IIC5YjYykw0r0Y+ApaBifeiDklKAbm6l3G9PKrQbVLt5ANihb7PKArFdm5DFYd0Uctl7yOqLhBsh2opIrGFXBSFH7TS002sB2GbLbqvk8ZMfNdqjq0lvfV7ChiSXw4D19yd6CqB69kGnb3gNgHZaHo0LqUZiAX9PieP9mkp+HFmqLH6hjQs7JPwVSMBpKSa5WQJ6jPFzsdGEoYW5P6xvLxoxGQUHa4eUDi8J1po5RnKJW3FZUmVHXC4p1W10+hJp96Eox5Zp1CoMZldW4XDJZ6sxPLU5pRHA401gOfjQMiLjmz+FpOXh4fV4ezItF/V0TfDa9T14xRUYVE4JSi21vSGzFlK14SmWzA0FLSkuMsxQSVkJiLX3Ni1m2/zCJ/EocKiqpWD4pOsrhQviU4Uk0b/lDKacSzsIhUbXgia4gHzyUn5HFTpehs7z4mWEeou+v5G7/AwwG3wN6NyR2gMd0kzWhXEMGwjyvUK1aAmU0k/2+cUY6CnnXHPFnvHMP7oA1ay1rpv9UOtID2HrjP7xnUBnUN3fqSx953wuwft4Fn8RJZd5VJgrHJnllOdtkc3lLW62OjECd4MxtUQD6FdBs9UCDTrn1t9HNBmzYpCZv6eWITNVTOaTd7g/ekieBag9XJT087AI1QGlbeTfKu6x4OquXcFmNeG41vJFArUkIY+J/Vnpf/H5Py8X95hCPYzcl4HeDFQ7YoRlDX86B3Hzbn5WTA+hFD32I+SAqXA715PprXuo+CApmyEgDIT3+9I+BAiXdfGxk+cAUco1wMdgkhX/y1U6rnzOOomtl+73zE7SKf129hd0/zdurC9P7dXBvehs1qUsgCzdbC7ee6cBAf/A9yqA1eaDQAA"
  },
  "scripts/summarize.py": {
    "sha256": "8def13b7efdaf2935aa3d6f7544d80f683b9719bc9c627e320f14d01dc91d937",
    "gzip_base64": "H4sIAAAAAAAC/21TsY7bMAzd8xWElrOLQLe38FCgH5CtQ1EIikQnusqSQcpJ06L/Xkp2muvhvMgSyUfyPVIpdaDsF4dggScb4x74bAntMSLQkoBwzlTgGso5LwXs4kPew5Q9RrhiOJ0LQyZwhB5TCTayVkrtwtTCLJ1mS4z3u+PL/feFc9qNlCeYbTnHcITNcJDrbrfzOAIv02Qp/MLOB0JXMt36jzuQbwwYPcMA39RLPqo9KC62LFz/QuJCyyTl1JvLyYcScmpOiH51GZEwOTSM1UHiGuzjUx4vwWF1ntH+MG7x1gg/2dmC3hxvBfl/IyEjXR62t4DTzOJIkra8B1TNnsIF6T0rEmVS3xvmKklrfn0YRYAxiGAhAYsJ/YMvfYr52KkPz1LeEouutKt+Y3FFc5m8gFWLjtl67iqYliHwpuDP0vX9K++WW9t5xlTTuNKtCHsQIYYWKZJLkzrZCbfQOj7wKAqeRYqm7U3LSKheZ4Hr1FVaTXiNIeGgVA+WQaREOz3KvVIoSFKuxOkvkv5re+hWv/06GDUzD+uM7EFaIMvW1SEYVDilTKj6N4i6HWfpWcDeN1K+crcRsHrMFFLpxqffUaq/W/4ISTZWNu5sfYI52pTQ10eWpUBIucC/IYRVG37qN33LQvfNY1mFMIIxtSdjYBhAGTPZkIxRKy1twyol923Tn+nU5v/QLJ1HdhTm1r8xPjtj+leR2npv7BbSKdl6kaHcZhzqKq6ej03cYtpRo7jrtYT0u79c8rqYTAQAAA=="
  },
  "tools/stem_eval/accelerator.py": {
    "sha256": "38363c026216828b1de42812cb0964a9e640e411a2275ebeaf5691afe87c25f8",
    "gzip_base64": "H4sIAAAAAAAC/6VTsW7bMBDd/RVXdrEAR0XQLYAHI+3SooBRJ7Nwps4VYYpUScqJG/jfexRtWYqddogGQeQ9vveO9ySEWJEmGQDBEWpAKXntMFgHaErebKwLsEa5JVPe+Iak2igJNQWnpIcnFSrbBrh//LIA9L6tm6Cs8bkQYjKZlLQB3wkUJe2UpCkTy2rGvL9b8oHKucA2WJHdTYAf2ZYIc+hAeVzkyhe4Q6VxrWmadaC68Yyp0GMILhHmR4N+BoLLIuvMj0s5F67Rqc3ZDcznkAwlP/EZFEFETyIe6ZyS9pQE41b0lXaMNSSukyeC6C5+DVVC68yxfv1opxNP8sflwVjtNh0q9vCzNUHV9NU566Y9eCMWgwG/9Pwf3AGCKnELgZynUiGUCpZ7Hq6B2pakcxA9i1giN79oGk2wUlpJBjU8DvMLXJI9Hf2c395CiYbXD3EY8GO5Ak1PGGIsQBkfUKNXI/bv5AwXv7XNnt3A2mqq2FfAhmNX4xZ7ZxqVyeGhM77mprSq2DfWN/EFW4L75WOemLNTGvdGVs4a9afPYkrmMYF862l9ntb5rge5HPKkHMXRv4bGyI2RyYUjT6HgbqzbF3wHwb/LTKJrCLdjzpPc+4SOCXsRnUAsFqi1lcjBKdY8Iy/uhm5qfD7Z6HHTbNbz9YM+88UG3O6/dCfYNDt0bB9HuaLnxnry/GM5RybMwNgAUWMGRx/WfSqd2nGodqg5+VCRo3wyapJHVhwZ3mwzjvUSk0y+7rQjTKr/5LuAnOgOk7/ngI9npgUAAA=="
  },
  "tools/stem_eval/audio.py": {
    "sha256": "a0be4f7be92c39ff3a3ce97ad867f48419ae1ab66ff880570a38f4239c085566",
    "gzip_base64": "H4sIAAAAAAAC/5VX32/bNhB+919x05OU2o7soGmR1gW6Itse1m7ouu6hKAxaomwiEqmRlJ206P++O5KiFNcJOqNIJf667767745KkuQNk0qKgtXAulIoaDVvmWZWKAlMllDUnMmZUZ0uONTCWC6F3EIjbrl5AVLB29+h5C2XJZeF4GaeJMlENK3SFpjW7K5/aZjd9c+m27RaFdyYOHIXHw9szyeTSquGRi1v1lap2kCY1vzfTmjuBieT968/XMMKLpbrPM8nk0nJK1zBynVbNGmLJrOrCeDvIOzOnTxXiDU1VvvZKSR6k2TAjPffr6afqPzIfMutYU17EKXdpRn8tIIlKD1MymLHpOS1OTVZadZwpJP7SYI7mKCfZsJw+Mjqjl9rrXSaXN+2vLC8BPRdc4W+wc1vX+DPN28Xl/DP649JFg8gXDU3SICjeu7+pskumQYERIWDYNIBbxjIMn8QOoo8zzd3litdck04k1pYW/Pk6tiUW2YOrE39Zs1tp2U/GwJw0MLyGIFpPxtCUTLLTiHuV/0PUHTUMaJHIn04HWnPjOGWEr8xabqcAv6jWE0hx33v/nh3jQATqSwUqkGJGMPLJMuOjnB+B3YdNKscOKQ6MFP0ahNfeOpFNQXLNIYFCcAHu0KDZecFuFrgy5YJucrnz6aooNv1aCrQiXr7BcVYuoVTkHyPbLVczwolS+F0LJVuyKTb+QKqqmn5Fjilm0HBq5ZtMUOdcgP3wgiJcGTB07HVKWxQdRllOHHx4KpUSPSnqhWzWVxNBWAuTCUksnRvQ1ySw8t7XsJLlHY+ROqkWGpRCBs5wxrV4GvTGQsbDkJCiiRe5J+TmFhM3qUj6PuHvdr/kCv7bKzoCtfsnV0XzyGaPpT+EDeFzuZjz8de32NhtIaOoPnFI6RUyd84wMDDC7ZerXA/FfR4aqDm69jUt56moULPdSfTT4O1UflNE59KCYlrJpWxpZAklNme/roEc69+UJDIUY4+77NpPDOZGdPPEVh3nA0jMUlokFU4WiV7VXcNX30lNr4l44OYDttIvX5LQcaXDkFxxeh/rExrs7jEUuLXegEGQJ+nUOx4cbP6oDvuyWCF7bBBrqDGghK7S9iVwTmkSzhz9WLIsQ1WXL9tBkOev4IFLr/fBU7E7y/fboUBpBnMDvvelWOeG2oMX2O4DKYXL1RJg97cN5Pcq8x+NJSfPVYBLEwYPUE1ak82zZChoaIg/GMdhIXU/13+koSGTjkstA9ryf6QluygJZyPEnH5GzO3ciOYvx6X2+HO8+/UKwxSxssUi3oAn2WPlZGfObPgSImVwwNyCSIKW9+hvQJjb+j24y9Gop1pXiOkPb/PeYDlOWdluca1xY35ruBv0CrRrw4yPLKi4BKBD7X9Pd2sdJ+CDqLFRolrqdGpziJQax0o7E2ObDvDS1OrYKtFGYv6cFWICRx06KZ91wop3ndiOD+HpZt2DqxxbovNdYVUWCcwzPp8ni/CERQVxEeBce4MhKuqwkiQbdXJMnWrxoLxWbTuiaDbQSDCUU3ZRoJy+2aOr8wVwaAlZ5pGyXRkc9Sd0R1hu5Ljwfl8sfQdbjCHdzdOjjyPOyonNlnc4Y7FZZ6f3LFc4pVzXPYdiYRBM7nFFidkOiZu2rM8C4SMczKyHFm6WD67fIY0DejPvFgwBV3BcS+twKcB7lkAcX7E7ve/sJ/ftunscrRvjDi7fwD5GC67ffdYHLngIokZewt0Ez6DNIT+iT8/w4dwwnfbQtZ9cts/437iz5Hgrj7pjJ6fT4/XPfGIA9Th6hlvVf210ssxfsKs8RMmKpI+M3pdDtp7XR/YnYEOY+0+g8AvfwHINooxnz91/djADect2IOCqqvrmcH7HQerGWo+ZkMUotJiKySr8TrApfs0KE+IcjoaQWyxr5A8+xPcFwUNxIMerXG/RnP+Sy+W01JUFdf4NYeu0KcHYJfYygb1RyMWcWC+QouFDGlLHuI59VmbMozIJnOV4WnmUoZhpSMevog2PeE9fYf8BzMi8gCJDgAA"
  },
  "tools/stem_eval/backends.py": {
    "sha256": "f3509ac5232a70faafd713ef75c10c07f4a9c319dfa45b572a8ed68468d417fd",
    "gzip_base64": "H4sIAAAAAAAC/61Y3W/jNhJ/91/B4z2cfJDVuEVaID09ZIu97QLXYm8/+hIEAi1RNhuK1JGUE2+Q//1mSMqiHG+2PZxfJPFjhvObmd8MTSm9bljvuLGEbZlQ1hG34wReel473pCht85w1pHrd29tQd5zJmG25YarmhPFeWPJm3efCFMNuediu3O2oJQuWqM70jO3k2JDRNdr48g7+FzE9y55twe7iBuM3nPFUPQ4t2PfXn6/WCwa3pI9k6JhjlfbAZ6wKoORgS+vFgR+oPa3uMAbYYceRYARr5jlZNxDDFNbTja81YYTqVkj1JZ0uuGSNLznqgHTBA9moFzREmERmkljTjZayyXRhijtzkxnQrmctCDdLY/L0OZC2FYo4cajj3Nr8o+S+CF8WV8Gk/BnmIDT/4ZTr43RJqOf4DsIIdc/vT4aVtmaSfQduVnn68tbuvQyDHeDUeEsUWmEM/irEgowd9ocstHZYB6r7wCJnNQ7Xt/12pvjMaoU63hJWc2t4/1qv75cbQBeGp1gNBhTTmHzDcnoJMNSRDMKJ2VJUA4lXIJFNIZPPPaZdRMmfyWfooK/WYBVqFXwn9dEmLSa8AeMYEs+DmajfXj+6xfSatlArOeEg8nkXrgdjjbCso3kTXGU36JXwE4EM5vMzsnMboeiKQzuGcfHv++5+m71utvwBmNqdVF8/4omey5//ryS3Wpd/PBqhGv8gbUYBZmH7xuvewmhUjXCZCdLz8ZES38R1mIgSw1RQGoN6aPAreQRZT39SMygFDek0dx6VY2+Vxj8ZA47/uzQtuIB1pXkkRaWtdxxZbWxaEyxEco/e+cfv1sdvj1M/s09OPo0YSmkF2V9Lma9B7dHZNHYwmyl3mT074n+kx9g0yMWKChbelfSombgaxqyT8F8zww4G+f6Ipwfx0dLln/Asy858by3mDpkqbaXwXqaDJ+7eULgj3n6Vx2dHD2HEk/9HdHExJpkwrGnVATSIr/CFiSgaXS05i+QcjNj5id7zkkfgGzlwfOuBgm1gPN9+Hj95jVJxaT6ffatVtNIEgFjzNxEoChQMviqrr779m73OYTeOAUZZTsm5WosP352Enu7SFwWBIPJ6DscSCML3VMAj7tsCY/Kis8cuedicpzfvnyJm9/z/wzCQM0JhARRKVqGPMSg2HQxR0Ec73p3+DLV+Uh+FmPlqVf+zMkgbKBarD4AE5HUJWMUtXpQzbxm3DxSLOH0igDZZvhaGG613ANaSwjrUJxx2r/4FcsY5/B6PNctFJxaMmvJdc1fBVPDUbEMVRXWsqrKLJdtTqYiNPmwxEj9SgHKMfCQ0Sqnq7ofyn9CHYBNDd+LGjbUQ8Ng0bxalj8UFwlqeILipJ6WZ9qO+ZKEOQ8Qfr7KK8uNyy5yj9xo0jKJcGx3og3FDrwtgZxjzwMgoZd+DqPnt0w9WNz0hgO9Mye0+kmrVmzzZOQdM6yDqrcNI7zqBivqudW9X1O5Qw+Y+ZHay0lH5vsBmecqnh/jJaWj4eWJzdkEVMetZVs8wVCD7b6SJFsLjB0gG8jVCiBHV2czqoKW8ndoBCqki3LmDAiwYCF6rExrQQyZ8Mhn4k5ibP45XzpYXrUQ9buKOUg2xGSMSaRrSIzKKx0HcX0nH6Dmuzh0FLdcnJSeCMY5Vn4/gKouZn1Eb9oPXDQwWQXTEEtAZIZnmAEC7KVwGb2iy5uL21T9XAIUivD21aO0nhsxehvyGPY8TdlOIrHA3EwBdi68HTxtWoAMCkcLZI98CVx15JAxyCKHWD2YGh05hFjM8UbjzNBx7GMt3FqALQbXDy5J/pABY3wl+TAPKMfsnR8uqeRb7P+sqSs2NEL7+Aq6l/NYCOprHwMtjUkSrimP09GeiDNgGEE+a4hWft5LxlB1/MFd0bncGi5vQeZbgI95bFN5rMZQY0p0Hjktpb5HKP39yB+U7JjptDrkBFyFU1h5mgMkgqjhDnTODJTFZPnR4E1HHgwsLOnN22Tulk7Yl0cnzCOE99C3O9BYQck1rrwogC2TUagR5Wp9qj+yXoUkaMvLi2eEfobCg8dL7/a5K3dC3YGuNANr7aoR1ZPhjoPvTwcl3CYHSLHTlPXtl8J7RaW06ZChWCL0TGoHNhrjL2HfbMNcvfP9SLkOmuEO2+iu8nbFA+G7LW/wcZuHqKla1OxKes/2SXcFqTZINyqa8/OMDHIS6m4fyb2OlG7ZnuPVxMd7zKNnFBW0FCNtQ08A2ZvFUX88u0T6WH+VOeIejh8oZ5QMndpgq0hxOIE9DjKFHLMmrExM921J6f+JmB8FSO4mNDu3zywJFX1sE796XHpssyCawx8QIwjY8wJekGPxgChy5hnfd6HCnGDzhbhDfxVeCqgyjaihPYXeK/gC5sLLNPd0bLY+OMDlf2m30o7o5Y4GG3CgP3ra2VhUXWBpnNoaXyh9xZs3AaFTLpP5LP3XYda//Z8pfzwZdsDKk5+dPs5ZNDghbQiZmVl+JCaGfz9uPoor8AwV33NzQN7ZZuFM99rccWM9m864AOl+RMWLHCsLuDqbgBsL9nHriEszJnhYdoRrFr6qsqzroUVHYoElwCigGnhkghEaKLirxA48nquMzzndWXcAAg4LN5yNexpuayMCo97g0K0Xuq02zIQicoYMY6IGj/ko8/gjkU3Rc0Q7Oxqd+51fzKjEREidyUiKzsCLjPcJTY4Mg96ML/w7QacoA8HS/2labQ5Vcq29SiLxafFffIcBi3MVAAA="
  },
  "tools/stem_eval/decoder_capture.py": {
    "sha256": "bc98dc466948f59d88b2efdc4f15dfc0a0f1f3250a9684de20e7e2ed106b3b26",
    "gzip_base64": "H4sIAAAAAAAC/6VXS2/jNhC++1dM1UOlQNE22W0OKQQ0TbOnthug2ewhMARaomJuJFIgqdhGkP/e4UNPOzGK6mDZ5Mxw+M03DwdB8KXRp4xDThrdSgqiBL2mQLck19BIenp/dQMV0ZRrBYQXIFaKymdaQEFzUVCJmlWloBQSBKfwXaySIAgWpRQ1lC3PtRC4zepGSA0bSRrl9hqi1xVbdTu3+HPhv/O2bnZAFPBm4YWleKac8Jx28mpNzn+5iNEi0zT7rgRfLBYFLVEts/ohuqyEjC4XgA+6dC2MUXDLoAVc336FshJEfzyHDdNr0WrI14Q/Mv5oQajxftVPyqvYaxlbkiJSvFstqCb5OoySvGnx0xrEt/MBV4V5ed8YVxrRyjzYIR5WVFTGUDBJESm5G9z9hlBZRL0Q+uEQh5VoeUHk7lcoBHDhnUZkMEKKVmgIo7Mi+RM14ZKItCQ11VSq/gr9eZBa5MPBgel+Uj/h9xBNGAKkd7KlMZKDKZ2Jp/QzqRR1GkIyxI1UaNA7nGTO3+yRciqRQVndKpZnSKoi84zyeNqIpvASIDy6VcElBBvCNMYhiCGwBMO1hyX+sDFBy88sp7imtOxQTNxiFFujw9Op6F2zp2HWImO12mbPhGYdfCi3Qt72gq1C752MTYJuvVtjygbib4zXgfO98bEHj8gaPXiCLmRTsdgZi14XLiDIHp95Icdoxi63mOCeMeb5zaZX2O/0G0ZbS5LTIjwh8lHFcHLytDHfRtrmoZhlLhKeP+iuO24UGtlybkLzOlFlpZWENIWg2SF38rW5TnA5gwMGPuJBnG51jzKKJwNZwyjaU7X+JW1TIH1CF+/UwNlr9SQAC+N808Z7atax78GzbJmQpsF7h/akqaSWu/3LSKraykDWob6H8OFLPHSALlE3cCUFeXfAvi027phprLY5bTTc2BcebAomlVLIy/dhc+emQUlYhSfGTiktgxeDTmh/RUmWmWhm2eslvNil12D/JpIwRRczVx3RHG3HkfXVK+15PKFJfEA2mtgYEmTPkN8ZGZkJR84dnyBdrYr61PIFuQjHxQlj+GayINuHmgkfIPA6CW92QZTYCokMnobCwgX3pGrpjYE0DP7oe6jrvlhHKlYzU8CxQZnq74snxneEf9eP06HbjR2Pxm72jhWshh9S+GhaQreIXbShDz8vzcaZ2TBVjDcJUyXj6EbYWUwwO45f52bbuO7jtHs/7TEFPJxh2TK5iJiansVppZaja+HBimA83wQ2hj406I/YZA3Lnyo67kNDSs/5bpngWoqzklm30gpDFU4A2SvhU9wzN3uk7vUODY6YqRlP3bjQaeEKVr3ev5ps5wJkawSOuKeLmRquhK4eIsR26+JTNKqDwxA1vY2nZWJ2EDaH66C2VxD7Ythl2JF8mihvMPQ4RdZjVjuDSNBo3ms66Smv+9U9Ys92zuzO+SHKd5IHOf9WGpfEXLwbiQtWWLO+IvpsUNiCqADSFkzMiumP8A+eaodObwPHPRsowKm0wcF0ZXxyEkhoM/8V2AFMPD7cXv91dmHn6WSKkp+UzbhYYq03DUKVExFVJjb2oemTk9AX7kqZdSJBTJDOPYgIbHL3Bgu7CqxI3VQ0M4NfbP4d1ESnwbere+SRaleWjMHnP79c3QWH+vE8ebsCbbQHw+mhw95IjjHFMleF0hkpTs+W72lPIHmvCBzAbn7HQ03dN/TfiaLHmvphkP5nR59289LkbzVL8P9YKKb9+/jfAUz9LtCLfwG8pH+5mw4AAA=="
  },
  "tools/stem_eval/prepare.py": {
    "sha256": "7de27593740951c7e0be235c8337d3450c28f5fa8550e0b201b8203218a746f3",
    "gzip_base64": "H4sIAAAAAAAC/51WS2/jNhC++1dMuRcZUJRki3YBtz6kbdrTtgGy2EsQCLQ0stlIokpSsbWL/e8dkpIo5rF9+GCb5Mw33zxJxtiNwo4rBA5a9qpA6Opeg+yMkC2v4fbD1W/XUNSieIBCtqWw+xoqJRvgLYg/bu1hB3jqpDIZY2wlGvsXuNoTsMZp/aeW7crpddwcarGD8eCGlpOQPvRG1Csvx/tSyEmKl2XuaOgUCt7KVhS8Fp8whUf6LbnB3IgG9WhCyUdseUv+zMj87Xffp3BUgkQdmdWqxIpEXQAS734avBTtPgXZm643680K6PM1Edg6R0aYdepXS8lpb4R0iA0aTtw5aVtKWS15qSOtTCEvc4Mnk6y9jqhmtWyPJmG6OGDDP6LSpMLW8M0WLj1h+1FcaISPvO7xWimpEnZ96rAwWMKtbPfveRf5AxEaXDJvtOwVtxJEdDJ+xxT+1aM27P6OTefs3om/cZXhY5Ed+SMIDby2rgxglGgaLDMY1TNtOCWIJELWUmgl5Zy+DqhAIz5kDneH3GjiEOc8iePhhFgKd/cU8onY6IY8tv8KYxZ8GYdyYAlqEp0l15nQut/ZPb8OOXgDV11Xo6N/vuMKnEnDm05DwwcoRVWRn7sBlOwpF5R2eM9NcYCjMAfRgmxxbIdK8QazGZmI8HZIfLSGhO90UsIZ7NbwI1UBnMO3by8uLqCShA4E5Jm5dWnXgX1g+3LV/DKHrum1oZpRCnVHlQNGgu7JP0El5WPmg+QLPWseSqES22Ot0dsPqqfs4klok8uH7a+81ujFjRoCh2WLz605dto5sFBZLKRmG+fIdaybJ1khu6ESNSYvdi7hRR1nO5EFjMXY28JndsU2kf0vy1w49+NIhsGVvMI/bFv1cdMhpSE/KSxC9ZzaHfuJ3RO/ABEJErUZKab3XylOMK/StAlePzOxZPqzZxohzQoNb0VFvWFjHaGMYy5/HOfcBi5TCHNnM5cB7WrqrBpzWiMduBZIn4A573J/K5CM/xOmd1QRz8TisR4jV+KEZT7i77mw1C6yd0tITVth8UR/PghWPz/gMNueU2EL2ncyHaduaTs6AGd00zU6WX95YoGGBVoK7Hfp8keqxpAnP9BVjnSpjw8BXh/5oKHX1NbWSk0ti+6KaMhFGowx6pEre2iB41E6H/hJOmuFxgl38qL+xnu59O2YzoURakuh6VU7H7h9PBXYGbh2PxSEzdNRoBqjEKNLeJ539CKgRsnzlkZsnsOWijTPG0phnjMP5F40impzet1kV2rfNzTZbtxJUqIulHCmt3leyiLP1wvNzLYaH1USdnbmY03umaHDrX0fpO5eFOT5NrTSK9rLOvy/GBSJf1QleTv8RgT3YzGotjzy+ISyW9k0rN0inrduS9qw/w3L3cPneQoAAA=="
  },
  "tools/stem_eval/provenance.py": {
    "sha256": "c76f5b079e782f084a86afac81e7bcad8952c85aa74101edd1445074c184c598",
    "gzip_base64": "H4sIAAAAAAAC/51UTY/TMBC951cMPqWQRunHpu2iHtCyCA4rkFi4LKvKcSaNaWJHttN2tep/x45dFRaJAzkktmfmzbyZ5xBCbqQwKAzUVNeoE+i4EFiClr1iCLy0Nm44aqCiBDx2DWfcuAUq3rrATsk9CioYpoSQiLedVB6u4cV5+1NLEVVKttBR4wwQDF/s9uyk+8KCMdQ6im4+3919uv8Ka3iOwD6EMiTXQBid4PKqwtV8lk0Wq+VsgnnB8gIXq4zNZtPJKqOzZZ6TxIdpQ7dDIM0mVwyzopxPGcvYarFcsXlVslm+nObZirGFha3mhQ08RVFUYgW6ptOrPHYVj64HuJJvURtbVKCXBpfRYD1wUw+EfEgqOxQxUQUZAdWgjULaehz3VFIBq3uxAy6AG1RxQ9uipNfBM7WvMp5k0zm8BvcZJVAQMrogXCpK+66kBuMBzxej0PRKnO01Hv3KlurJHZTNuXFzGapNYE+bHgO6O7EsL1yGU4NuTlQ9WZM7TR3hje6rih9jkpq2Iy8cU5/F4NHELlVa9m2n4yFVYnk7da2niVWbMpsdPun1vXIW2jTysLGiWn+gjcYRvAHyQ/yFrrBrrC5CiZ4Xq5HtNn3nmxi4FZTtUJSBHZNty90UL4JLfZjsTdeb+IFsuSEJkPGNfVskn8EeKNyPO6o0OuvH23fvyWMCjt5Q+Ci1vrwLcuDVOdOrNQQ9P4RKHi9TVJRrhO+uJbdKSRVX5NbeLmbsLeRCh9VzCDydMZ9fIp4S2Ep77u2n0Cyrf9Pr/+LqQwfT2DacYUO58NteGOXSluOKN6jXQv7ZhzN/D/EPruRbGBQEQBgA7f3aox0lFVss30Jvg0yNv7Uj/J7sOLjm8qyMoHnfgegXQsDtmt0EAAA="
  },
  "tools/stem_eval/run.py": {
    "sha256": "5b8fb4163c595f45a57ef9c2f08270b504d5d226398d2040e9fa6f3855a0abbc",
    "gzip_base64": "H4sIAAAAAAAC/7Uba3PbNvK7fwXC+2AqlRg7SZuOU96M49hp2ibxxM7dzWg0HJgEJTYUyRKkH/Xov9/uAiDBh2Sn18uHWgCB3cW+d4E6jnOe8ozlJRO3IqwrwUpRlHlUh8lVKphMsmUqZrISa1hQiDJZi6ySr1kBuyTLxLUoWbIu0juWZLEoRRYKz3GcPZjLy4rxclnwUgozVn/S5Mpbi4pHvOLmy+8yz8zvNa9W5ncu9+IyX7MC5mCfhsDOrSVASxXn5dqM5Z1sftZXcJpQyGamghOY33WdRHsKPK+jJDfAQ57lWRLyNPlTTFmawPEzYESwTm716jAUqSh5lZdmz1qs8/IukBWv5BSYKEUVdOckbAmrIBLXSQhg5V0WrkrA86dQQIFsYHNQ5XkqDdRS/FEnpQiIPPVp2kziUDOnzK9FxoH5ZuPJpw8f3l9eTFm4EuHXoC5kVQq+BrQr/vz7H6bspkwqERDX9/YiEZNE3QKkz0sRTdkVD7+KDH4kGWytUex4BBHJydEeg39rniWxkBXzSXZemvNIug0E9ow55reHC5yJBxREQSVuK3cyIRhJ3IDxlqJyHQnkrnkAWiUT3MKe+OxQ4cN/JU+kYP/iaS1OyzIvXedLJusCTwwYFTpewU6mADkNGnXsDnVhnkUJLgbRagoRnSFo3lkQKADOYhcxJ9YGYDzPloCJxxXYiEVbS1TLWpblYBwZc50rLqUzZU4EH6Qz2YXuDQeVSjJhwZFgcYIhDDRpDcOgQxwkQfyWiswFHXWVSOnkagqHuIBndy7uOGA/+bSN/cSeP3364pCBrakJINjWiC0SgnGdJX/UQiO/EtWNEBkA5lnEnh++fPXyxxc/vHylCW24LkGz5s6xs0DatToy32cOD4XDRApw0TTdEYFJZ6GA/Z5fEZgFjZDwZg1S325oTyDzugQ78pmlK+Mo5s1g0ew2bCYgXiKDOEmFSwzVc+Aa8vQa5jyEDqJ/0qJqP7b0PKBq2nOta7DEK8E4Q4R4tmol2iOEaVKwCJxGCC7rTnO6axqKvG0m0Oi/dehvpfEhizAi6uhWFwnK0+NFAarg3rfUOUetKMF2WnuAD7b/chAkzOGfaQdw55+jeIELq9Lty20yNQsMU456PNwFOqr1mY8sNjeTi41iRSmquszYfd8hHrFDxA4BpZYwcNBpZ3AkmNQWArPGdXeocEwICMJ8vU6QMzpGzPX6BQBB/sIX/NPbPuYMjx7pVvugUsFBocZZaIPSnL/h10MYNjkwkCBhPLujVC3P0rvXoJI3GO9hybNwlZfge4CvHDxmRHomrkFZudIZNNp1HokU9KWoK+lsdFgMOUggCoAjbmM/U2QQRMNkmYEgSqGtlSzPb80MjwBaU6eV4oPth3GtJ27Bf0nb2LXcP+aZ0HpA0bwTYmnrWChVi3UgVSpC5uwsRYbJCjJoKyZkSMbXQkWhrnC6JmXMaGop89SSiB20ulQhfKII2NdOPOxGIL8UGODDVZ19BS+X5tlSsioHVseUdFZaaq9ZLdEHZuKGlbUVaZ0lpHqYHwUS0joIH3BKIMJmRwgKAjj8Dht726ZdHouqAt3CcH2/mag5dGnrzswAxCvvYDLpckjjVpyZ97cs/hYOGaBbeWQfTRUD5PeNkitdaoa786BH0AP6EidLrUDjRFHZ4TPXtqiukkO+jOZAOgkD5Somra/upD64ACJyKaBeSK4xf24hW+6dYrX257CFjq2wzhXGR6WC5DbUCRLJ1onEUgpB6ygI3knENU0CWyQcIqugiioFMgLccycOKPTaI+lCzYXaCoqBtiozXggLAu3SmzriOBRvTEJ/UfFlO7oRyXJVBUkGBYTybeAVQVOgODAaQ2DNAOQxWOCCNHgFoVKRtEvlRwoaLSUofzws8jxwNaKs3IMpRV8s9NyApBwEk0H2JOfPFxQqwjIpwG1rNMQEUCSs9gKRXSdQaFGebeqq4SfaZy/1x1ZpYrve8uez4Ocvb4KT45OfT1EVYfyf08t2/OH8tzfHJ7+efnyrRyefPp69f/f2/WfbV+bS04jmCHoBBFiI1ZzKkDslHTHdaws8Gmr5N/pvT1IOLVEJHMrAFQACWuQJ8khSWKAioPvJozODPcQKwOyeFrTBYeNJHoPlZzIvH6heLi6P350afZCUrkLdH67ITOI4CROe2hWSRSAqA1KijeQf7HIl0H1A6ZPmoHJ2q8Jj/yYNh5odMEmd+ktckgKOilggVSoqlN7ocAkLvb2eYIyoP52d/fb+46mDQnIOncGyy8/HHy/OPn3+cPr5YnSx6UbkZbiiCdUWYH63TeDSAi1UNfUomWpoGPzDOuJOXxCflWZrUQD7kNMF4AWfpQTDI15gmt4I6OTL22OPYTV3fHI6u6hEAVkWOy7A07OLBHgJ7DLBxDgTOE7fv+xS12lf3/QEFHwBJWfbz441oVXD/Z2+pPEnREIAtbWoGscKKguapjtFptVkdumxOrLaF5ikXx/NMKJFZZwOmRqczIwd9h01rDz8z0sgeiVu50c/UnXc87+YdK7R755xqJInqlY2gJw+Jubbp3gMsXqY1xUGZIveyZgzc0BfLi5PzwPwhie/nn96//HyIkDfh/aAYtEQrBCsAgKwDo3VhzJIaZGkcq4JVA4JBOa6SrJVPxQfpOaTpWZY0ZkmIoAzPz3zwx1WdI6yMFht+nnOsqhhSAbrodFhjqLNmJjjTpAyY+fGMrVwOnYEXP1wfoGxQtd9ePD7zaYJPQWcDGxdRR/CiIvpBwVWGpU8k0g9gMBxihzEcgkHWb0u7mz/XJV33SxTc3/ekrCYa7Qot2Ej19MLXb2q1QVxG4pirPfrnaulH/PqLK+ziNxRl4yCa3tqhGkXR6jmMLNb/ZUymbx1uHws150SljbWJ1kkbqnsQ54LYB8VVG4baOaqdl50i58H/JRuaQwzfqTSzr4oQjRZsG/bX7x/T9QdHbyINsE9gWtrsYWZsgo4mMMCTs1TKbfY7A/ReOuv8NulIjXIv/qKrb3KhSSAEedRdTIGXkzFkbk9OZdgh27szNVhvjvcPLvHfqTF4H1k8P5islmwD2LNv/JElxcYMSWM7iCxhgQgzaVKFoA2EGSc1nLlX5a1RbruM0L1Vgv7PEZSOgPqWUTFqc3s0x2CB3TFQQhqC67T7YJuddVtE+9+4FPRBB2D3/VfU0x/sNSH9DgIi1p91nNT7UB843a6muOb4WRbC2q3c7Rrg9FIbTnNLophFAohgQChgwsM4VilsantOtAEWr0j0BBMrEUV5mlqpgd7e99dfUQPCq4oFaU5QZes7imsSxmTetlZ18A5KiGSsKRAq5NkuyMKwmZGgfYeI44hZxqcvdjclKMYTVXZOhbCJ9sBCijJQ6qI1XGQLwRuPqpE9w6mVro5ij+pIdpv4+H8ZnS/CmDVilq8dHPjmhQA3Fkv4fOWKXgU56mH4Wp4hsWYWALdiLOcpKM/WY24cTdmgxjpz41hgryFwrvtkxt8RQnqk9cyuCedUJkSgNzo1mgHcnsn59oIpgZduxrsTKyLqtdtdB09vzVZnFjHJYNvtg/devfUGrIOCpOe81aHfLD5aeEwe3YwuVmiGWzDjg3wRzO4aaVGSVi5KjBRk9THzFgnRiO5gN+Gr70RQW09sOkVTvb+Wox7p5u2jOL0fhu/9xcbtq5TCHj3po+6T64liEtg0/6UHUyevXh+cHBw5D2PN3JLDBxkfMM7650u8NHR8CYBW1dpcfNGgKx7TOYlv0F1V41VAG58uGlhqxpOZS7qkmYxmaocqr1E2XH/svXfIEnSUFV+NDX6/5fChe5Z1gV2zdyWCTpk+DvDxa7D9HIBwzff/Jiyp093CLQXwbM47940tK8nVNwPwLsVdeUO44L9JMF14hi2XQkHA8PsGusNgZk9/pjJVX4TAIPLRFBZghUKr3z7JmGWx/hHWxFViPymSysIBC8/lEoPEhF+JV3MlirkdT7XOKCAsXQEOdxTmwn7JzvwDr4f08p+98pYZ8TMfuzxUgNL3Tupe6jOfZNusSQDv2+/N3HJAox/f2b1tacNKr+v72wJWPzDHaqy5rdBs72XqOE3gNhkMFN2eNBjqn1lBzpijFC14e2Pjs6ytl3jdcF2nta4NpxtLLBmYYuCucPWjIdvb8CmzOrh+yZhGsG16xaXNTcPPqrn+H5v9KJhMpmMFj6MXegyJuIpX7N7feWwP3AX+4sj73C7X9fV9in9oct3ycSwsB7nU8zhTMgk2uHHzn11V0CZi6OJF1C5EgSbI3ZPU5vtCcw3xMXGwv6H+GoX4vruk9qtreD17ckaLMUEHnqaVqpMkX57x+WS3P85fXEjoXqBaDRBEOVhEEysnR6PooDrLa4zmxmtB+KQbz7aSfNeK7IEtWW/ucPHZ1s5eGjpN++4Hg+kc1Vr4AweFmEQiDlw01dfdoKklxjmUNSXzdDKfec7p4UzP5ji64Tni52gIIT8Ve6An5qB05m1fqqlpznnIVDx4sA6Hk6sRFr4zqlu+ad3LMd+VFbldO3w4kCDZHVGHQW8uMwzDDAQZWS1mze6npy1Za51OIX4bcKXWS6rJDxipqbFC5bjk1OMQft4G8HDioG7UG+4qINP5DOKY7oi3k2IKf1n5taN6KD9LTdeeQ07EDs+IbNuhg/Z1R123LDdf/aO8TUwLIbApN5IGJJKgUYk2avd9Oh7SnTaFPR8MMwccwSQ8+6dpudg83K3DJquxBj71YVGaxYzDMN4LmZdVtGl0m6yoCqdqZazbVcwCz6/mF0ffg/2K+ku2p67TdW0bXGDPbvxUiU/M5X82BHPE3wGxIr6CkLqSuDFXZXEqFNm1+vG0CgXQeE3xc5u9LrT3TlzXVGPmTrYmEEUXY9C33e7AdXOGlcOcyp6Oqh1gZ2cfzGNMbwxh4PUkIu8g1mV4Wp86Jfw8Z5CS38QsbllHr0dH7/x7hRIlNR0e22jN+Ej60yAorDc5pAjcVkTTbMuJhYq7o7fej0xjTv0FmPkwQKw9i3QB/4C23I8o7s9fDyFHrLzVpUwmDfiDcpGOoOOaR9b+7xc3ydaZv4AkkGrsvnS63E9REVH6W1KehZGFtJY0DUkFsYJ9kgdwT9sbTaNOnU71amwxqB03nbpFqPTeWHag/jE73XN+lcoHR4gsvbxNr0bpbKF2Q6un1ShReH78E5uP3p5az+3pAn1QrjDtl4L1mab9SJhoOj61fLInQs9E59sE/tbjcaE3kbwFHVR19tYbAlYv39OJPWV0bgHZZeLuQdFWPVsiF4R0r2zjJMMctj+lk7SqzccsJ/6lSg+uSYe9EozUkxzCbXr6qnLgF9gR1OmoiPCZ9iY+TSPIAbJFZRm6+YVko2qXywu2Dipg436JhSZj8EvaneqL4MNjTq3K5upweLulUm7Q88/8t2ADXHkSm7Ey3YeLm+VzONu/LaBN41l3XvVjxUoX8aOpb6esy47rfLJ7oG3dJkKyno8NuaCd1wPb32E9q11qEXU/60W/TYmPFiSPh7c3h5w1FBK+hYEmDsHgWP+JxqsRvf+C+LR04+HNQAA"
  },
  "tools/stem_eval/system_tools.py": {
    "sha256": "d31e7ebceba70244ccbb61f9f994939914057085892310ce8c91465a6f67a3b7",
    "gzip_base64": "H4sIAAAAAAAC/5WTTWvcMBCG7/4Vgy61Idj3wB4KKYQSktAEekxke7w7sT7ckdTdUPLfO/5Y7yYtlOoiyZp59frxa6XUjW90RNCpJQ/RexMugFxjZO+2cO0t1ox7qMlpJgzgUwzUSgN8TcNrRIYe2aH5FOD+8+N1qZTKOvYWBh13hmogO3iOcC/bbFkPRsfOsz3uwy5FMlmWtdgBY/DmJz6NXnKnLRaXGcjAAzYp6togbJaOcr+jZjcXTTXUnZXNbeNgjInd2dGx+GikDK8hos0L2GxAXWnek1OnfimBgbGjg6CBXFV+iNVuQVMJGnUBqkqBKyM0zfSkOLWPo9GupXYkvZlQ5LNeARWM9t/VirG1vKTw1JHB/IPe2WuFyPlaP3NYTm69w5Xqj0T8D6p/ov9/qqwpIHxLLpLFL8ye8xNH9WtUfYMoXntoSZinXrsSrggeka2EzIDVzd0DvGijnZzB85Q/ciFqY6Dr7IDbZ+G9qiqjTRq9Ry1ZmtO4hrMVhVUqybwF5yPW3vfluYYYuCGXDtV3cq3fh+XC5b5JpusG9jWCxfFCkoA3vd7KXjuZGO4eylmw+MB8+rcmqOH4GRd2E47Lv3ycKXLjag7c7GJM2WJCFW/ZbwM+CQu9AwAA"
  },
  "tools/stem_eval/vae_reference.py": {
    "sha256": "62fa0c1b7113630a99dbfd3b99ec69be4aa1b5253d895c6774cee3aee5178104",
    "gzip_base64": "H4sIAAAAAAAC/4VUy27bMBC8+yu2OkmJ5MTOpTCiS5Nc2x7aXgwjIMSVTVQiCZJy7L/v8iHFjyA1YMMi9zEzO6ssy76pQXLk1ZuQXL3B08/fwLFRHA0YbNGgbBDehNsBHljjQO3RdExXXNiGGQ6W9bpDYE1DhZyQ23mWZTPRa2UcyKHXR2AWpJ7NZhzbVPvViQ553jGH0tkynZbQ7Ab597U1rEdbL5Zfy7Fd/bAsVjOgD1V/DtEgOCWLhnWwXpRP5a8NTPUMOiYkgQFkzQ4aZRCUJ8Ikh4Tc3zZKOjy4ANkXF+1YYy656OFLDQ+gzHRod0zj+n7jLxbXF9ViA4/xQipir+fCtoTD4Ui1mLOuyxMV/zFMWIQ/rBvwxRhl8uzloLFxyCFmUiWJvXbHS5ZZMUL2ve7hsR7FIgynQt4tP2v3I+X0g3XQIduTSKCVFU7Q3ziYIGDqZ50h4aE+awEVLG9S9xClmfFz2Cnth8qkxM5SznpTwnfiE39DZEtiWUfhICTBk1vM78trXcvU+EQ6pFnW0AuZx/zbFPJBdjElddg68ofY7pxPZgfqFtKrBL8oQ8lrANTudoyZykV9PA4aNrPMGHbM4+FYYr0qV6VvuwpdN9SAu6PGmjLaTjFH1p7qiXFD+LkBx8MrA176LAVe+ezj4Z9vuxuMJC5C7lkn+B0ZL1mQDVyorDhxkRPKb1lPW+Zza1qqfa94fo4zzC2wrrwAZzRDjXFb3ivRQ06uAWEDNW+TsLR5jCfa/vZKkUVQZHRa8V/iz4m0z9gS58mj3o3hjVY5VcUJRqgn9C9dneS4RDTFh2WYM63JQqNCwRXRuFGbG6q6yini/XEzb5Q+5kXsHOfjh01vrcYjo28eF40dhK2rRTH7B6EklcPRBQAA"
  }
}
# END EMBEDDED BRIDGE SOURCE
identity = hashlib.sha256(json.dumps({k:v["sha256"] for k,v in BRIDGE_SOURCE.items()}, sort_keys=True).encode()).hexdigest()
ROOT = BASE / ("ace_context_source_" + identity[:16])
verified = {}
for name, payload in BRIDGE_SOURCE.items():
    relative = PurePosixPath(name)
    assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in name
    target = (ROOT / name).resolve()
    assert target.is_relative_to(ROOT)
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Source checksum mismatch: {name}"
    if target.exists():
        assert target.is_file() and target.read_bytes() == data, f"Source lokal berubah; jangan timpa: {target}"
    verified[name] = data
for name, data in verified.items():
    target = ROOT / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "tools/stem_eval")]
print("Source bridge siap:", ROOT)
print("Snapshot:", identity)


## Periksa GPU, alat audio dan ruang disk

In [ ]:
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "tools/stem_eval")]
from cuda_reference import check_cuda
from runtime_environment import runtime_environment
from stream_process import run_logged
from system_tools import require_audio_tools
MODEL_PYTHON = ROOT / "vendor/ACE-Step-1.5/.venv/bin/python"
UPSTREAM = ROOT / "vendor/ACE-Step-1.5"
ASSETS = ROOT / "configs/assets-base.json"
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Pilih Runtime → Change runtime type → GPU"
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))
    if any(shutil.which(name) is None for name in ("ffmpeg", "ffprobe")):
        run_logged(["apt-get", "update"], check=True)
        run_logged(["apt-get", "install", "-y", "ffmpeg"], check=True)
require_audio_tools()
missing = sum(item["size"] for item in json.loads(ASSETS.read_text())["files"] if not (UPSTREAM / item["destination"]).is_file())
required = 2 * missing + 5 * 1024**3 + (8 * 1024**3 if not MODEL_PYTHON.is_file() else 0)
print("Ruang kosong GiB:", round(shutil.disk_usage(ROOT).free/1024**3, 1), "| Estimasi ruang baru GiB:", round(required/1024**3, 1))
if not DRY_RUN:
    assert shutil.disk_usage(ROOT).free >= required, "Ruang disk belum cukup untuk runtime/cache dan bobot"


## Pasang runtime terisolasi dan verifikasi CUDA

In [ ]:
if DRY_RUN:
    print("DRY RUN: setup runtime dilewati")
else:
    environment = runtime_environment(ROOT)
    needs_setup = not MODEL_PYTHON.is_file()
    if not needs_setup:
        probe = subprocess.run([str(MODEL_PYTHON), "-c", "import torch,torchaudio,transformers,acestep.handler"], env=environment, capture_output=True, text=True)
        needs_setup = probe.returncode != 0
    if needs_setup:
        import venv
        bootstrap = ROOT / ".bootstrap"
        bootstrap_python = bootstrap / "bin/python"
        if not bootstrap_python.is_file():
            venv.EnvBuilder(with_pip=False).create(bootstrap)
        # Support source bundles with the older ensurepip-based installer too.
        run_logged([sys.executable, "-m", "pip", "--python", str(bootstrap_python), "install", "pip", "uv==0.10.0"], env=environment, check=True)
        run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], env=environment, check=True)
    else:
        print("Memakai runtime model yang sudah tersedia")
    print(json.dumps(check_cuda(MODEL_PYTHON), indent=2))


## Sediakan bobot model pinned

In [ ]:
if DRY_RUN:
    print("DRY RUN: unduhan bobot dilewati")
else:
    records = json.loads(ASSETS.read_text())["files"]
    available = all((UPSTREAM / item["destination"]).is_file() and (UPSTREAM / item["destination"]).stat().st_size == item["size"] for item in records)
    if available:
        print("Semua file bobot tersedia. Hash diverifikasi lagi saat generasi.")
    else:
        run_logged([str(MODEL_PYTHON), str(ROOT / "scripts/model_assets.py"), "--manifest", str(ASSETS), "--upstream", str(UPSTREAM), "--download"], env=runtime_environment(ROOT), check=True)


## Piano asli dan plan seluruh lagu

Upload satu file piano. Keputusan konteks 90 detik sudah disimpan, jadi tidak ada probe baru. Margin dan crossfade tidak menjamin konsistensi musik; hasil sambungan perlu didengarkan.


In [ ]:
from google.colab import files

missing = [name for name in ("numpy", "scipy", "soundfile") if importlib.util.find_spec(name) is None]
if missing and not DRY_RUN:
    run_logged([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-context-chunks.txt")], check=True)
assert not [name for name in ("numpy", "scipy", "soundfile") if importlib.util.find_spec(name) is None], "Kernel memerlukan numpy/scipy/soundfile"
# Load this snapshot explicitly when other helpers were imported earlier.
spec = importlib.util.spec_from_file_location("electric_full_song", ROOT / "scripts/electric_full_song.py")
single = importlib.util.module_from_spec(spec); spec.loader.exec_module(single)
sys.modules["electric_full_song"] = single
spec = importlib.util.spec_from_file_location("fresh_context", ROOT / "scripts/ace_context_chunks.py")
context = importlib.util.module_from_spec(spec); spec.loader.exec_module(context)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload tepat satu piano asli"
    incoming = ROOT / "runs" / ("input_" + uuid.uuid4().hex[:10])
    incoming.mkdir(parents=True)
    SOURCE_PATH = incoming / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
CONTEXT_OUT = globals().get("CONTEXT_OUT")
if CONTEXT_OUT is None:
    CONTEXT_OUT = ROOT / "runs" / ("full_context_90_" + uuid.uuid4().hex[:10])
    plan = context.prepare(SOURCE_PATH, CONTEXT_OUT, context=90, margin=6, fade=.58)
else:
    CONTEXT_OUT = Path(CONTEXT_OUT)
    _, plan, _ = context.load_plan(CONTEXT_OUT / "plan.json")
    assert plan["original_sha256"] == context.sha256(SOURCE_PATH) and plan["context_cap_seconds"] == 90
PLAN_PATH = CONTEXT_OUT / "plan.json"
print("Plan:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "model", chunk["model_duration"], "detik; termasuk margin")


## Generasi seluruh lagu dan stitching

Tiga konteks untuk piano 4:03 yang sudah diuji; dua sambungan. Tiap konteks memuat model kembali. Jika proses terputus tetapi VM masih hidup, ulangi **sel ini** untuk melanjutkan chunk yang tersisa. Keadaan VM tidak berpindah antar akun; unduh hasil sebelum pindah. `DRY_RUN=True` melewati seluruh inferensi.


In [ ]:
from IPython.display import Audio, display

if DRY_RUN:
    print("DRY RUN: tidak menjalankan model atau stitching")
else:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    finished = CONTEXT_OUT / "stitched/seams.json"
    if finished.is_file():
        seam_report = json.loads(finished.read_text())
        assert context.sha256(CONTEXT_OUT / "stitched/electric_bass.wav") == seam_report["stem_sha256"]
    else:
        seam_report = context.stitch_run(PLAN_PATH)
    print("Perlu review sambungan:", [s["index"] for s in seam_report["seams"] if s["manual_review"]])
    display(Audio(filename=str(CONTEXT_OUT / "stitched/mix_full.mp3"), normalize=False))


## Unduh hasil dan kirim feedback

Simpan paket WAV sebagai hasil utama. Upload `chunked_feedback.zip` kembali ke chat untuk memeriksa dua sambungan. ZIP tidak berisi bobot model atau piano asli. Sel ini hanya mengemas hasil yang sudah selesai; tidak menjalankan inferensi.


In [ ]:
stitched = CONTEXT_OUT / "stitched"
if (stitched / "seams.json").is_file():
    feedback = CONTEXT_OUT / "chunked_feedback.zip"
    result = CONTEXT_OUT / "chunked_result.zip"
    for target, include_wav in [(feedback, False), (result, True)]:
        with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(PLAN_PATH, "plan.json")
            for path in sorted(stitched.iterdir()):
                if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_") or include_wav and path.suffix == ".wav":
                    archive.write(path, "stitched/" + path.name)
    files.download(str(result))
    files.download(str(feedback))
else:
    print("Belum ada hasil lengkap; tidak ada ZIP hasil model yang dibuat")
